# Notebook 02 — Experimental Development

Run this notebook first. It calculates fresh development selections and exports their evidence.

**Delivery status:** source verification and lightweight test results are in `docs/VERIFICATION_REPORT.md`. The production cells are intentionally unexecuted. Full optimisation, final training and held-out evaluation are the student’s Run All responsibilities. No archived output is presented as a fresh result.

[Colab execution guide](../docs/EXECUTION_GUIDE.md) · [Methodology comparison](../docs/METHODOLOGY_PRESERVATION.md) · [Source API](../docs/html/index.html)

These chapters change the presentation, not the original chronological experiment sequence. Mathematical explanations accompany the actual computations. Timed loops and grouped-fold loops remain intact where splitting them would change timing boundaries or execution semantics.

**Contents**

1. Engineering context and research questions
2. Environment and source dataset
3. Data integrity and the fixed profile split
4. Physical inputs and causal thermal-memory features
5. Grouped validation and preliminary numerical investigations
6. OLS Bayesian pilot and expanded search
7. Five-model Bayesian pilots
8. SVR computation and nested-sample scaling
9. Full-development QR OLS and Bayesian optimisation
10. Larger-sample five-model Bayesian searches
11. Full-development training feasibility
12. Efficient SVR investigations: original Stage 9A–9M
13. Development selections and authoritative handoff
14. Evidence, limitations and conclusions
15. Bibliography


## 1. Engineering context and research questions


The engineering objective is to estimate permanent-magnet temperature (°C) of a PMSM using observable electrical, mechanical and thermal operating conditions. The research aim is to investigate accurate and computationally feasible regression methods using profile-disjoint validation and historical numerical experiments.

**RQ1 — Leakage-safe time-series validation:** How should temporally correlated motor recordings be processed and assigned to folds without overlapping profiles or using future samples in causal features?

**RQ2 — Baseline model accuracy–cost:** Under the original Bayesian optimisations, what accuracy and computational costs arise for OLS, k-NN, Random Forest, Extra Trees and MLP?

**RQ3 — SVR feasibility:** How do exact RBF-SVR, matched-kernel Nyström approximations and direct LinearSVR compare as the number of training observations increases?

**RQ4 — Numerical evidence:** How do numerical checks, grouped-fold variability, timing, convergence and regularisation sensitivity support or limit the development-stage conclusions?

The benchmark [KWB21] motivates the inputs and thermal-memory features. The project’s QR checks, controlled scaling studies, sensitivity experiments and software tests are distinguished from that literature. No new confirmatory hypothesis or numerical finding is invented. Bracketed citations resolve in the bibliography.


Motor losses heat the rotor while ambient/coolant conditions remove heat. The analogy
$$C_\theta\frac{dT}{dt}=P_{loss}-\frac{T-T_c}{R_\theta}$$
uses thermal capacitance $C_\theta$ (J/K), thermal resistance $R_\theta$ (K/W), heat input $P_{loss}$ (W) and coolant temperature $T_c$ (°C). It motivates memory features; this project does not identify that differential equation or enforce its energy balance. The intended use is offline temperature-estimator comparison. Laboratory interpolation cannot establish a safety-certified motor-protection system or extrapolation to a different motor.


## 2. Environment and source dataset


<a id="s-2-1"></a>

### 2.1. Portable runtime and reproducibility controls

**Research trace:** RQ1–RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 2.1.


#### 2.1.1. Purpose

Create a clean run identity, portable project paths and a record of the unchanged production protocol.


#### 2.1.2. Mathematical basis and assumptions

A random seed fixes the random-number streams requested by the source code. It does not promise bitwise identity across BLAS implementations or thread schedules. Dataset, source and evidence hashes establish which inputs produced a run.


#### 2.1.3. Implementation and parameter choices

The bootstrap locates the extracted src/ tree, creates a new UUID run directory and installs figure export. Production accepts only full mode. Lightweight checks have their own script and cannot export an authoritative manifest.


#### 2.1.4. Results and verification evidence

The printed protocol lists original bounds and budgets. Outputs are written below results/full/development/runs/<run_id>/. No previous model or result table is loaded as a fresh optimum.


#### 2.1.5. Interpretation and limitations

An interrupted run is incomplete. Keep the same extracted source package and numerical dependency versions for both notebooks. The full Run All is intentionally left to the student.


In [ ]:
# Locate the complete extracted project; no personal paths or Drive session.
import os, sys, json, time, platform, hashlib
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import numpy as np
import pandas as pd
from IPython.display import display
start=Path(os.environ.get('PMSM_PROJECT_ROOT',Path.cwd())).resolve()
PROJECT_ROOT=next((p for p in [start,*start.parents] if (p/'src/pmsm_project').is_dir()),None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Extract the complete project ZIP, then open notebooks/02_Experimental_Development.ipynb.')
sys.path.insert(0,str(PROJECT_ROOT/'src'))
from pmsm_project.data import resolve_csv, verify_csv_bytes, validate_dataset
from pmsm_project.config import EXPECTED_SHA256, LOCKED_DEVELOPMENT_IDS, LOCKED_TEST_IDS
from pmsm_project.features import make_paper_inputs, make_temporal_features, vectorised_temporal_sample, stage6_feature_names
from pmsm_project.study import DevelopmentStudy
from pmsm_project.sampling import nested_sample_order
from pmsm_project.search import SPAN_SPACE, MODEL_SPACES, decode_spans, encode_spans, protocol
from pmsm_project.handoff import select_winner
import sklearn, scipy
RUN_MODE=os.environ.get('PMSM_RUN_MODE','full')
if RUN_MODE!='full':
    raise ValueError('Notebook 02 preserves the complete original budgets. For smoke verification use scripts/verify_reconstruction.py; it cannot publish a final manifest.')
RUN_ID=str(uuid4())
STARTED_UTC=datetime.now(timezone.utc).isoformat()
OUTPUT_ROOT=Path(os.environ.get('PMSM_OUTPUT_DIR',PROJECT_ROOT/'results')).resolve()
HANDOFF_PATH=OUTPUT_ROOT/'full/development/development_selected_configurations.json'
from pmsm_project.handoff import start_development_run
start_development_run(HANDOFF_PATH,RUN_ID,STARTED_UTC)
BASE_DIR=HANDOFF_PATH.parent/'runs'/RUN_ID
BASE_DIR.mkdir(parents=True,exist_ok=True)
os.chdir(BASE_DIR)  # All original relative CSV exports belong to this new run.
OUTPUT_DIR=str(BASE_DIR/'bayesian_model_pilots')
SEARCH_DIR=str(BASE_DIR/'scaled_model_bayesian')
for directory in [OUTPUT_DIR,SEARCH_DIR]:Path(directory).mkdir(parents=True,exist_ok=True)
# A plot hook saves every displayed figure, including original cells without savefig.
import matplotlib.pyplot as plt


In [ ]:
from pmsm_project.plotting import install_figure_export
install_figure_export(BASE_DIR/'figures',notebook='02_Experimental_Development',run_id=RUN_ID)
print('New development run:',RUN_ID,'| Original production budgets:',protocol())


<a id="s-2-2"></a>

### 2.2. Dataset provenance, schema and byte integrity

**Research trace:** RQ1. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 2.2.


#### 2.2.1. Purpose

Establish that the observations are the intended public PMSM measurements before engineering features.


#### 2.2.2. Mathematical basis and assumptions

One row is one 0.5-s sample within an ordered recording. There are 1,330,816 rows in 69 profiles. Elapsed time within profile is the local row index divided by 2 Hz; the CSV has no absolute timestamp.


#### 2.2.3. Implementation and parameter choices

resolve_csv supports a local original CSV or a public depositor download. validate_dataset checks columns, finiteness and chronology blocks; verify_csv_bytes requires the exact documented SHA-256. See docs/DATA_DICTIONARY.md for units and exclusions.


#### 2.2.4. Results and verification evidence

Printed shape, profile counts and SHA-256 are input checks, not predictive-accuracy evidence. Tests test_data.py exercise valid and invalid schemas plus optional original-data integration.


#### 2.2.5. Interpretation and limitations

Do not reorder or re-save the CSV to bypass the exact-byte check. Sensor calibration uncertainty and motor-to-motor variability are not identifiable from these files.


In [ ]:
CSV_PATH=resolve_csv(PROJECT_ROOT)
csv_path=str(CSV_PATH)
df=pd.read_csv(CSV_PATH)
print(validate_dataset(df))


In [ ]:
DATASET_SHA256_EXPECTED=EXPECTED_SHA256
DATASET_SHA256_ACTUAL=verify_csv_bytes(CSV_PATH)
print('Verified original CSV SHA-256:',DATASET_SHA256_ACTUAL)


## 3. Data integrity and the fixed profile split


<a id="s-3-1"></a>

### 3.1. Profile statistics and data-quality checks

**Research trace:** RQ1. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 3.1.


#### 3.1.1. Purpose

Identify recording lengths, target ranges and data-quality issues that affect leakage and interpretation.


#### 3.1.2. Mathematical basis and assumptions

A session maximum is max(T) within a profile. Consecutive profile blocks must represent exactly one chronological block per ID. Adjacent samples are dependent; row count is not the number of independent experiments.


#### 3.1.3. Implementation and parameter choices

Original descriptive summaries and block-continuity checks are retained. Only observable channels enter the model; target maxima are used retrospectively for stratification, as in the supplied split procedure.


#### 3.1.4. Results and verification evidence

Profile summaries, missing-value counts and repeated-block checks describe the raw source. Tests verify rejection of missing, nonnumeric and nonfinite inputs.


#### 3.1.5. Interpretation and limitations

Exploring the complete source for the established split is disclosed. No held-out prediction error is used to select a model.


In [ ]:
profile_summary = (
    df.groupby("profile_id")
    .agg(
        observations=("pm", "size"),
        pm_min=("pm", "min"),
        pm_max=("pm", "max")
    )
    .reset_index()
)

profile_summary["duration_hours"] = (
    profile_summary["observations"] / (2 * 3600)
)

display(profile_summary)


In [ ]:
# Identify consecutive blocks of profile IDs
blocks = df["profile_id"].ne(
    df["profile_id"].shift()
).cumsum()

# Count distinct continuous blocks
n_blocks = blocks.nunique()
n_profiles = df["profile_id"].nunique()

print("Measurement profiles:", n_profiles)
print("Continuous blocks:", n_blocks)

if n_blocks == n_profiles:
    print("PASS: Each profile appears in one continuous block.")
else:
    print("CHECK: Some profiles may appear in multiple blocks.")

# Check missing values and duplicate rows
print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


<a id="s-3-2"></a>

### 3.2. Row-split diagnostic and profile-disjoint assignment

**Research trace:** RQ1. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 3.2.


#### 3.2.1. Purpose

Explain why a naïve chronological row cutoff can share a recording between development and evaluation, then retain the established grouped split.


#### 3.2.2. Mathematical basis and assumptions

The diagnostic cutoff is floor(0.8N). The adopted allocation instead operates on 69 profile IDs with three target-maximum strata and seed 42. It yields 55 development profiles (1,055,198 rows) and 14 held-out profiles (275,618 rows).


#### 3.2.3. Implementation and parameter choices

The original row-cutoff demonstration precedes the stratified profile allocation. Code verifies exact locked IDs, no overlap, profile counts and row counts. It does not fit a model on the row-cutoff split.


#### 3.2.4. Results and verification evidence

The displayed allocations and recording-duration figure establish partition identity. test_sampling_validation.py and original-data tests independently check the locked allocation.


#### 3.2.5. Interpretation and limitations

Historical project iterations already inspected held-out scores. This fixed retrospective partition is not a pristine confirmatory test of the entire project. The corrected development computations never optimise on its performance.


In [ ]:
cutoff = int(len(df) * 0.80)

train = df.iloc[:cutoff]
test = df.iloc[cutoff:]

shared_profiles = (
    set(train["profile_id"])
    & set(test["profile_id"])
)

print("Training rows:", len(train))
print("Testing rows:", len(test))
print("Profiles appearing in both:", shared_profiles)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# STEP 4A: Summarise each measurement profile
profile_table = (
    df.groupby("profile_id")
    .agg(
        observations=("pm", "size"),
        pm_min=("pm", "min"),
        pm_max=("pm", "max"),
        pm_mean=("pm", "mean")
    )
    .reset_index()
    .sort_values("profile_id")
    .reset_index(drop=True)
)

# STEP 4B: Stratify profiles by maximum PM temperature
# Three approximately equal-sized temperature groups
profile_table["temperature_group"] = pd.qcut(
    profile_table["pm_max"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# STEP 4C: Split complete profiles, NOT individual rows
train_ids, test_ids = train_test_split(
    profile_table["profile_id"],
    test_size=0.20,
    random_state=42,
    stratify=profile_table["temperature_group"]
)

train_ids = sorted(train_ids.tolist())
test_ids = sorted(test_ids.tolist())


In [ ]:
# STEP 4D: Create development and final test datasets
train_df = df[df["profile_id"].isin(train_ids)].copy()
test_df = df[df["profile_id"].isin(test_ids)].copy()

# STEP 4E: Verify no profile overlap
assert set(train_ids).isdisjoint(set(test_ids))
assert len(train_df) + len(test_df) == len(df)

print("Development profiles:", len(train_ids))
print("Test profiles:", len(test_ids))
print("Development observations:", len(train_df))
print("Test observations:", len(test_df))

print("\nActual proportions:")
print(f"Development: {len(train_df)/len(df)*100:.2f}%")
print(f"Final test:  {len(test_df)/len(df)*100:.2f}%")

print("\nFinal test profile IDs:", test_ids)
print("Profile overlap:", set(train_ids) & set(test_ids))


In [ ]:
# Assign split labels to profiles
profile_table["split"] = profile_table["profile_id"].apply(
    lambda pid: "Test" if pid in test_ids else "Development"
)

# Count low/medium/high temperature profiles in each set
print("Temperature-group distribution:")
display(
    pd.crosstab(
        profile_table["split"],
        profile_table["temperature_group"]
    )
)

# Compare measurement coverage
for name, data in [("Development", train_df), ("Test", test_df)]:
    print(f"\n{name} set:")
    print("Profiles:", data["profile_id"].nunique())
    print("PM range (°C):", data["pm"].min(), "to", data["pm"].max())
    print("Motor speed range:", data["motor_speed"].min(),
          "to", data["motor_speed"].max())
    print("Torque range:", data["torque"].min(),
          "to", data["torque"].max())

# Save profile assignments for reproducibility
profile_table.to_csv("profile_split.csv", index=False)


## 4. Physical inputs and causal thermal-memory features


<a id="s-4-1"></a>

### 4.1. Measured and derived operating-state inputs

**Research trace:** RQ1. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 4.1.


#### 4.1.1. Purpose

Represent electrical loading, speed and available thermal conditions using the original 12 instantaneous inputs.


#### 4.1.2. Mathematical basis and assumptions

Seven measured predictors are ambient and coolant temperature (°C), direct/quadrature voltages $u_d,u_q$ (V), direct/quadrature currents $i_d,i_q$ (A), and speed $n$ (rpm). The five derived predictors are
$$u_s=\sqrt{u_d^2+u_q^2},\quad i_s=\sqrt{i_d^2+i_q^2},\quad S_{el}=1.5u_si_s,\quad i_s\omega,\quad S_{el}\omega,\qquad \omega=2\pi n/60.$$
$S_{el}$ has apparent-power units VA; its speed products are empirical operating-state proxies, not measured rotor heat-generation rates. The target is the measured `pm` temperature $T$ in °C. Stator-temperature channels and torque are retained in the source CSV but excluded from the original seven-input model [KWB21].


#### 4.1.3. Implementation and parameter choices

make_paper_inputs maps source names to the benchmark-style names and calculates five derived quantities. The reusable function is imported by both notebooks. Units remain physical until fold-local scaling.


#### 4.1.4. Results and verification evidence

Feature names and values are checked against explicit magnitude/product identities in test_features.py and against the uploaded implementation in test_original_parity.py.


#### 4.1.5. Interpretation and limitations

These features encode plausible thermal influence but do not decompose copper, iron and mechanical losses or establish causation.


In [ ]:
from pmsm_project.features import make_paper_inputs


<a id="s-4-2"></a>

### 4.2. Causal EWMA/EWMS implementation

**Research trace:** RQ1. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 4.2.


#### 4.2.1. Purpose

Represent slow thermal response without using future samples or carrying state between independent recordings.


#### 4.2.2. Mathematical basis and assumptions

For a span $s$ measured in samples, $\alpha=2/(s+1)$ and $w_{t,j}=(1-\alpha)^{t-j}$ for $j\leq t$. The adjusted mean and biased standard deviation are
$$\mu_t=\frac{\sum_{j=0}^{t}w_{t,j}x_j}{\sum_{j=0}^{t}w_{t,j}},\qquad
\sigma_t=\sqrt{\frac{\sum_{j=0}^{t}w_{t,j}(x_j-\mu_t)^2}{\sum_{j=0}^{t}w_{t,j}}}.$$
Pandas `adjust=True` and `std(bias=True)` implement these moments [PANDAS]. Thus $\mu_0=x_0$ and $\sigma_0=0$. Four spans for each of 12 inputs give $12+2(12)(4)=108$ features. Filters reset at every recording and use full original chronological histories before selecting rows. A span of 200 samples corresponds to 100 s of sample count at 2 Hz; an exponential filter has infinite decaying support rather than a hard 100-s window.


#### 4.2.3. Implementation and parameter choices

make_temporal_features constructs the original adjusted moments. Feature history is calculated before the original evenly spaced or nested row sampling. Model-specific spans are selected by development CV later; the preliminary demonstration spans are invariant diagnostic inputs.


#### 4.2.4. Results and verification evidence

The first-row limits, feature count, source row alignment, profile-reset behaviour and prefix invariance are tested. Altering future data must not change an earlier feature.


#### 4.2.5. Interpretation and limitations

Causal features eliminate future-feature leakage; they do not make a model physically causal or remove dependence among rows within a recording.


In [ ]:
from pmsm_project.features import make_temporal_features


In [ ]:
# Test 1: Compare EWMA/EWMS against manual calculations

signal = np.array([10.0, 20.0, 30.0])
span = 3
alpha = 2 / (span + 1)

weights = (1 - alpha) ** np.arange(2, -1, -1)

manual_mean = np.sum(weights * signal) / weights.sum()

manual_std = np.sqrt(
    np.sum(weights * (signal - manual_mean)**2)
    / weights.sum()
)

pandas_ewm = pd.Series(signal).ewm(
    span=span, adjust=True
)

np.testing.assert_allclose(
    pandas_ewm.mean().iloc[-1], manual_mean
)

np.testing.assert_allclose(
    pandas_ewm.std(bias=True).iloc[-1],
    manual_std
)

print("PASS: EWMA/EWMS mathematical verification")
print("Manual EWMA:", manual_mean)
print("Manual EWMS:", manual_std)


In [ ]:
# Test 2: Check that memory resets between profiles

toy_X = pd.DataFrame({
    "sensor": [10., 20., 30., 100., 100.]
})

toy_ids = pd.Series([1, 1, 1, 2, 2])

toy_features = make_temporal_features(
    toy_X, toy_ids, spans=[3]
)

assert toy_features.loc[3, "sensor_ewma_s3"] == 100.
assert toy_features.loc[3, "sensor_ewms_s3"] == 0.

print("PASS: Profile memory reset")


# Test 3: Verify that future values do not affect past features

modified_X = toy_X.copy()
modified_X.loc[2, "sensor"] = 100000.

modified_features = make_temporal_features(
    modified_X, toy_ids, spans=[3]
)

np.testing.assert_allclose(
    toy_features.loc[:1, "sensor_ewma_s3"],
    modified_features.loc[:1, "sensor_ewma_s3"]
)

print("PASS: No future-data leakage")


<a id="s-4-3"></a>

### 4.3. Numerical and visual verification of temporal memory

**Research trace:** RQ1, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 4.3.


#### 4.3.1. Purpose

Check the feature implementation independently before optimising any spans.


#### 4.3.2. Mathematical basis and assumptions

A manual weighted sum supplies an independent reference for the adjusted moments. Constant input must retain its constant mean and zero weighted standard deviation; appending later samples must leave the earlier prefix invariant.


#### 4.3.3. Implementation and parameter choices

The original synthetic and real-data checks and EWMA plot are retained. Tests compare both reusable feature paths in the exact production feature order.


#### 4.3.4. Results and verification evidence

Assertion outputs establish implementation identities; the plotted current response illustrates shorter versus longer memory. The plot is saved with its chapter and source-cell trace.


#### 4.3.5. Interpretation and limitations

Passing a numerical feature check is verification of an equation’s implementation, not validation of temperature-prediction accuracy.


In [ ]:
# Select two development profiles only
demo_ids = train_ids[:2]

demo_df = (
    train_df[train_df["profile_id"].isin(demo_ids)]
    .groupby("profile_id", sort=False)
    .head(300)
    .copy()
)

# Construct the twelve input variables
X_demo = make_paper_inputs(demo_df)

# Illustrative spans (NOT yet optimised)
example_spans = [10, 50, 200, 1000]

# Generate temporal features
X_temporal = make_temporal_features(
    X_demo,
    demo_df["profile_id"],
    spans=example_spans
)

# Verify feature matrix
assert X_temporal.shape[1] == 108
assert X_temporal.isna().sum().sum() == 0
assert np.isfinite(X_temporal.to_numpy()).all()

print("Profiles:", demo_df["profile_id"].nunique())
print("Original inputs:", X_demo.shape[1])
print("Total features:", X_temporal.shape[1])
print("Missing values:", X_temporal.isna().sum().sum())

display(X_temporal.iloc[:5, :16])


In [ ]:
from pmsm_project.plotting import set_figure_context
set_figure_context(section='4.3', source_notebook='fresh_reconstruction.ipynb', source_cell=30)


In [ ]:
import matplotlib.pyplot as plt

# Examine one measurement profile
pid = demo_ids[0]

mask = demo_df["profile_id"] == pid
time = np.arange(mask.sum()) / 2  # Seconds at 2 Hz

plt.figure(figsize=(11, 5))

plt.plot(
    time,
    X_temporal.loc[mask, "i_q"].to_numpy(),
    label="Raw q-axis current"
)

for span in [10, 50, 200,1000]:
    plt.plot(
        time,
        X_temporal.loc[
            mask, f"i_q_ewma_s{span}"
        ].to_numpy(),
        label=f"EWMA span={span}"
    )

plt.xlabel("Elapsed time (s)")
plt.ylabel("q-axis current (A)")
plt.title(f"EWMA verification — Profile {pid}")
plt.legend()
plt.grid(True)
plt.show()


## 5. Grouped validation and preliminary numerical investigations


<a id="s-5-1"></a>

### 5.1. Stratified grouped three-fold validation

**Research trace:** RQ1, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 5.1.


#### 5.1.1. Purpose

Define the fixed development-only validation mechanism used by all later objectives.


#### 5.1.2. Mathematical basis and assumptions

Each profile belongs to one validation fold. Its other two folds provide the fitting profiles. For fold $f$,
$$MSE_f=\frac1{N_f}\sum_{i\in f}(\hat T_i-T_i)^2,\qquad J=\frac13\sum_{f=1}^{3}MSE_f.$$
The objective $J$ has units °C² and weights folds equally. It is different from pooling every validation row before calculating MSE. Feature and target standardisers are fitted only on the fitting profiles and predictions are inverse-transformed before scoring. Profile-level temperature maxima define three strata for the fixed shuffled `StratifiedKFold(3, random_state=42)` assignment [ROBERTS17].


#### 5.1.3. Implementation and parameter choices

The original allocation and fold audit are run in chronological source order. Whole profiles are assigned once; row-level fitting/validation masks are derived from that profile lookup. The split is reused across candidates.


#### 5.1.4. Results and verification evidence

Fold distributions, profile counts and explicit no-overlap assertions are generated. The later manifest records the complete profile-to-fold map.


#### 5.1.5. Interpretation and limitations

Three folds quantify variation across these recordings only. Fold MSE values share training profiles and are not independent repeated trials.


In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# =====================================================
# STEP 6.1A — Restore the existing development/test split
# =====================================================

# df is already loaded from KaggleHub

test_ids = [
    13, 16, 23, 24, 29, 42, 44,
    47, 53, 56, 62, 67, 71, 78
]

train_df = df[
    ~df["profile_id"].isin(test_ids)
].copy()

test_df = df[
    df["profile_id"].isin(test_ids)
].copy()

# Verify the previously established partition
assert len(df) == 1330816
assert len(train_df) == 1055198
assert len(test_df) == 275618
assert set(train_df["profile_id"]).isdisjoint(
    set(test_df["profile_id"])
)

print("PASS: Development/test split restored")


In [ ]:
# =====================================================
# STEP 6.1B — Summarise development profiles
# =====================================================

profile_cv = (
    train_df.groupby("profile_id")["pm"]
    .agg(
        observations="size",
        pm_min="min",
        pm_max="max"
    )
    .reset_index()
    .sort_values("profile_id")
    .reset_index(drop=True)
)

# Create three temperature categories based
# on each profile's maximum PM temperature
profile_cv["temperature_group"] = pd.qcut(
    profile_cv["pm_max"],
    q=3,
    labels=["Low", "Medium", "High"]
)


# =====================================================
# STEP 6.1C — Construct three stratified folds
# =====================================================

skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

profile_cv["cv_fold"] = -1


In [ ]:
for fold, (_, val_indices) in enumerate(
    skf.split(
        profile_cv[["pm_max"]],
        profile_cv["temperature_group"]
    ),
    start=1
):
    profile_cv.loc[val_indices, "cv_fold"] = fold

print("PASS: Three validation folds created")


# =====================================================
# STEP 6.1D — Verify profile separation
# =====================================================

development_ids = set(profile_cv["profile_id"])

assert len(development_ids) == 55
assert profile_cv["cv_fold"].isin([1, 2, 3]).all()
assert development_ids.isdisjoint(set(test_ids))


In [ ]:
for fold in [1, 2, 3]:

    val_ids = set(
        profile_cv.loc[
            profile_cv["cv_fold"] == fold,
            "profile_id"
        ]
    )

    fold_train_ids = development_ids - val_ids

    assert val_ids.isdisjoint(fold_train_ids)
    assert val_ids.isdisjoint(set(test_ids))

    print(f"\nFold {fold}")
    print("Training profiles:", len(fold_train_ids))
    print("Validation profiles:", len(val_ids))

print("\nPASS: No profile leakage detected")

# Save assignments for reproducibility
profile_cv.to_csv(
    "profile_cv_assignments.csv",
    index=False
)

print("Saved: profile_cv_assignments.csv")


In [ ]:
# =====================================================
# STEP 6.1E — Verify temperature-group distribution
# =====================================================

print("Temperature-group distribution:")

display(
    pd.crosstab(
        profile_cv["cv_fold"],
        profile_cv["temperature_group"]
    )
)


# =====================================================
# STEP 6.1F — Verify fold sizes and coverage
# =====================================================

fold_summary = (
    profile_cv.groupby("cv_fold")
    .agg(
        profiles=("profile_id", "size"),
        observations=("observations", "sum"),
        min_pm=("pm_min", "min"),
        max_pm=("pm_max", "max")
    )
)

# The original dataset is sampled at 2 Hz
fold_summary["duration_hours"] = (
    fold_summary["observations"] / 7200
)

print("\nValidation-fold summary:")
display(fold_summary)


In [ ]:
# Confirm that every development observation is assigned
assert fold_summary["observations"].sum() == len(train_df)
assert fold_summary["profiles"].sum() == 55

print("\nPASS: All development observations assigned")


<a id="s-5-2"></a>

### 5.2. Preliminary estimator interface checks

**Research trace:** RQ1, RQ2, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 5.2.


#### 5.2.1. Purpose

Verify that the original pipelines construct, fit and predict consistently before committing to expensive searches.


#### 5.2.2. Mathematical basis and assumptions

StandardScaler transforms x to (x−μ)/σ using fitting data. TransformedTargetRegressor applies an analogous target transform and returns predictions in °C. This interface is distinct from the raw-target preliminary OLS solver.


#### 5.2.3. Implementation and parameter choices

Retain the supplied model prototypes and small preliminary datasets as interface investigations. These initial settings are documented experimental inputs and do not bypass later Bayesian selection.


#### 5.2.4. Results and verification evidence

Finite predictions, sample counts and preliminary scores expose construction errors. Automated tests separately check all actual factories and estimator serialization.


#### 5.2.5. Interpretation and limitations

A working prototype does not establish an optimal model. Preliminary results do not become final configurations unless the subsequent search selects them afresh.


In [ ]:
import numpy as np
import pandas as pd

# =====================================================
# STEP 6.2A — Prepare a small development-data sample
# =====================================================

# These are example spans, not optimised values.
SMOKE_SPANS = [10, 50, 200, 1000]
SAMPLES_PER_PROFILE = 80


In [ ]:
def make_pipeline_test_data(data, spans, samples_per_profile=80):
    """Build the original preliminary profile-sampled feature table.
    
    Parameters
    ----------
    data : pandas.DataFrame
        Raw chronological source rows; target pm in °C.
    spans : sequence[int]
        Four ordered temporal spans in original 2-Hz samples.
    samples_per_profile : int, default 80
        Original preliminary row-selection budget per profile; not a new sampling rate.
    
    Returns
    -------
    pandas.DataFrame
        Sampled features, temperature target in °C and recording IDs.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    sampled_profiles = []

    for pid, profile in data.groupby(
        "profile_id", sort=False
    ):
        # Construct the 12 original input variables
        X = make_paper_inputs(profile)

        # Calculate temporal features from full history
        X_full = make_temporal_features(
            X, profile["profile_id"], spans
        )

        # Select representative positions across this profile
        n = min(samples_per_profile, len(profile))

        positions = np.linspace(
            0, len(profile) - 1, n, dtype=int
        )

        sample = X_full.iloc[positions].copy()

        # Target is stored separately from predictors
        sample["pm_target"] = (
            profile["pm"].iloc[positions].to_numpy()
        )
        sample["profile_id"] = pid

        sampled_profiles.append(sample)

    return pd.concat(
        sampled_profiles, ignore_index=True
    )


In [ ]:
smoke_df = make_pipeline_test_data(
    train_df,
    SMOKE_SPANS,
    samples_per_profile=SAMPLES_PER_PROFILE
)

# Verification
assert smoke_df["profile_id"].nunique() == 55
assert len(smoke_df) == 55 * SAMPLES_PER_PROFILE
assert smoke_df.shape[1] == 110  # 108 + target + profile ID
assert smoke_df.isna().sum().sum() == 0
assert np.isfinite(smoke_df.to_numpy()).all()

print("PASS: Pipeline test dataset prepared")
print("Observations:", len(smoke_df))
print("Predictor features:", smoke_df.shape[1] - 2)
print("Profiles:", smoke_df["profile_id"].nunique())


In [ ]:
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# =====================================================
# STEP 6.2B — Select the first grouped CV fold
# =====================================================

val_ids = set(
    profile_cv.loc[
        profile_cv["cv_fold"] == 1,
        "profile_id"
    ]
)

train_sample = smoke_df[
    ~smoke_df["profile_id"].isin(val_ids)
].copy()

val_sample = smoke_df[
    smoke_df["profile_id"].isin(val_ids)
].copy()


In [ ]:
# Separate inputs and target
feature_cols = [
    c for c in smoke_df.columns
    if c not in ["pm_target", "profile_id"]
]

X_train = train_sample[feature_cols]
y_train = train_sample["pm_target"]

X_val = val_sample[feature_cols]
y_val = val_sample["pm_target"]

# Verify no profile overlap
assert set(train_sample["profile_id"]).isdisjoint(
    set(val_sample["profile_id"])
)

assert len(feature_cols) == 108

print("Training profiles:",
      train_sample["profile_id"].nunique())
print("Validation profiles:",
      val_sample["profile_id"].nunique())


In [ ]:
# =====================================================
# STEP 6.2C — Define the six regression models
# =====================================================

models = {
    "OLS": LinearRegression(),

    "SVR": SVR(
        kernel="rbf",
        C=1.0,
        epsilon=0.1
    ),

    "k-NN": KNeighborsRegressor(
        n_neighbors=10,
        weights="distance"
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=20,
        max_depth=12,
        random_state=42,
        n_jobs=2
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=20,
        max_depth=12,
        random_state=42,
        n_jobs=2
    ),

    "MLP": MLPRegressor(
        hidden_layer_sizes=(32,),
        max_iter=150,
        random_state=42
    )
}


In [ ]:
# =====================================================
# STEP 6.2D — Train, predict and evaluate each model
# =====================================================

results = []


In [ ]:
for name, estimator in models.items():

    # StandardScaler is fitted on TRAINING data only.
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", estimator)
    ])

    # Train
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    training_time = time.perf_counter() - start

    # Predict on unseen validation profiles
    start = time.perf_counter()
    predictions = pipeline.predict(X_val)
    inference_time = time.perf_counter() - start

    # Calculate prediction errors
    errors = y_val.to_numpy() - predictions

    results.append({
        "Model": name,
        "MSE": mean_squared_error(y_val, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_val, predictions)),
        "MAE": mean_absolute_error(y_val, predictions),
        "R2": r2_score(y_val, predictions),
        "Max Error": np.max(np.abs(errors)),
        "Training Time (s)": training_time,
        "Inference Time (s)": inference_time
    })

    print(f"PASS: {name} trained and evaluated")

# Display comparison
results_df = pd.DataFrame(results)


In [ ]:
display(results_df.round(4))

# Save preliminary results
results_df.to_csv(
    "model_pipeline_smoke_test.csv",
    index=False
)

print("\nPASS: All six regression pipelines completed")


<a id="s-5-3"></a>

### 5.3. Preliminary OLS error localisation

**Research trace:** RQ2, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 5.3.


#### 5.3.1. Purpose

Investigate unusually large residuals and profile-start predictions before interpreting the preliminary OLS error.


#### 5.3.2. Mathematical basis and assumptions

For residual $e_i=\hat T_i-T_i$, profile MSE is the mean of $e_i^2$. Large coefficients in a correlated temporal design can cancel in predictions and are not directly interpretable physical coefficients.


#### 5.3.3. Implementation and parameter choices

The original OLS residual rankings, target/prediction comparisons and start-of-profile examination are preserved without changing training data.


#### 5.3.4. Results and verification evidence

The existing per-profile diagnostics and ranked residual tables are generated from current predictions. They are separate from later grouped-CV selection.


#### 5.3.5. Interpretation and limitations

Error localisation can identify model limitations; an outlier should not be removed merely because it is inconvenient for an accuracy table.


In [ ]:
# STEP 6.2E — Investigate OLS prediction outliers

ols_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

ols_pipeline.fit(X_train, y_train)

ols_predictions = ols_pipeline.predict(X_val)

ols_diagnostics = pd.DataFrame({
    "profile_id": val_sample["profile_id"].to_numpy(),
    "actual_pm": y_val.to_numpy(),
    "predicted_pm": ols_predictions
})

ols_diagnostics["absolute_error"] = abs(
    ols_diagnostics["actual_pm"]
    - ols_diagnostics["predicted_pm"]
)

print("Ten largest OLS prediction errors:")

display(
    ols_diagnostics.nlargest(
        10, "absolute_error"
    )
)

print(
    "\nMeasured PM temperature range:",
    y_val.min(), "to", y_val.max()
)


In [ ]:
print(
    "Predicted PM temperature range:",
    ols_predictions.min(),
    "to",
    ols_predictions.max()
)

print(
    "Maximum absolute error:",
    ols_diagnostics["absolute_error"].max()
)


<a id="s-5-4"></a>

### 5.4. Recording-start sensitivity

**Research trace:** RQ1, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 5.4.


#### 5.4.1. Purpose

Assess whether initial exponential-filter states coincide with large OLS errors.


#### 5.4.2. Mathematical basis and assumptions

At t=0 the adjusted EWMA equals the current value and EWMS is zero. This mathematical initialisation contains no pre-recording thermal history. A motor can nevertheless start a recording warm.


#### 5.4.3. Implementation and parameter choices

Retain the original early-row investigation and its residual comparisons. No new warm-up exclusion or target-dependent feature is introduced.


#### 5.4.4. Results and verification evidence

Start-of-recording tables and corresponding current-run residual diagnostics identify whether the observed issue is local to session beginnings.


#### 5.4.5. Interpretation and limitations

Coincidence with initialisation is not proof of a single physical cause. Earlier motor operation is unobserved; the production protocol retains all original rows.


In [ ]:
# STEP 6.2F — Investigate OLS errors at profile starts

# Identify the position of each sampled observation
# within its own validation profile.
ols_diagnostics["position_in_profile"] = (
    ols_diagnostics.groupby("profile_id").cumcount()
)

# Inspect the largest prediction errors
display(
    ols_diagnostics.nlargest(
        10, "absolute_error"
    )
)

# Inspect the first five sampled observations
# from the problematic profile
print("\nFirst five sampled observations of profile 65:")

display(
    ols_diagnostics[
        ols_diagnostics["profile_id"] == 65
    ].head(5)
)

# Compare errors at the first sample versus later samples
first_samples = ols_diagnostics[
    ols_diagnostics["position_in_profile"] == 0
]

later_samples = ols_diagnostics[
    ols_diagnostics["position_in_profile"] > 0
]

print("\nMean absolute error at profile starts:",
      first_samples["absolute_error"].mean())


In [ ]:
print("Mean absolute error at later observations:",
      later_samples["absolute_error"].mean())


<a id="s-5-5"></a>

### 5.5. OLS outlier influence and treatment decision

**Research trace:** RQ2, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 5.5.


#### 5.5.1. Purpose

Check how influential observations affect the preliminary linear fit while preserving the adopted unfiltered protocol.


#### 5.5.2. Mathematical basis and assumptions

Least squares penalises large residuals quadratically, so a small group of extreme residuals can influence a fit strongly. Diagnostic exclusion is a sensitivity experiment, not an automatic cleaning rule.


#### 5.5.3. Implementation and parameter choices

Execute the original comparison and retain its treatment decision. Any trial exclusion remains confined to this diagnostic experiment; later optimisation still uses the original specified datasets.


#### 5.5.4. Results and verification evidence

The sensitivity table compares the original diagnostic alternatives. It does not export an alternative final preprocessing rule.


#### 5.5.5. Interpretation and limitations

Removing a difficult state can improve an apparent score while weakening coverage of the motor’s operating conditions. Interpret the supplied diagnostic without changing the research protocol.


In [ ]:
# STEP 6.2G — Investigate the extreme prediction

# Examine raw measurements at the start of profile 65
print("First five raw measurements of profile 65:")

display(
    df.loc[df["profile_id"] == 65].head(5)
)

# Quantify influence of the largest error
squared_errors = (
    ols_diagnostics["actual_pm"]
    - ols_diagnostics["predicted_pm"]
) ** 2

worst_idx = squared_errors.idxmax()

outlier_contribution = (
    squared_errors.loc[worst_idx]
    / squared_errors.sum()
    * 100
)

print(
    "\nPercentage of total OLS squared error "
    "from the single largest prediction:",
    round(outlier_contribution, 2), "%"
)

# Diagnostic sensitivity calculation only
print(
    "\nValidation MSE with all observations:",
    squared_errors.mean()
)


In [ ]:
print(
    "Validation MSE excluding largest error:",
    squared_errors.drop(worst_idx).mean()
)


## 6. OLS Bayesian pilot and expanded search


<a id="s-6-1"></a>

### 6.1. Initial OLS Gaussian-process pilot

**Research trace:** RQ2, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 6.1.


#### 6.1.1. Purpose

Test whether temporal spans improve the original OLS representation using the first eight Bayesian evaluations.


#### 6.1.2. Mathematical basis and assumptions

Each profile belongs to one validation fold. Its other two folds provide the fitting profiles. For fold $f$,
$$MSE_f=\frac1{N_f}\sum_{i\in f}(\hat T_i-T_i)^2,\qquad J=\frac13\sum_{f=1}^{3}MSE_f.$$
The objective $J$ has units °C² and weights folds equally. It is different from pooling every validation row before calculating MSE. Feature and target standardisers are fitted only on the fitting profiles and predictions are inverse-transformed before scoring. Profile-level temperature maxima define three strata for the fixed shuffled `StratifiedKFold(3, random_state=42)` assignment [ROBERTS17].

A GP surrogate models the expensive objective. Expected improvement (EI) balances improvement and uncertainty; the original gp_minimize call uses eight evaluations, four initial points and seed 42 [SKOPT].


#### 6.1.3. Implementation and parameter choices

Retain initial independent integer ranges [5,30], [31,150], [151,700], [701,3000]. Each candidate processes full histories then selects 80 observations per development profile. evaluate_ols_cv is the original objective extracted into DevelopmentStudy.


#### 6.1.4. Results and verification evidence

trial_records contains eight fresh candidate rows and three fold MSE values per row. There are no supplied winners in the production selection path.


#### 6.1.5. Interpretation and limitations

Eight evaluations are a pilot, not evidence of the global minimum. Later expanded and full-data searches remain required.


In [ ]:
# STEP 6.3A — Bayesian optimisation setup



import time
import numpy as np
import pandas as pd

from skopt import gp_minimize
from skopt.space import Integer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Preliminary optimisation settings
SAMPLES_PER_PROFILE = 80
RANDOM_STATE = 42

# Use the exact CV assignments from Step 6.1
fold_lookup = profile_cv.set_index(
    "profile_id"
)["cv_fold"]

assert len(train_df) == 1055198
assert len(fold_lookup) == 55
assert set(fold_lookup.unique()) == {1, 2, 3}






print("PASS: Bayesian optimisation objective defined")

study=DevelopmentStudy(train_df,profile_cv)
build_candidate_features=study.build_candidate_features


In [ ]:
evaluate_ols_cv=study.evaluate_ols_cv


<a id="s-6-2"></a>

### 6.2. Review of pilot intervals and boundary evidence

**Research trace:** RQ2, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 6.2.


#### 6.2.1. Purpose

Retain the historical review of whether the pilot’s span bounds constrain the observed minimum.


#### 6.2.2. Mathematical basis and assumptions

The current incumbent is the candidate with smallest J. A boundary-adjacent candidate motivates the historical expanded domain, but does not establish that the true optimum lies outside the pilot space.


#### 6.2.3. Implementation and parameter choices

The original pilot tables and search-bound visualisation are executed before the expanded optimiser. Both the diagnostic and later expanded domains remain unchanged.


#### 6.2.4. Results and verification evidence

Current pilot rankings and interval plots are saved as development evidence, separately from historical interpretations stored in provenance.


#### 6.2.5. Interpretation and limitations

The expanded search is part of the original exploratory sequence. The narrative must be reassessed if a fresh pilot chooses different spans.


In [ ]:
# STEP 6.3B — Verify Bayesian objective function

baseline_spans = [10, 50, 200, 1000]

baseline_mse, baseline_folds = evaluate_ols_cv(
    baseline_spans
)

print("Example spans:", baseline_spans)

for fold, mse in enumerate(baseline_folds, start=1):
    print(f"Fold {fold} MSE: {mse:.4f} °C²")

print(f"\nAverage CV MSE: {baseline_mse:.4f} °C²")

# Verify against the previously observed result
np.testing.assert_allclose(
    baseline_mse,
    64.6399,
    rtol=1e-3
)

print("PASS: OLS CV objective reproduces baseline")


In [ ]:
# STEP 6.3C — Gaussian-process Bayesian optimisation

search_space = [
    Integer(5, 30, name="span_1"),
    Integer(31, 150, name="span_2"),
    Integer(151, 700, name="span_3"),
    Integer(701, 3000, name="span_4")
]

trial_records = []


In [ ]:
def bayesian_objective(candidate):
    """Evaluate the original OLS grouped-CV candidate and record its trial.
    
    Parameters
    ----------
    candidate : sequence[int]
        Proposed four spans from the unchanged initial search dimensions.
    
    Returns
    -------
    float
        Unweighted mean of three fold MSE values, in °C².
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    spans = [int(s) for s in candidate]

    start = time.perf_counter()

    mean_mse, fold_scores = evaluate_ols_cv(spans)

    duration = time.perf_counter() - start

    record = {
        "trial": len(trial_records) + 1,
        "s1": spans[0],
        "s2": spans[1],
        "s3": spans[2],
        "s4": spans[3],
        "fold1_mse": fold_scores[0],
        "fold2_mse": fold_scores[1],
        "fold3_mse": fold_scores[2],
        "mean_cv_mse": mean_mse,
        "runtime_s": duration
    }

    trial_records.append(record)

    print(
        f"Trial {record['trial']}: "
        f"spans={spans}, "
        f"CV MSE={mean_mse:.4f}, "
        f"time={duration:.1f}s",
        flush=True
    )

    return mean_mse


In [ ]:
# Small initial search to verify optimisation
result = gp_minimize(
    func=bayesian_objective,
    dimensions=search_space,
    n_calls=8,
    n_initial_points=4,
    acq_func="EI",
    random_state=RANDOM_STATE
)

pd.DataFrame(trial_records).to_csv(
    "bayesian_ols_pilot_trials.csv", index=False
)
print("\nPASS: OLS Bayesian optimisation pilot completed")
print("Best spans:", result.x)
print(f"Best CV MSE: {result.fun:.4f} °C²")

display(
    pd.DataFrame(trial_records)
    .sort_values("mean_cv_mse")
    .round(4)
)


In [ ]:
from pmsm_project.plotting import set_figure_context
set_figure_context(section='6.2', source_notebook='fresh_reconstruction.ipynb', source_cell=54)


In [ ]:
# STEP 6.3D — Visualise Bayesian search progress

import matplotlib.pyplot as plt

history = pd.DataFrame(trial_records)

history["best_so_far"] = (
    history["mean_cv_mse"].cummin()
)

plt.figure(figsize=(10, 5))

plt.plot(
    history["trial"],
    history["mean_cv_mse"],
    "o-",
    label="Candidate CV MSE"
)

plt.plot(
    history["trial"],
    history["best_so_far"],
    "s-",
    label="Best MSE so far"
)

plt.axhline(
    baseline_mse,
    linestyle="--",
    label="Fixed-span baseline"
)

plt.xlabel("Bayesian optimisation trial")
plt.ylabel("Mean 3-fold CV MSE (°C²)")
plt.title("OLS Bayesian optimisation — Pilot")
plt.legend()
plt.grid(True)


In [ ]:
plt.show()


<a id="s-6-3"></a>

### 6.3. Expanded OLS ask/tell search

**Research trace:** RQ2, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 6.3.


#### 6.3.1. Purpose

Continue the original OLS investigation in an ordered, wider span domain without replacing Bayesian optimisation by a grid.


#### 6.3.2. Mathematical basis and assumptions

The expanded temporal dimensions are first span s1 ∈ [5, 100] and positive gaps g2 ∈ [1, 400], g3 ∈ [1, 1500], g4 ∈ [1, 6000], all integer. The actual spans are (s1, s1+g2, s1+g2+g3, s1+g2+g3+g4). These are fixed search bounds, not previously selected spans.

Each profile belongs to one validation fold. Its other two folds provide the fitting profiles. For fold $f$,
$$MSE_f=\frac1{N_f}\sum_{i\in f}(\hat T_i-T_i)^2,\qquad J=\frac13\sum_{f=1}^{3}MSE_f.$$
The objective $J$ has units °C² and weights folds equally. It is different from pooling every validation row before calculating MSE. Feature and target standardisers are fitted only on the fitting profiles and predictions are inverse-transformed before scoring. Profile-level temperature maxima define three strata for the fixed shuffled `StratifiedKFold(3, random_state=42)` assignment [ROBERTS17].


#### 6.3.3. Implementation and parameter choices

Retain Optimizer(base_estimator="GP", n_initial_points=10, acq_func="EI", random_state=42), seed it with the eight current pilot observations and evaluate 20 new proposals. The original acquisition optimiser default is retained. The objective still uses 80 observations per profile.


#### 6.3.4. Results and verification evidence

expanded_df contains 20 new evaluations; combined records compare all 28 current-run OLS candidates. The incumbent supplies the subsequent feasibility/QR seed, never a hardcoded old optimum.


#### 6.3.5. Interpretation and limitations

A minimum of sampled CV scores is subject to selection bias. This stage selects a promising span configuration under its sampling design; it does not certify full-data behaviour.


In [ ]:
# STEP 6.3B.1 — Expanded Bayesian search setup

import os
import time
import numpy as np
import pandas as pd

from skopt import Optimizer
from skopt.space import Integer

PILOT_PATH = "bayesian_ols_pilot_trials.csv"
EXPANDED_PATH = "bayesian_ols_expanded_trials.csv"

# New search space: first span + three positive gaps
expanded_space = [
    Integer(5, 100, name="s1"),
    Integer(1, 400, name="gap_2"),
    Integer(1, 1500, name="gap_3"),
    Integer(1, 6000, name="gap_4")
]





# Previous eight trials are generated earlier in THIS Run All execution.
# Deliberately ignore files from old runtimes.
pilot_df = pd.DataFrame(trial_records).copy()
pilot_df.to_csv(PILOT_PATH, index=False)

assert len(pilot_df) == 8


In [ ]:
# Confirm all old trials fit in the expanded space
for _, row in pilot_df.iterrows():
    old_spans = [
        int(row["s1"]), int(row["s2"]),
        int(row["s3"]), int(row["s4"])
    ]

    params = encode_spans(old_spans)

    assert all(
        dimension.low <= value <= dimension.high
        for dimension, value in zip(expanded_space, params)
    )

print("PASS: All eight previous trials are reusable")

print("\nPrevious best configuration:")
best_old = pilot_df.loc[
    pilot_df["mean_cv_mse"].idxmin()
]

print("Spans:", [
    int(best_old[f"s{i}"]) for i in range(1, 5)
])
print("CV MSE:", best_old["mean_cv_mse"])

from pmsm_project.search import decode_spans, encode_spans


In [ ]:
# STEP 6.3B.2 — Continue Bayesian optimisation

N_ADDITIONAL_TRIALS = 20

# Always perform a fresh expanded search after the freshly generated pilot.
expanded_df = pd.DataFrame(columns=pilot_df.columns)

# Reconstruct the optimiser from recorded results
optimizer = Optimizer(
    dimensions=expanded_space,
    base_estimator="GP",
    n_initial_points=10,
    acq_func="EI",
    random_state=42
)

existing_history = pd.concat(
    [pilot_df, expanded_df],
    ignore_index=True
)

# Reuse all completed evaluations
for _, row in existing_history.iterrows():

    spans = [
        int(row["s1"]), int(row["s2"]),
        int(row["s3"]), int(row["s4"])
    ]

    parameters = encode_spans(spans)

    optimizer.tell(
        parameters,
        float(row["mean_cv_mse"])
    )

print("Existing evaluations:", len(existing_history))


In [ ]:
remaining = max(
    0,
    N_ADDITIONAL_TRIALS - len(expanded_df)
)

print("Additional evaluations remaining:", remaining)


In [ ]:
# Search for new configurations
for _ in range(remaining):

    parameters = optimizer.ask()
    spans = decode_spans(parameters)

    start = time.perf_counter()

    mean_mse, fold_scores = evaluate_ols_cv(spans)

    duration = time.perf_counter() - start

    optimizer.tell(parameters, mean_mse)

    record = {
        "trial": len(pilot_df) + len(expanded_df) + 1,
        "s1": spans[0],
        "s2": spans[1],
        "s3": spans[2],
        "s4": spans[3],
        "fold1_mse": fold_scores[0],
        "fold2_mse": fold_scores[1],
        "fold3_mse": fold_scores[2],
        "mean_cv_mse": mean_mse,
        "runtime_s": duration
    }

    expanded_df = pd.concat(
        [expanded_df, pd.DataFrame([record])],
        ignore_index=True
    )

    print(
        f"Trial {record['trial']}: "
        f"spans={spans}, "
        f"MSE={mean_mse:.4f}, "
        f"time={duration:.1f}s",
        flush=True
    )


In [ ]:
expanded_df.to_csv(EXPANDED_PATH, index=False)
print("\nPASS: Expanded Bayesian pilot completed")

# Combine original and new results
all_trials = pd.concat(
    [pilot_df, expanded_df],
    ignore_index=True
)

best_result = all_trials.loc[
    all_trials["mean_cv_mse"].idxmin()
]

best_spans = [
    int(best_result[f"s{i}"]) for i in range(1, 5)
]

print("\nBest spans:", best_spans)
print(
    "Best 3-fold CV MSE:",
    round(float(best_result["mean_cv_mse"]), 4)
)

display(
    all_trials.sort_values("mean_cv_mse")
    .head(10)
    .round(4)
)


In [ ]:
from pmsm_project.plotting import set_figure_context
set_figure_context(section='6.3', source_notebook='fresh_reconstruction.ipynb', source_cell=58)


In [ ]:
# STEP 6.3B.3 — Compare original and expanded search

import matplotlib.pyplot as plt

history = all_trials.copy()

history["best_so_far"] = (
    history["mean_cv_mse"].cummin()
)

plt.figure(figsize=(11, 5))

plt.plot(
    history["trial"],
    history["mean_cv_mse"],
    "o-",
    label="Candidate CV MSE",
    alpha=0.7
)

plt.plot(
    history["trial"],
    history["best_so_far"],
    "s-",
    label="Best MSE so far"
)

# Original manually selected span configuration
plt.axhline(
    64.6399,
    linestyle="--",
    label="Fixed-span baseline"
)


In [ ]:
# Mark where the expanded search begins
plt.axvline(
    8.5,
    linestyle=":",
    color="grey",
    label="Expanded search begins"
)

plt.xlabel("Bayesian optimisation trial")
plt.ylabel("Mean 3-fold CV MSE (°C²)")
plt.title("OLS Bayesian Optimisation — Expanded Pilot")
plt.legend()
plt.grid(True)
plt.show()

print("Original pilot best MSE:",
      round(pilot_df["mean_cv_mse"].min(), 4))

print("Expanded search best MSE:",
      round(history["mean_cv_mse"].min(), 4))

print("Total evaluations:", len(history))


## 7. Five-model Bayesian pilots

The five families share one unchanged optimiser loop. These model sections explain its distinct mathematics and spaces before that loop runs. This avoids repeated code while keeping model choices explicit.


<a id="s-7-1"></a>

### 7.1. SVR

**Research trace:** RQ2, RQ3, RQ4. Original five-model pilot; source cells 65–70. Larger search: section 10.1.


#### 7.1.1. Purpose

Investigate SVR as a credible alternative under the same profile-grouped validation.


#### 7.1.2. Mathematical basis and assumptions

The RBF kernel is $k(x,z)=\exp(-\gamma\|x-z\|^2)$ and the epsilon-insensitive loss is $L_\epsilon(r)=\max(0,|r|-\epsilon)$. The primal objective is $\frac12\|w\|^2+C\sum_iL_\epsilon(z_i-f(x_i))$ [SMOLA04]. $C$ and $\epsilon$ apply to the standardised target in the supplied pipelines. `gamma='scale'` is inferred only from fitting inputs; fresh spans, $C$ and $\epsilon$ come from the original GP search.


#### 7.1.3. Implementation and parameter choices

C: log-uniform [0.001, 10]; epsilon: log-uniform [0.01, 0.3]. The expanded temporal dimensions are first span s1 ∈ [5, 100] and positive gaps g2 ∈ [1, 400], g3 ∈ [1, 1500], g4 ∈ [1, 6000], all integer. The actual spans are (s1, s1+g2, s1+g2+g3, s1+g2+g3+g4). These are fixed search bounds, not previously selected spans. Each model has eight GP ask/tell evaluations, five initial points, EI with sampling acquisition optimisation and seed 42. The pilot selects 80 observations per profile. MLP pilot max_iter=200 with early_stopping=False is retained.


#### 7.1.4. Results and verification evidence

The original loop writes all_results["SVR"], candidate JSON, three fold scores, means, timings and convergence counts. test_recovered_protocol.py compares the extracted objective arithmetic with the source. No pilot has been executed as part of this delivery certification.


#### 7.1.5. Interpretation and limitations

The pilot incumbent is freshly rescored when it seeds the later larger-sample search. Results are conditioned on this finite trial budget and sample selection. Final ranking must use the completed current-run evidence.


<a id="s-7-2"></a>

### 7.2. k-NN

**Research trace:** RQ2, RQ3, RQ4. Original five-model pilot; source cells 65–70. Larger search: section 10.2.


#### 7.2.1. Purpose

Investigate k-NN as a credible alternative under the same profile-grouped validation.


#### 7.2.2. Mathematical basis and assumptions

For the $k$ nearest standardised training states $\mathcal N_k(x)$, $\hat z(x)=\sum_{i\in\mathcal N_k}w_i z_i/\sum_iw_i$. Uniform weights have $w_i=1$; distance weights follow scikit-learn’s inverse-distance convention, including its zero-distance handling. $z=(T-\mu_T)/\sigma_T$ uses development/fold fitting statistics. Standardisation is essential because volts, amperes and rpm otherwise contribute incompatible Euclidean scales [SKLEARN].


#### 7.2.3. Implementation and parameter choices

n_neighbors: integer [1, 512]; weights: uniform or distance. The expanded temporal dimensions are first span s1 ∈ [5, 100] and positive gaps g2 ∈ [1, 400], g3 ∈ [1, 1500], g4 ∈ [1, 6000], all integer. The actual spans are (s1, s1+g2, s1+g2+g3, s1+g2+g3+g4). These are fixed search bounds, not previously selected spans. Each model has eight GP ask/tell evaluations, five initial points, EI with sampling acquisition optimisation and seed 42. The pilot selects 80 observations per profile. MLP pilot max_iter=200 with early_stopping=False is retained.


#### 7.2.4. Results and verification evidence

The original loop writes all_results["k-NN"], candidate JSON, three fold scores, means, timings and convergence counts. test_recovered_protocol.py compares the extracted objective arithmetic with the source. No pilot has been executed as part of this delivery certification.


#### 7.2.5. Interpretation and limitations

The pilot incumbent is freshly rescored when it seeds the later larger-sample search. Results are conditioned on this finite trial budget and sample selection. Final ranking must use the completed current-run evidence.


<a id="s-7-3"></a>

### 7.3. Random Forest

**Research trace:** RQ2, RQ3, RQ4. Original five-model pilot; source cells 65–70. Larger search: section 10.3.


#### 7.3.1. Purpose

Investigate Random Forest as a credible alternative under the same profile-grouped validation.


#### 7.3.2. Mathematical basis and assumptions

A regression tree partitions feature space into leaves and predicts each leaf’s mean target. Splits seek reductions in squared error. A forest averages $B$ tree predictions, $\hat z=B^{-1}\sum_{b=1}^B h_b(x)$. Bootstrap sampling and random feature subsets reduce correlation between trees [BREIMAN01]. The supplied model retains `max_features='sqrt'`, seed 42 and two workers; `bootstrap` remains an optimised yes/no choice.


#### 7.3.3. Implementation and parameter choices

n_estimators: integer [20, 100]; max_depth: [5, 30]; min_samples_split: [2, 20]; min_samples_leaf: [1, 10]; bootstrap: yes or no. The expanded temporal dimensions are first span s1 ∈ [5, 100] and positive gaps g2 ∈ [1, 400], g3 ∈ [1, 1500], g4 ∈ [1, 6000], all integer. The actual spans are (s1, s1+g2, s1+g2+g3, s1+g2+g3+g4). These are fixed search bounds, not previously selected spans. Each model has eight GP ask/tell evaluations, five initial points, EI with sampling acquisition optimisation and seed 42. The pilot selects 80 observations per profile. MLP pilot max_iter=200 with early_stopping=False is retained.


#### 7.3.4. Results and verification evidence

The original loop writes all_results["Random Forest"], candidate JSON, three fold scores, means, timings and convergence counts. test_recovered_protocol.py compares the extracted objective arithmetic with the source. No pilot has been executed as part of this delivery certification.


#### 7.3.5. Interpretation and limitations

The pilot incumbent is freshly rescored when it seeds the later larger-sample search. Results are conditioned on this finite trial budget and sample selection. Final ranking must use the completed current-run evidence.


<a id="s-7-4"></a>

### 7.4. Extra Trees

**Research trace:** RQ2, RQ3, RQ4. Original five-model pilot; source cells 65–70. Larger search: section 10.4.


#### 7.4.1. Purpose

Investigate Extra Trees as a credible alternative under the same profile-grouped validation.


#### 7.4.2. Mathematical basis and assumptions

Extra Trees averages $B$ regression trees, $\hat z=B^{-1}\sum_bh_b(x)$. Random candidate thresholds introduce additional split randomness compared with the Random Forest implementation [GEURTS06]. The same original tree-count, depth, leaf-size, split-size and bootstrap dimensions are searched. `max_features='sqrt'`, seed 42 and two workers remain fixed; the algorithm is not substituted with a differently configured forest.


#### 7.4.3. Implementation and parameter choices

n_estimators: integer [20, 100]; max_depth: [5, 30]; min_samples_split: [2, 20]; min_samples_leaf: [1, 10]; bootstrap: yes or no. The expanded temporal dimensions are first span s1 ∈ [5, 100] and positive gaps g2 ∈ [1, 400], g3 ∈ [1, 1500], g4 ∈ [1, 6000], all integer. The actual spans are (s1, s1+g2, s1+g2+g3, s1+g2+g3+g4). These are fixed search bounds, not previously selected spans. Each model has eight GP ask/tell evaluations, five initial points, EI with sampling acquisition optimisation and seed 42. The pilot selects 80 observations per profile. MLP pilot max_iter=200 with early_stopping=False is retained.


#### 7.4.4. Results and verification evidence

The original loop writes all_results["Extra Trees"], candidate JSON, three fold scores, means, timings and convergence counts. test_recovered_protocol.py compares the extracted objective arithmetic with the source. No pilot has been executed as part of this delivery certification.


#### 7.4.5. Interpretation and limitations

The pilot incumbent is freshly rescored when it seeds the later larger-sample search. Results are conditioned on this finite trial budget and sample selection. Final ranking must use the completed current-run evidence.


<a id="s-7-5"></a>

### 7.5. MLP

**Research trace:** RQ2, RQ3, RQ4. Original five-model pilot; source cells 65–70. Larger search: section 10.5.


#### 7.5.1. Purpose

Investigate MLP as a credible alternative under the same profile-grouped validation.


#### 7.5.2. Mathematical basis and assumptions

For hidden layer $\ell$, $h^{(\ell)}=\max(0,W^{(\ell)}h^{(\ell-1)}+b^{(\ell)})$ and the output layer is linear. Adam adjusts weights using gradients of squared prediction loss plus the implementation’s L2 penalty controlled by $\alpha$ [KINGMA15,SKLEARN]. Equal hidden-layer widths are $(n_{units},)\times n_{layers}$. Input and target scaling improves numerical conditioning; it does not establish physical validity.


#### 7.5.3. Implementation and parameter choices

n_layers: integer [1, 3]; n_units: [4, 32]; alpha: log-uniform [1e-7, 0.01]; learning_rate_init: log-uniform [1e-4, 0.01]. The expanded temporal dimensions are first span s1 ∈ [5, 100] and positive gaps g2 ∈ [1, 400], g3 ∈ [1, 1500], g4 ∈ [1, 6000], all integer. The actual spans are (s1, s1+g2, s1+g2+g3, s1+g2+g3+g4). These are fixed search bounds, not previously selected spans. Each model has eight GP ask/tell evaluations, five initial points, EI with sampling acquisition optimisation and seed 42. The pilot selects 80 observations per profile. MLP pilot max_iter=200 with early_stopping=False is retained.


#### 7.5.4. Results and verification evidence

The original loop writes all_results["MLP"], candidate JSON, three fold scores, means, timings and convergence counts. test_recovered_protocol.py compares the extracted objective arithmetic with the source. No pilot has been executed as part of this delivery certification.


#### 7.5.5. Interpretation and limitations

The pilot incumbent is freshly rescored when it seeds the later larger-sample search. Results are conditioned on this finite trial budget and sample selection. Final ranking must use the completed current-run evidence.


In [ ]:
from pmsm_project.search import SPAN_SPACE, MODEL_SPACES, decode_spans
assert len(train_df)==1055198 and len(profile_cv)==55


In [ ]:
from pmsm_project.models import make_regressor
evaluate_model_cv=study.evaluate_model_cv


<a id="s-7-6"></a>

### 7.6. Common five-model Bayesian implementation and pilot comparison

**Research trace:** RQ2, RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 7.6.


#### 7.6.1. Purpose

Execute the original common optimisation loop and compare all five completed pilot searches.


#### 7.6.2. Mathematical basis and assumptions

Each profile belongs to one validation fold. Its other two folds provide the fitting profiles. For fold $f$,
$$MSE_f=\frac1{N_f}\sum_{i\in f}(\hat T_i-T_i)^2,\qquad J=\frac13\sum_{f=1}^{3}MSE_f.$$
The objective $J$ has units °C² and weights folds equally. It is different from pooling every validation row before calculating MSE. Feature and target standardisers are fitted only on the fitting profiles and predictions are inverse-transformed before scoring. Profile-level temperature maxima define three strata for the fixed shuffled `StratifiedKFold(3, random_state=42)` assignment [ROBERTS17]. The expanded temporal dimensions are first span s1 ∈ [5, 100] and positive gaps g2 ∈ [1, 400], g3 ∈ [1, 1500], g4 ∈ [1, 6000], all integer. The actual spans are (s1, s1+g2, s1+g2+g3, s1+g2+g3+g4). These are fixed search bounds, not previously selected spans.


#### 7.6.3. Implementation and parameter choices

make_regressor constructs each unchanged candidate. run_model_pilot performs all eight proposals without loading an archived trial table. The following plotting and summary cells use the newly produced pilot records.


#### 7.6.4. Results and verification evidence

The five tables, convergence counts and accuracy/cost graphics provide the pilot evidence. Production configuration export later requires all 8+12 trial counts for every family.


#### 7.6.5. Interpretation and limitations

Fit-only timing and end-to-end timing must be labelled separately. Hardware, target scaling and solver settings affect cost; failed convergence weakens any apparent accuracy advantage.


In [ ]:
# ============================================================
# STEP 6.3C.3 — Automated Bayesian Optimisation
# All Five Regression Models (No Checkpointing)
# ============================================================

import os
import json
import time
import numpy as np
import pandas as pd

from skopt import Optimizer

# ============================================================
# 1. Configuration
# ============================================================

OUTPUT_DIR = str(BASE_DIR / "bayesian_model_pilots")
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_TRIALS = 8

MODELS = [
    "SVR",
    "k-NN",
    "Random Forest",
    "Extra Trees",
    "MLP"
]


In [ ]:
# Convert NumPy scalar values to standard Python types
def make_json_safe(values):
    """Convert NumPy proposal scalars to standard JSON-compatible scalars.
    
    Parameters
    ----------
    values : sequence
        Proposal scalars in original dimension order; C/epsilon use standardised target units.
    
    Returns
    -------
    list
        Same values and order; no parameter selection.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    return [
        v.item() if isinstance(v, np.generic) else v
        for v in values
    ]


**Atomic experiment block.** The following original timed or grouped-fold loop is kept together to preserve its execution scope and timer boundaries. Its inputs, mathematical objective and generated outputs are explained in this section.


In [ ]:
# ============================================================
# 2. Bayesian Optimisation Function
# ============================================================

def run_model_pilot(model_name, n_trials=8):
    """Run the original eight-evaluation GP/EI pilot for one model family.
    
    Parameters
    ----------
    model_name : str
        Original estimator family in the common search/feasibility loop.
    n_trials : int
        Original stage trial count (8 pilot or 12 larger); production call preserves it.
    
    Returns
    -------
    pandas.DataFrame
        Chronological proposals, hyperparameters, fold MSE (°C²), timings and warnings.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    OSError
        If the original per-trial output tables cannot be written.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """

    search_space = SPAN_SPACE + MODEL_SPACES[model_name]

    optimizer = Optimizer(
        dimensions=search_space,
        base_estimator="GP",
        acq_func="EI",
        acq_optimizer="sampling",
        n_initial_points=5,
        random_state=42
    )

    records = []

    print(f"\nStarting optimisation: {model_name}")

    for trial_number in range(1, n_trials + 1):

        # Bayesian optimisation proposes hyperparameters
        parameters = make_json_safe(
            optimizer.ask()
        )

        start = time.perf_counter()

        # Evaluate the configuration using grouped 3-fold CV
        evaluation = evaluate_model_cv(
            model_name,
            parameters
        )

        duration = time.perf_counter() - start

        # Update Bayesian surrogate model
        optimizer.tell(
            parameters,
            evaluation["mean_cv_mse"]
        )

        record = {
            "trial": trial_number,
            "model": model_name,
            "parameters_json": json.dumps(parameters),
            "hyperparameters_json": json.dumps(
                evaluation["hyperparameters"],
                default=lambda x: x.item()
            ),
            "s1": evaluation["spans"][0],
            "s2": evaluation["spans"][1],
            "s3": evaluation["spans"][2],
            "s4": evaluation["spans"][3],
            "fold1_mse": evaluation["fold_scores"][0],
            "fold2_mse": evaluation["fold_scores"][1],
            "fold3_mse": evaluation["fold_scores"][2],
            "mean_cv_mse": evaluation["mean_cv_mse"],
            "convergence_warnings": evaluation[
                "convergence_warnings"
            ],
            "runtime_s": duration
        }

        records.append(record)

        print(
            f"Trial {trial_number}/{n_trials} | "
            f"CV MSE = {record['mean_cv_mse']:.4f} | "
            f"Runtime = {duration:.1f}s | "
            f"Warnings = {record['convergence_warnings']}",
            flush=True
        )

    # ========================================================
    # 3. Save Results After Model Completion
    # ========================================================

    results = pd.DataFrame(records)

    filename = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "")
    )

    save_path = os.path.join(
        OUTPUT_DIR,
        f"{filename}_trials.csv"
    )

    results.to_csv(save_path, index=False)

    best = results.loc[
        results["mean_cv_mse"].idxmin()
    ]

    print(f"\nPASS: {model_name} completed")
    print("Best CV MSE:", round(best["mean_cv_mse"], 4))
    print("Best spans:", [
        int(best[f"s{i}"]) for i in range(1, 5)
    ])
    print("Results saved:", save_path)

    return results


In [ ]:
# ============================================================
# 4. Automatically Optimise All Five Models
# ============================================================

all_results = {}
summary = []

overall_start = time.perf_counter()


In [ ]:
for i, model_name in enumerate(MODELS, start=1):

    print("\n" + "=" * 60)
    print(f"MODEL {i}/{len(MODELS)}: {model_name}")
    print("=" * 60)

    results = run_model_pilot(
        model_name,
        n_trials=N_TRIALS
    )

    all_results[model_name] = results

    best = results.loc[
        results["mean_cv_mse"].idxmin()
    ]

    summary.append({
        "Model": model_name,
        "Trials": len(results),
        "Best CV MSE": best["mean_cv_mse"],
        "Span 1": int(best["s1"]),
        "Span 2": int(best["s2"]),
        "Span 3": int(best["s3"]),
        "Span 4": int(best["s4"]),
        "Total Runtime (s)": results["runtime_s"].sum(),
        "Convergence Warnings": (
            results["convergence_warnings"].sum()
        )
    })


# ============================================================
# 5. Final Comparison
# ============================================================

pilot_summary = pd.DataFrame(summary)


In [ ]:
pilot_summary = pilot_summary.sort_values(
    "Best CV MSE"
).reset_index(drop=True)

display(pilot_summary.round(4))

# Save consolidated summary
pilot_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "five_model_pilot_summary.csv"
    ),
    index=False
)

total_duration = time.perf_counter() - overall_start

print("\n" + "=" * 60)
print("PASS: ALL FIVE BAYESIAN OPTIMISATIONS COMPLETED")
print("=" * 60)

print("Total Bayesian evaluations:", len(MODELS) * N_TRIALS)
print("Total model fits:", len(MODELS) * N_TRIALS * 3)
print("Total runtime (min):", round(total_duration / 60, 2))
print("Results directory:", OUTPUT_DIR)


In [ ]:
# STEP 6.3C.4 — Summarise model-specific Bayesian pilots

summary = []

for model_name in MODEL_SPACES:

    filename = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "")
    )

    filepath = os.path.join(
        OUTPUT_DIR,
        f"{filename}_trials.csv"
    )

    if not os.path.exists(filepath):
        print(f"Not completed: {model_name}")
        continue

    trials = pd.read_csv(filepath)

    best = trials.loc[
        trials["mean_cv_mse"].idxmin()
    ]

    summary.append({
        "Model": model_name,
        "Trials": len(trials),
        "Best CV MSE": best["mean_cv_mse"],
        "Span 1": int(best["s1"]),
        "Span 2": int(best["s2"]),
        "Span 3": int(best["s3"]),
        "Span 4": int(best["s4"]),
        "Total Runtime (s)": trials["runtime_s"].sum()
    })


In [ ]:
pilot_summary = pd.DataFrame(summary)

if not pilot_summary.empty:
    pilot_summary = pilot_summary.sort_values(
        "Best CV MSE"
    )

    display(pilot_summary.round(4))

    pilot_summary.to_csv(
        os.path.join(
            OUTPUT_DIR, "five_model_pilot_summary.csv"
        ),
        index=False
    )


In [ ]:
# STEP 6.3C.5 — Verify completed Bayesian pilot results

import numpy as np
import pandas as pd
import os


In [ ]:
for model_name in MODEL_SPACES:

    filename = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "")
    )

    path = os.path.join(
        OUTPUT_DIR,
        f"{filename}_trials.csv"
    )

    trials = pd.read_csv(path)

    # Check trial count and numerical validity
    assert len(trials) == 8

    metrics = trials[
        ["fold1_mse", "fold2_mse",
         "fold3_mse", "mean_cv_mse"]
    ]

    assert np.isfinite(metrics.to_numpy()).all()

    # Verify the mean was calculated correctly
    calculated_mean = trials[
        ["fold1_mse", "fold2_mse", "fold3_mse"]
    ].mean(axis=1)

    np.testing.assert_allclose(
        calculated_mean,
        trials["mean_cv_mse"]
    )

    print(f"\nPASS: {model_name}")
    print("Completed trials:", len(trials))
    print("Best CV MSE:", trials["mean_cv_mse"].min())
    print(
        "Convergence warnings:",
        trials["convergence_warnings"].sum()
    )


In [ ]:
print("\nPASS: All five Bayesian pilot results verified")


## 8. SVR computation and nested-sample scaling


<a id="s-8-1"></a>

### 8.1. Initial and fixed-validation RBF-SVR scaling

**Research trace:** RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 8.1.


#### 8.1.1. Purpose

Separate sample-size effects from a simultaneous change in validation rows using the original preliminary scaling experiments.


#### 8.1.2. Mathematical basis and assumptions

The RBF kernel is $k(x,z)=\exp(-\gamma\|x-z\|^2)$ and the epsilon-insensitive loss is $L_\epsilon(r)=\max(0,|r|-\epsilon)$. The primal objective is $\frac12\|w\|^2+C\sum_iL_\epsilon(z_i-f(x_i))$ [SMOLA04]. $C$ and $\epsilon$ apply to the standardised target in the supplied pipelines. `gamma='scale'` is inferred only from fitting inputs; fresh spans, $C$ and $\epsilon$ come from the original GP search. Exact RBF kernels can have quadratic storage and superlinear fitting cost in the observation count; the actual solver need not explicitly store a full dense Gram matrix at every step.


#### 8.1.3. Implementation and parameter choices

The reference spans/C/epsilon come from the current five-model SVR pilot. Retain the original initial sample sizes and controlled 80/160 training comparison with a fixed 160-observation/profile validation set. Nested-sample experiments follow separately.


#### 8.1.4. Results and verification evidence

Initial and fixed-validation CSVs record row counts, fitting/inference time and MSE. The preliminary changing-validation comparison is explicitly distinguished from the controlled design.


#### 8.1.5. Interpretation and limitations

A time difference confounds workload and hardware variability unless the original controls are respected. No held-out profiles enter this scaling investigation.


In [ ]:
# STEP 6.3D.1 — Preliminary SVR scalability experiment

import os
import json
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

MODEL_NAME = "SVR"
SAMPLE_SIZES = [80, 160]

# Load the best previously optimised configuration
trial_path = os.path.join(
    OUTPUT_DIR, "svr_trials.csv"
)

trials = pd.read_csv(trial_path)

best = trials.loc[
    trials["mean_cv_mse"].idxmin()
]

spans = [
    int(best[f"s{i}"])
    for i in range(1, 5)
]

hp = json.loads(best["hyperparameters_json"])

print("Model:", MODEL_NAME)


In [ ]:
print("Fixed optimised spans:", spans)
print("Fixed model hyperparameters:", hp)


# Generate temporal features using full profile histories
# and select different numbers of observations afterward.

sample_blocks = {n: [] for n in SAMPLE_SIZES}

for pid, profile in train_df.groupby(
    "profile_id", sort=False
):

    X_original = make_paper_inputs(profile)

    X_full = make_temporal_features(
        X_original,
        profile["profile_id"],
        spans
    )

    for n in SAMPLE_SIZES:

        positions = np.linspace(
            0, len(profile) - 1, n, dtype=int
        )

        sample = X_full.iloc[positions].copy()

        sample["pm_target"] = (
            profile["pm"].iloc[positions].to_numpy()
        )

        sample["profile_id"] = pid

        sample_blocks[n].append(sample)


In [ ]:
# Use the original Fold 1 validation profiles
val_ids = set(
    profile_cv.loc[
        profile_cv["cv_fold"] == 1,
        "profile_id"
    ]
)

scaling_results = []


In [ ]:
for n in SAMPLE_SIZES:

    sample_df = pd.concat(
        sample_blocks[n],
        ignore_index=True
    )

    validation_mask = sample_df[
        "profile_id"
    ].isin(val_ids)

    training = sample_df.loc[~validation_mask]
    validation = sample_df.loc[validation_mask]

    feature_columns = [
        c for c in sample_df.columns
        if c not in ["pm_target", "profile_id"]
    ]

    X_train = training[feature_columns]
    y_train = training["pm_target"]

    X_val = validation[feature_columns]
    y_val = validation["pm_target"]

    assert len(feature_columns) == 108
    assert len(sample_df) == 55 * n

    assert set(training["profile_id"]).isdisjoint(
        set(validation["profile_id"])
    )

    # Reconstruct the selected SVR configuration
    estimator = make_regressor(MODEL_NAME, hp)

    pipeline = TransformedTargetRegressor(
        regressor=Pipeline([
            ("scaler", StandardScaler()),
            ("model", estimator)
        ]),
        transformer=StandardScaler()
    )

    # Training time
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    # Inference time
    start = time.perf_counter()
    predictions = pipeline.predict(X_val)
    inference_time = time.perf_counter() - start

    mse = mean_squared_error(y_val, predictions)

    # Memory occupied by the numeric input matrices
    matrix_mb = (
        X_train.to_numpy().nbytes
        + X_val.to_numpy().nbytes
    ) / 1e6

    scaling_results.append({
        "Samples/Profile": n,
        "Train Rows": len(X_train),
        "Validation Rows": len(X_val),
        "Training Time (s)": train_time,
        "Inference Time (s)": inference_time,
        "Input Matrix (MB)": matrix_mb,
        "Validation MSE": mse
    })

    print(f"PASS: {n} observations/profile evaluated")


In [ ]:
scaling_df = pd.DataFrame(scaling_results)

display(scaling_df.round(4))

scaling_df.to_csv(
    "svr_scalability_pilot.csv",
    index=False
)


In [ ]:
# STEP 6.3D.2 — Controlled SVR scalability comparison
# Keep validation data fixed while increasing training size

from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import time
import numpy as np
import pandas as pd

# Same validation profiles for both experiments
val_ids = set(
    profile_cv.loc[
        profile_cv["cv_fold"] == 1,
        "profile_id"
    ]
)

# Fixed validation dataset: 160 samples per profile
full_160 = pd.concat(
    sample_blocks[160], ignore_index=True
)

fixed_val = full_160[
    full_160["profile_id"].isin(val_ids)
].copy()

feature_cols = [
    c for c in fixed_val.columns
    if c not in ["pm_target", "profile_id"]
]

X_val = fixed_val[feature_cols]
y_val = fixed_val["pm_target"]

assert len(fixed_val) == 19 * 160


In [ ]:
controlled_results = []


In [ ]:
for n in [80, 160]:

    sample_df = pd.concat(
        sample_blocks[n], ignore_index=True
    )

    # Training data changes, validation data does not
    training = sample_df[
        ~sample_df["profile_id"].isin(val_ids)
    ]

    X_train = training[feature_cols]
    y_train = training["pm_target"]

    assert set(training["profile_id"]).isdisjoint(
        set(fixed_val["profile_id"])
    )

    pipeline = TransformedTargetRegressor(
        regressor=Pipeline([
            ("scaler", StandardScaler()),
            ("model", make_regressor("SVR", hp))
        ]),
        transformer=StandardScaler()
    )

    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    predictions = pipeline.predict(X_val)
    inference_time = time.perf_counter() - start

    mse = mean_squared_error(y_val, predictions)

    controlled_results.append({
        "Training Samples/Profile": n,
        "Training Rows": len(X_train),
        "Fixed Validation Rows": len(X_val),
        "Training Time (s)": train_time,
        "Inference Time (s)": inference_time,
        "Validation MSE": mse
    })


In [ ]:
controlled_df = pd.DataFrame(controlled_results)

display(controlled_df.round(4))

controlled_df.to_csv(
    "svr_scalability_controlled.csv",
    index=False
)

print("PASS: Fixed-validation scalability comparison")


In [ ]:
# STEP 6.3D.3 — Expand SVR scalability assessment

# Generate larger sampled datasets from the full
# original 2 Hz temporal histories.

for n in [320, 640]:
    sample_blocks[n] = []

for pid, profile in train_df.groupby(
    "profile_id", sort=False
):

    X_original = make_paper_inputs(profile)

    X_full = make_temporal_features(
        X_original,
        profile["profile_id"],
        spans
    )

    for n in [320, 640]:

        positions = np.linspace(
            0, len(profile) - 1,
            n, dtype=int
        )

        sample = X_full.iloc[positions].copy()

        sample["pm_target"] = (
            profile["pm"].iloc[positions].to_numpy()
        )

        sample["profile_id"] = pid

        sample_blocks[n].append(sample)


In [ ]:
print("PASS: Larger training samples generated")


In [ ]:
# STEP 6.3D.2 — Second part of doing the Controlled SVR scalability comparison
# but changing from for n in [80, 160]: to for n in [80, 160, 320, 640]:
# Keep validation data fixed while increasing training size

from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import time
import numpy as np
import pandas as pd

# Same validation profiles for both experiments
val_ids = set(
    profile_cv.loc[
        profile_cv["cv_fold"] == 1,
        "profile_id"
    ]
)

# Fixed validation dataset: 160 samples per profile
full_160 = pd.concat(
    sample_blocks[160], ignore_index=True
)

fixed_val = full_160[
    full_160["profile_id"].isin(val_ids)
].copy()

feature_cols = [
    c for c in fixed_val.columns
    if c not in ["pm_target", "profile_id"]
]

X_val = fixed_val[feature_cols]
y_val = fixed_val["pm_target"]


In [ ]:
assert len(fixed_val) == 19 * 160

controlled_results = []


In [ ]:
for n in [80, 160, 320, 640]:

    sample_df = pd.concat(
        sample_blocks[n], ignore_index=True
    )

    # Training data changes, validation data does not
    training = sample_df[
        ~sample_df["profile_id"].isin(val_ids)
    ]

    X_train = training[feature_cols]
    y_train = training["pm_target"]

    assert set(training["profile_id"]).isdisjoint(
        set(fixed_val["profile_id"])
    )

    pipeline = TransformedTargetRegressor(
        regressor=Pipeline([
            ("scaler", StandardScaler()),
            ("model", make_regressor("SVR", hp))
        ]),
        transformer=StandardScaler()
    )

    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    predictions = pipeline.predict(X_val)
    inference_time = time.perf_counter() - start

    mse = mean_squared_error(y_val, predictions)

    controlled_results.append({
        "Training Samples/Profile": n,
        "Training Rows": len(X_train),
        "Fixed Validation Rows": len(X_val),
        "Training Time (s)": train_time,
        "Inference Time (s)": inference_time,
        "Validation MSE": mse
    })


In [ ]:
controlled_df = pd.DataFrame(controlled_results)

display(controlled_df.round(4))

controlled_df.to_csv(
    "svr_scalability_controlled.csv",
    index=False
)

print("PASS: Fixed-validation scalability comparison")


<a id="s-8-2"></a>

### 8.2. Nested sampling and repeated three-fold timings

**Research trace:** RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 8.2.


#### 8.2.1. Purpose

Use truly nested fitting samples to evaluate whether exact RBF-SVR remains computationally practical.


#### 8.2.2. Mathematical basis and assumptions

The original nested_sample_order chooses deterministic ordered positions such that the 80-row set is contained in 160, then 320, then 640. Validation retains 640 observations per profile. Four budgets × three folds × three timing repeats gives 36 records.


#### 8.2.3. Implementation and parameter choices

Retain seed 42, every original row budget and all three repetitions. Histories are filtered before row selection. The reference SVR is selected earlier in this execution; no old spans or SVR winner is restored.


#### 8.2.4. Results and verification evidence

results_df stores 36 timing/error records. The plotted training-time and MSE scaling summaries come from these records; sampling and causality tests check prefix/nesting identities independently.


#### 8.2.5. Interpretation and limitations

Timing repetitions reuse the same data and model setting; they quantify runtime variability, not 36 independent generalisation experiments. Do not equate reduced row count with a lower sensor sampling frequency.


In [ ]:
# STEP 6.3D.4A — Construct nested sample positions

import os
import json
import time
import heapq
import numpy as np
import pandas as pd

SAMPLE_SIZES = [80, 160, 320, 640]
MAX_SAMPLES = max(SAMPLE_SIZES)

# Restore the best SVR pilot configuration
svr_trials = pd.read_csv(
    os.path.join(OUTPUT_DIR, "svr_trials.csv")
)

best_svr = svr_trials.loc[
    svr_trials["mean_cv_mse"].idxmin()
]

svr_spans = [
    int(best_svr[f"s{i}"])
    for i in range(1, 5)
]

svr_hp = json.loads(
    best_svr["hyperparameters_json"]
)




print("Optimised SVR spans:", svr_spans)
print("Model hyperparameters:", svr_hp)


In [ ]:
# Build temporal features from complete recording histories
start_features = time.perf_counter()

nested_blocks = []


In [ ]:
for pid, profile in train_df.groupby(
    "profile_id", sort=False
):

    # Preserve original chronological order
    assert profile.index.is_monotonic_increasing

    order = nested_sample_order(
        len(profile), MAX_SAMPLES
    )

    # Selected positions, ordered chronologically
    positions = np.sort(
        np.asarray(order, dtype=int)
    )

    # Rank tells us when each position was added
    rank_lookup = {
        position: rank
        for rank, position in enumerate(order)
    }

    X_original = make_paper_inputs(profile)

    # Use the FULL 2 Hz history before sampling
    X_full = make_temporal_features(
        X_original,
        profile["profile_id"],
        svr_spans
    )

    sample = X_full.iloc[positions].copy()

    sample["pm_target"] = (
        profile["pm"].iloc[positions].to_numpy()
    )

    sample["profile_id"] = pid

    sample["nest_rank"] = [
        rank_lookup[position]
        for position in positions
    ]

    nested_blocks.append(sample)


In [ ]:
nested_df = pd.concat(
    nested_blocks, ignore_index=True
)

feature_generation_time = (
    time.perf_counter() - start_features
)

# Verify full sample
assert len(nested_df) == 55 * MAX_SAMPLES
assert nested_df["profile_id"].nunique() == 55
assert nested_df["nest_rank"].between(
    0, MAX_SAMPLES - 1
).all()

FEATURE_COLS = [
    c for c in nested_df.columns
    if c not in [
        "pm_target", "profile_id", "nest_rank"
    ]
]

assert len(FEATURE_COLS) == 108
assert np.isfinite(
    nested_df[FEATURE_COLS + ["pm_target"]].to_numpy()
).all()


In [ ]:
# Verify nested membership and observation counts
for n in SAMPLE_SIZES:

    subset = nested_df[
        nested_df["nest_rank"] < n
    ]

    counts = subset.groupby("profile_id").size()

    assert len(subset) == 55 * n
    assert (counts == n).all()

    print(
        f"PASS: {n} observations/profile "
        f"({len(subset):,} observations)"
    )

print("\nPASS: All training samples are nested")
print("Predictor features:", len(FEATURE_COLS))
print(
    "Feature generation time:",
    round(feature_generation_time, 2), "seconds"
)

from pmsm_project.sampling import nested_sample_order


In [ ]:
# STEP 6.3D.4B — Nested 3-fold SVR scalability

from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import time
import pandas as pd
import numpy as np

N_REPEATS = 3

RESULT_PATH = "svr_nested_scalability_v1.csv"

# Fresh execution: no restoration of old checkpoint CSVs.
records = []

fold_lookup = profile_cv.set_index(
    "profile_id"
)["cv_fold"]

nested_df["cv_fold"] = (
    nested_df["profile_id"]
    .map(fold_lookup)
    .astype(int)
)

assert nested_df["cv_fold"].isin([1, 2, 3]).all()


**Atomic experiment block.** The following original timed or grouped-fold loop is kept together to preserve its execution scope and timer boundaries. Its inputs, mathematical objective and generated outputs are explained in this section.


In [ ]:
for n in SAMPLE_SIZES:

    for fold in [1, 2, 3]:

        # Training data changes with n
        training = nested_df.loc[
            (nested_df["cv_fold"] != fold)
            & (nested_df["nest_rank"] < n)
        ]

        # Validation data ALWAYS uses 640
        # observations per validation profile
        validation = nested_df.loc[
            nested_df["cv_fold"] == fold
        ]

        X_train = training[FEATURE_COLS]
        y_train = training["pm_target"]

        X_val = validation[FEATURE_COLS]
        y_val = validation["pm_target"]

        train_ids = set(training["profile_id"])
        val_ids = set(validation["profile_id"])

        assert train_ids.isdisjoint(val_ids)

        assert len(X_train) == len(train_ids) * n
        assert len(X_val) == len(val_ids) * MAX_SAMPLES

        for repeat in range(1, N_REPEATS + 1):

            estimator = make_regressor(
                "SVR", svr_hp
            )

            # Fit scalers exclusively on training fold
            pipeline = TransformedTargetRegressor(
                regressor=Pipeline([
                    ("scaler", StandardScaler()),
                    ("model", estimator)
                ]),
                transformer=StandardScaler()
            )

            # Time model training
            start = time.perf_counter()

            pipeline.fit(X_train, y_train)

            training_time = time.perf_counter() - start

            # Time prediction on fixed validation data
            start = time.perf_counter()

            predictions = pipeline.predict(X_val)

            inference_time = time.perf_counter() - start

            mse = mean_squared_error(
                y_val, predictions
            )

            fitted_svr = (
                pipeline.regressor_
                .named_steps["model"]
            )

            record = {
                "Samples_per_Profile": n,
                "Fold": fold,
                "Repeat": repeat,
                "Training_Rows": len(X_train),
                "Validation_Rows": len(X_val),
                "Training_Time_s": training_time,
                "Inference_Time_s": inference_time,
                "MSE": float(mse),
                "Support_Vectors": len(
                    fitted_svr.support_
                )
            }

            records.append(record)
            print(
                f"n={n}, fold={fold}, "
                f"repeat={repeat}: "
                f"MSE={mse:.4f}, "
                f"train={training_time:.2f}s",
                flush=True
            )


In [ ]:
results_df = pd.DataFrame(records)
results_df.to_csv(RESULT_PATH, index=False)

# Check all 36 requested runs are present
expected_runs = (
    len(SAMPLE_SIZES) * 3 * N_REPEATS
)

assert len(results_df) == expected_runs
assert np.isfinite(
    results_df[
        ["MSE", "Training_Time_s", "Inference_Time_s"]
    ].to_numpy()
).all()

print(
    "\nPASS: All",
    expected_runs,
    "SVR training/evaluation runs completed"
)


In [ ]:
from pmsm_project.plotting import set_figure_context
set_figure_context(section='8.2', source_notebook='fresh_reconstruction.ipynb', source_cell=83)


In [ ]:
# STEP 6.3D.4C — Summarise nested scalability results

import pandas as pd
import matplotlib.pyplot as plt

# Average repeated results within each fold
fold_results = (
    results_df.groupby(
        ["Samples_per_Profile", "Fold"],
        as_index=False
    )
    .agg(
        MSE=("MSE", "mean"),
        Training_Rows=("Training_Rows", "first"),
        Validation_Rows=("Validation_Rows", "first"),
        Training_Time_s=("Training_Time_s", "median"),
        Inference_Time_s=("Inference_Time_s", "median"),
        Support_Vectors=("Support_Vectors", "median")
    )
)

print("Results for each validation fold:")
display(fold_results.round(4))


In [ ]:
# Aggregate across all three folds
scalability_summary = (
    fold_results.groupby(
        "Samples_per_Profile",
        as_index=False
    )
    .agg(
        Mean_CV_MSE=("MSE", "mean"),
        Fold_MSE_SD=("MSE", "std"),
        Total_3Fold_Training_s=("Training_Time_s", "sum"),
        Total_3Fold_Inference_s=("Inference_Time_s", "sum"),
        Mean_Support_Vectors=("Support_Vectors", "mean")
    )
)

print("\nOverall scalability summary:")
display(scalability_summary.round(4))

scalability_summary.to_csv(
    "svr_nested_scalability_summary.csv",
    index=False
)


# Graph 1 — Accuracy versus training size
plt.figure(figsize=(9, 5))

plt.plot(
    scalability_summary["Samples_per_Profile"],
    scalability_summary["Mean_CV_MSE"],
    marker="o"
)

plt.xlabel("Training observations per profile")
plt.ylabel("Mean 3-fold CV MSE (°C²)")
plt.title("SVR Accuracy versus Nested Training-Set Size")


In [ ]:
plt.grid(True)
plt.show()


# Graph 2 — Computational cost
plt.figure(figsize=(9, 5))

plt.plot(
    scalability_summary["Samples_per_Profile"],
    scalability_summary["Total_3Fold_Training_s"],
    marker="o"
)

plt.xlabel("Training observations per profile")
plt.ylabel("Total 3-fold training time (s)")
plt.title("SVR Training Cost versus Dataset Size")
plt.grid(True)
plt.show()

print("PASS: Scalability analysis completed")


In [ ]:
# STEP 6.3D.4D — Inspect fold-level accuracy differences

fold_mse_table = fold_results.pivot(
    index="Samples_per_Profile",
    columns="Fold",
    values="MSE"
)

print("Validation MSE across the three folds:")
display(fold_mse_table.round(4))

print("\nChange in MSE from 320 to 640 samples/profile:")

mse_change = (
    fold_mse_table.loc[640]
    - fold_mse_table.loc[320]
)

display(
    pd.DataFrame({
        "MSE change (°C²)": mse_change
    }).round(4)
)


## 9. Full-development QR OLS and Bayesian optimisation


<a id="s-9-1"></a>

### 9.1. Full-development QR OLS feasibility

**Research trace:** RQ2, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 9.1.


#### 9.1.1. Purpose

Retain the full-data linear solve while avoiding a single huge stacked design.


#### 9.1.2. Mathematical basis and assumptions

Let $A_p=[\mathbf1,X_p,y_p]$ for profile $p$, with $A_p=Q_pR_p$. For $v=(b,\beta^T,-1)^T$, orthogonality gives $\|A_pv\|_2^2=\|R_pv\|_2^2$. Therefore stacking the small $R_p$ factors preserves the least-squares objective of all original rows. Training-fold feature moments are combined using
$$\mu=\frac{\sum_p n_p\mu_p}{N},\qquad v_X=\frac{\sum_p n_p[v_p+(\mu_p-\mu)^2]}{N}.$$
The augmented intercept column permits the equivalent scaling transformation without rebuilding a full design matrix. `scipy.linalg.lstsq(..., lapack_driver='gelsd')` provides a rank-aware SVD solve [SCIPY]. No normal-equation inverse is formed. Numerical rank, residual norms and predictions are more informative than individual coefficients in a nearly dependent design.


#### 9.1.3. Implementation and parameter choices

compress_profile_ols constructs the original per-profile features and augmented factors. The expanded OLS winner is evaluated afresh on all development rows for a three-fold feasibility result.


#### 9.1.4. Results and verification evidence

full_ols_result records fold MSE, numerical rank, feature time, CV time and total time. It becomes the fresh first observation of the later 20-evaluation full-data search.


#### 9.1.5. Interpretation and limitations

QR compression preserves residual sums of squares up to floating-point error, not necessarily bitwise coefficients. Peak process RSS and elapsed time depend on the execution host.


In [ ]:
# STEP 6.3D.5A — Memory-efficient full-data OLS

import time
import numpy as np
import pandas as pd

from scipy.linalg import qr, lstsq

# We retain all 55 original development profiles.
assert len(train_df) == 1055198
assert train_df["profile_id"].nunique() == 55

# Original grouped CV assignments from Step 6.1
cv_lookup = (
    profile_cv.set_index("profile_id")["cv_fold"]
    .astype(int)
    .to_dict()
)

assert set(cv_lookup.values()) == {1, 2, 3}






print("PASS: Full-data OLS evaluation function defined")

from pmsm_project.numerics import compress_profile_ols
evaluate_full_ols_cv=study.evaluate_full_ols_cv


In [ ]:
# STEP 6.3D.5 — Verify QR compression numerically

import numpy as np

# Use a small subset of an existing development profile
pid = int(profile_cv["profile_id"].iloc[0])

demo = train_df.loc[
    train_df["profile_id"] == pid
].head(300).copy()

spans = list(map(int, best_spans))

X_demo = make_temporal_features(
    make_paper_inputs(demo),
    demo["profile_id"],
    spans
).to_numpy(dtype=np.float64)

y_demo = demo["pm"].to_numpy(dtype=np.float64)

compressed = compress_profile_ols(
    demo, spans, chunk_size=128
)

# A deliberately chosen example regression equation
w = np.zeros(110)
w[0] = 25.0
w[1] = 0.01
w[-1] = -1.0

# Calculate the residual sum of squares directly
direct_residuals = (
    w[0] + 0.01 * X_demo[:, 0] - y_demo
)

direct_sse = np.sum(direct_residuals**2)


In [ ]:
# Calculate equivalent residuals from QR factor
compressed_sse = np.sum(
    (compressed["R"] @ w)**2
)

np.testing.assert_allclose(
    direct_sse,
    compressed_sse,
    rtol=1e-8
)

print("PASS: QR residual calculation verified")
print("Direct SSE:", direct_sse)
print("Compressed SSE:", compressed_sse)


In [ ]:
# STEP 6.3D.5B — First full-data OLS feasibility run

import time
import pandas as pd

# Best spans from our expanded 28-trial pilot
# Use the winner of this execution's 8+20 pilot/expanded search.
# The original OLS feasibility method is unchanged; only its seed provenance is fixed.
assert len(all_trials) == 28
candidate_spans = list(map(int, best_spans))

print("Evaluating full-data OLS configuration")
print("Spans:", candidate_spans)
print("Development observations:", len(train_df))

full_ols_result = evaluate_full_ols_cv(
    candidate_spans
)

print("\nPASS: Full-data OLS evaluation completed")

print("\nIndividual fold results:")
display(
    full_ols_result["fold_results"].round(4)
)

print(
    "\nMean 3-fold CV MSE:",
    round(full_ols_result["mean_cv_mse"], 4),
    "°C²"
)

print(
    "Feature generation and compression:",
    round(full_ols_result["feature_time_s"], 2),
    "seconds"
)


In [ ]:
print(
    "Cross-validation fitting and evaluation:",
    round(full_ols_result["cv_time_s"], 2),
    "seconds"
)

print(
    "Total runtime:",
    round(full_ols_result["total_time_s"], 2),
    "seconds"
)

# Save results
full_ols_result["fold_results"].to_csv(
    "ols_full_data_first_trial_folds.csv",
    index=False
)

full_trial_summary = pd.DataFrame([{
    "s1": candidate_spans[0],
    "s2": candidate_spans[1],
    "s3": candidate_spans[2],
    "s4": candidate_spans[3],
    "mean_cv_mse": full_ols_result["mean_cv_mse"],
    "feature_time_s": full_ols_result["feature_time_s"],
    "cv_time_s": full_ols_result["cv_time_s"],
    "total_time_s": full_ols_result["total_time_s"]
}])

full_trial_summary.to_csv(
    "ols_full_data_first_trial_summary.csv",
    index=False
)


<a id="s-9-2"></a>

### 9.2. QR equivalence and numerical rank verification

**Research trace:** RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 9.2.


#### 9.2.1. Purpose

Independently check that compressed and direct OLS represent the same numerical problem.


#### 9.2.2. Mathematical basis and assumptions

Let $A_p=[\mathbf1,X_p,y_p]$ for profile $p$, with $A_p=Q_pR_p$. For $v=(b,\beta^T,-1)^T$, orthogonality gives $\|A_pv\|_2^2=\|R_pv\|_2^2$. Therefore stacking the small $R_p$ factors preserves the least-squares objective of all original rows. Training-fold feature moments are combined using
$$\mu=\frac{\sum_p n_p\mu_p}{N},\qquad v_X=\frac{\sum_p n_p[v_p+(\mu_p-\mu)^2]}{N}.$$
The augmented intercept column permits the equivalent scaling transformation without rebuilding a full design matrix. `scipy.linalg.lstsq(..., lapack_driver='gelsd')` provides a rank-aware SVD solve [SCIPY]. No normal-equation inverse is formed. Numerical rank, residual norms and predictions are more informative than individual coefficients in a nearly dependent design.


#### 9.2.3. Implementation and parameter choices

Execute the original direct/scikit-learn versus QR comparisons, residual and prediction checks. The unit tests use independent synthetic least-squares problems, including rank deficiency.


#### 9.2.4. Results and verification evidence

Numerical differences, tolerances and rank diagnostics are displayed. The production solver remains gelsd; tests do not replace it with a solver selected to match an archived number.


#### 9.2.5. Interpretation and limitations

Near collinearity can amplify coefficient differences even when predictions agree. Retain the diagnostic rank rather than assuming every newly selected span set must have rank 109.


In [ ]:
# STEP 6.3D.6A — Verify QR-based OLS numerically

import numpy as np
import pandas as pd

from scipy.linalg import qr, lstsq
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Use the existing 4,400-observation verification dataset.
# It is based on full profile histories before subsampling.
assert len(smoke_df) == 4400
assert smoke_df["profile_id"].nunique() == 55

feature_cols = [
    c for c in smoke_df.columns
    if c not in ["pm_target", "profile_id"]
]

assert len(feature_cols) == 108

val_ids = set(
    profile_cv.loc[
        profile_cv["cv_fold"] == 1,
        "profile_id"
    ]
)

training = smoke_df.loc[
    ~smoke_df["profile_id"].isin(val_ids)
]

validation = smoke_df.loc[
    smoke_df["profile_id"].isin(val_ids)
]


In [ ]:
X_train = training[feature_cols].to_numpy()
y_train = training["pm_target"].to_numpy()

X_val = validation[feature_cols].to_numpy()
y_val = validation["pm_target"].to_numpy()

assert set(training["profile_id"]).isdisjoint(
    set(validation["profile_id"])
)

# A — Reference implementation using Scikit-learn
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

reference_model = LinearRegression()
reference_model.fit(X_train_scaled, y_train)

reference_predictions = reference_model.predict(
    X_val_scaled
)

reference_mse = mean_squared_error(
    y_val, reference_predictions
)


# B — Compress training profiles independently with QR
R_blocks = []
profile_statistics = []


In [ ]:
for pid, group in training.groupby(
    "profile_id", sort=False
):

    Xp = group[feature_cols].to_numpy()
    yp = group["pm_target"].to_numpy()

    augmented = np.column_stack([
        np.ones(len(Xp)),
        Xp,
        yp
    ])

    R = qr(
        augmented,
        mode="r",
        check_finite=False
    )[0]

    R_blocks.append(R)

    profile_statistics.append({
        "n": len(Xp),
        "mean": Xp.mean(axis=0),
        "var": Xp.var(axis=0)
    })


# C — Reconstruct training-fold standardisation
n_total = sum(
    item["n"] for item in profile_statistics
)

mu = sum(
    item["n"] * item["mean"]
    for item in profile_statistics
) / n_total


In [ ]:
variance = sum(
    item["n"] * (
        item["var"] + (item["mean"] - mu)**2
    )
    for item in profile_statistics
) / n_total

scale = np.sqrt(np.maximum(variance, 0))
scale[scale == 0] = 1.0

np.testing.assert_allclose(
    mu, scaler.mean_, rtol=1e-9, atol=1e-9
)

np.testing.assert_allclose(
    scale, scaler.scale_, rtol=1e-9, atol=1e-9
)


# D — Fit OLS using the compressed matrices
R_train = np.vstack(R_blocks)

R_scaled = R_train.copy()

R_scaled[:, 1:-1] = (
    R_train[:, 1:-1]
    - R_train[:, [0]] * mu
) / scale

coefficients, _, rank, _ = lstsq(
    R_scaled[:, :-1],
    R_scaled[:, -1],
    lapack_driver="gelsd"
)

beta = coefficients[1:] / scale


In [ ]:
intercept = (
    coefficients[0] - np.dot(mu, beta)
)

qr_predictions = X_val @ beta + intercept

qr_mse = mean_squared_error(
    y_val, qr_predictions
)


# E — Compare predictions and validation errors
max_difference = np.max(
    np.abs(reference_predictions - qr_predictions)
)

np.testing.assert_allclose(
    qr_predictions,
    reference_predictions,
    rtol=1e-8,
    atol=1e-6
)

np.testing.assert_allclose(
    qr_mse,
    reference_mse,
    rtol=1e-8
)

print("PASS: QR and Scikit-learn agree")
print("Numerical rank:", rank)
print("Maximum prediction difference:", max_difference)
print("Scikit-learn MSE:", reference_mse)
print("QR-based OLS MSE:", qr_mse)


<a id="s-9-3"></a>

### 9.3. Twenty-evaluation full-data Bayesian OLS search

**Research trace:** RQ2, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 9.3.


#### 9.3.1. Purpose

Select OLS spans using the original full-development grouped QR objective.


#### 9.3.2. Mathematical basis and assumptions

The expanded temporal dimensions are first span s1 ∈ [5, 100] and positive gaps g2 ∈ [1, 400], g3 ∈ [1, 1500], g4 ∈ [1, 6000], all integer. The actual spans are (s1, s1+g2, s1+g2+g3, s1+g2+g3+g4). These are fixed search bounds, not previously selected spans.

Each profile belongs to one validation fold. Its other two folds provide the fitting profiles. For fold $f$,
$$MSE_f=\frac1{N_f}\sum_{i\in f}(\hat T_i-T_i)^2,\qquad J=\frac13\sum_{f=1}^{3}MSE_f.$$
The objective $J$ has units °C² and weights folds equally. It is different from pooling every validation row before calculating MSE. Feature and target standardisers are fitted only on the fitting profiles and predictions are inverse-transformed before scoring. Profile-level temperature maxima define three strata for the fixed shuffled `StratifiedKFold(3, random_state=42)` assignment [ROBERTS17].


#### 9.3.3. Implementation and parameter choices

Use GP Optimizer with EI, sampling acquisition optimisation, six initial points and seed 42. The freshly evaluated feasibility incumbent is trial 1; proposals 2–20 are evaluated from scratch. No historical seed score is reused.


#### 9.3.4. Results and verification evidence

full_trials has 20 rows with rank, fold errors and timing. The minimum determines the final OLS spans exported to Notebook 01.


#### 9.3.5. Interpretation and limitations

The best sampled full-data CV result remains an exploratory selection. It need not reproduce an earlier span winner or held-out RMSE, and differences must not be fixed by changing the algorithm.


In [ ]:
# STEP 6.3D.6B — Fresh full-data Bayesian optimisation configuration
# No Google Drive mount, checkpoint restore or prior CSV required.
from skopt import Optimizer
from skopt.space import Integer
import numpy as np
import pandas as pd
import json
import os

OLS_RESULTS_DIR = str(BASE_DIR / "ols_full_bayesian")
os.makedirs(OLS_RESULTS_DIR, exist_ok=True)
RESULTS_PATH = os.path.join(OLS_RESULTS_DIR, "ols_full_bayesian_trials.csv")
TARGET_TRIALS = 20  # includes the earlier feasibility evaluation

search_space = [
    Integer(5, 100, name="s1"),
    Integer(1, 400, name="gap_2"),
    Integer(1, 1500, name="gap_3"),
    Integer(1, 6000, name="gap_4")
]

SEED_SPANS = list(map(int, candidate_spans))
assert decode_spans(encode_spans(SEED_SPANS)) == SEED_SPANS

print("PASS: Fresh full-data Bayesian search configured")
print("Full-data OLS evaluations (including seed):", TARGET_TRIALS)


In [ ]:
# STEP 6.3D.6C — Full-data OLS Bayesian optimisation, from scratch
# Relies only on the current Run All execution's full_ols_result.
import json
import time
import numpy as np
import pandas as pd
from skopt import Optimizer

assert full_ols_result["spans"] == SEED_SPANS
assert len(full_ols_result["fold_results"]) == 3

ols_optimizer = Optimizer(
    dimensions=search_space,
    base_estimator="GP",
    n_initial_points=6,
    acq_func="EI",
    acq_optimizer="sampling",
    random_state=42
)
seed_parameters = encode_spans(SEED_SPANS)
ols_optimizer.tell(seed_parameters, float(full_ols_result["mean_cv_mse"]))

seed_folds = full_ols_result["fold_results"].sort_values("Fold")


In [ ]:
records = [{
    "trial": 1,
    "source": "full-data feasibility run",
    "s1": SEED_SPANS[0],
    "s2": SEED_SPANS[1],
    "s3": SEED_SPANS[2],
    "s4": SEED_SPANS[3],
    "parameters_json": json.dumps(seed_parameters),
    "fold1_mse": float(seed_folds.iloc[0]["MSE"]),
    "fold2_mse": float(seed_folds.iloc[1]["MSE"]),
    "fold3_mse": float(seed_folds.iloc[2]["MSE"]),
    "mean_cv_mse": float(full_ols_result["mean_cv_mse"]),
    "min_numerical_rank": int(seed_folds["Numerical_Rank"].min()),
    "feature_time_s": float(full_ols_result["feature_time_s"]),
    "cv_time_s": float(full_ols_result["cv_time_s"]),
    "total_time_s": float(full_ols_result["total_time_s"])
}]


In [ ]:
# Evaluations 2...20 have brand-new proposals and scores.
for trial_number in range(2, TARGET_TRIALS + 1):
    parameters = [int(v) for v in ols_optimizer.ask()]
    spans = decode_spans(parameters)

    print(f"OLS full-data trial {trial_number}/{TARGET_TRIALS}: {spans}", flush=True)
    evaluation = evaluate_full_ols_cv(spans)
    score = float(evaluation["mean_cv_mse"])
    assert np.isfinite(score)
    ols_optimizer.tell(parameters, score)
    f = evaluation["fold_results"].sort_values("Fold")
    record = {
        "trial": trial_number,
        "source": "Bayesian proposal",
        "s1": spans[0], "s2": spans[1], "s3": spans[2], "s4": spans[3],
        "parameters_json": json.dumps(parameters),
        "fold1_mse": float(f.iloc[0]["MSE"]),
        "fold2_mse": float(f.iloc[1]["MSE"]),
        "fold3_mse": float(f.iloc[2]["MSE"]),
        "mean_cv_mse": score,
        "min_numerical_rank": int(f["Numerical_Rank"].min()),
        "feature_time_s": evaluation["feature_time_s"],
        "cv_time_s": evaluation["cv_time_s"],
        "total_time_s": evaluation["total_time_s"]
    }
    records.append(record)
    print(f"CV MSE {score:.4f}; best {min(r['mean_cv_mse'] for r in records):.4f}", flush=True)

full_trials = pd.DataFrame(records)
assert len(full_trials) == TARGET_TRIALS
full_trials.to_csv(RESULTS_PATH, index=False)
best_trial = full_trials.loc[full_trials["mean_cv_mse"].idxmin()]
best_spans_full = [int(best_trial[f"s{i}"]) for i in range(1, 5)]
print("PASS: OLS full-data Bayesian optimisation complete")
print("Best spans:", best_spans_full)
print("Best mean CV MSE:", best_trial["mean_cv_mse"])
display(full_trials.sort_values("mean_cv_mse").head(10).round(4))


In [ ]:
from pmsm_project.plotting import set_figure_context
set_figure_context(section='9.3', source_notebook='fresh_reconstruction.ipynb', source_cell=98)


In [ ]:
# STEP 6.3D.6D — Analyse full-data Bayesian results

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load directly from the persistent CSV
history = pd.read_csv(RESULTS_PATH)

history = history.sort_values("trial").copy()

history["best_so_far"] = (
    history["mean_cv_mse"].cummin()
)

# Plot Bayesian optimisation history
plt.figure(figsize=(11, 5))

plt.plot(
    history["trial"],
    history["mean_cv_mse"],
    "o-",
    label="Candidate 3-fold CV MSE"
)

plt.plot(
    history["trial"],
    history["best_so_far"],
    "s-",
    label="Best CV MSE so far"
)

plt.axhline(
    history.iloc[0]["mean_cv_mse"],
    linestyle="--",
    label="Initial full-data configuration"
)


In [ ]:
plt.xlabel("Full-data Bayesian optimisation trial")
plt.ylabel("Average 3-fold CV MSE (°C²)")
plt.title("Full-Data OLS Bayesian Optimisation")
plt.legend()
plt.grid(True)
plt.show()


# Compare initial and final configurations
best = history.loc[
    history["mean_cv_mse"].idxmin()
]

initial_mse = float(
    history.iloc[0]["mean_cv_mse"]
)

best_mse = float(best["mean_cv_mse"])

improvement = (
    (initial_mse - best_mse) / initial_mse
) * 100


In [ ]:
comparison = pd.DataFrame([
    {
        "Configuration": "Initial",
        "s1": int(history.iloc[0]["s1"]),
        "s2": int(history.iloc[0]["s2"]),
        "s3": int(history.iloc[0]["s3"]),
        "s4": int(history.iloc[0]["s4"]),
        "CV MSE": initial_mse
    },
    {
        "Configuration": "Best full-data",
        "s1": int(best["s1"]),
        "s2": int(best["s2"]),
        "s3": int(best["s3"]),
        "s4": int(best["s4"]),
        "CV MSE": best_mse
    }
])

display(comparison.round(4))

print(f"Relative MSE improvement: {improvement:.2f}%")

print(
    "Mean runtime per evaluation:",
    round(history["total_time_s"].mean(), 2),
    "seconds"
)

print(
    "Total recorded evaluation runtime:",
    round(history["total_time_s"].sum() / 60, 2),
    "minutes"
)


In [ ]:
print(
    "Minimum numerical rank:",
    history["min_numerical_rank"].min()
)


## 10. Larger-sample five-model Bayesian searches

Each model’s current pilot incumbent is freshly rescored at the larger historical budget. The common loop keeps all five original searches and their order.


<a id="s-10-1"></a>

### 10.1. SVR

**Research trace:** RQ2, RQ3, RQ4. Common execution: following unchanged evaluate_larger_model_cv and optimiser cells.


#### 10.1.1. Purpose

Assess SVR with the original larger sampling budget and select its fresh configuration.


#### 10.1.2. Mathematical basis and assumptions

The RBF kernel is $k(x,z)=\exp(-\gamma\|x-z\|^2)$ and the epsilon-insensitive loss is $L_\epsilon(r)=\max(0,|r|-\epsilon)$. The primal objective is $\frac12\|w\|^2+C\sum_iL_\epsilon(z_i-f(x_i))$ [SMOLA04]. $C$ and $\epsilon$ apply to the standardised target in the supplied pipelines. `gamma='scale'` is inferred only from fitting inputs; fresh spans, $C$ and $\epsilon$ come from the original GP search.


#### 10.1.3. Implementation and parameter choices

C: log-uniform [0.001, 10]; epsilon: log-uniform [0.01, 0.3]. Training selects 320 observations per fitting profile; validation selects 640 per validation profile. Twelve evaluations include one freshly rescored pilot seed. GP/EI, sampling acquisition optimisation, five initial points and seed 42 are preserved.


#### 10.1.4. Results and verification evidence

scaled_results["SVR"] records all 12 current-run proposals, parameter JSON, fold MSE, means, timing and warnings. The manifest independently verifies trial counts and reselects the same minimum.


#### 10.1.5. Interpretation and limitations

The different model-family training budgets are part of the supplied feasibility design. Interpret rankings with that restriction rather than claiming equal-budget universal superiority.


<a id="s-10-2"></a>

### 10.2. k-NN

**Research trace:** RQ2, RQ3, RQ4. Common execution: following unchanged evaluate_larger_model_cv and optimiser cells.


#### 10.2.1. Purpose

Assess k-NN with the original larger sampling budget and select its fresh configuration.


#### 10.2.2. Mathematical basis and assumptions

For the $k$ nearest standardised training states $\mathcal N_k(x)$, $\hat z(x)=\sum_{i\in\mathcal N_k}w_i z_i/\sum_iw_i$. Uniform weights have $w_i=1$; distance weights follow scikit-learn’s inverse-distance convention, including its zero-distance handling. $z=(T-\mu_T)/\sigma_T$ uses development/fold fitting statistics. Standardisation is essential because volts, amperes and rpm otherwise contribute incompatible Euclidean scales [SKLEARN].


#### 10.2.3. Implementation and parameter choices

n_neighbors: integer [1, 512]; weights: uniform or distance. Training selects 640 observations per fitting profile; validation selects 640 per validation profile. Twelve evaluations include one freshly rescored pilot seed. GP/EI, sampling acquisition optimisation, five initial points and seed 42 are preserved.


#### 10.2.4. Results and verification evidence

scaled_results["k-NN"] records all 12 current-run proposals, parameter JSON, fold MSE, means, timing and warnings. The manifest independently verifies trial counts and reselects the same minimum.


#### 10.2.5. Interpretation and limitations

The different model-family training budgets are part of the supplied feasibility design. Interpret rankings with that restriction rather than claiming equal-budget universal superiority.


<a id="s-10-3"></a>

### 10.3. Random Forest

**Research trace:** RQ2, RQ3, RQ4. Common execution: following unchanged evaluate_larger_model_cv and optimiser cells.


#### 10.3.1. Purpose

Assess Random Forest with the original larger sampling budget and select its fresh configuration.


#### 10.3.2. Mathematical basis and assumptions

A regression tree partitions feature space into leaves and predicts each leaf’s mean target. Splits seek reductions in squared error. A forest averages $B$ tree predictions, $\hat z=B^{-1}\sum_{b=1}^B h_b(x)$. Bootstrap sampling and random feature subsets reduce correlation between trees [BREIMAN01]. The supplied model retains `max_features='sqrt'`, seed 42 and two workers; `bootstrap` remains an optimised yes/no choice.


#### 10.3.3. Implementation and parameter choices

n_estimators: integer [20, 100]; max_depth: [5, 30]; min_samples_split: [2, 20]; min_samples_leaf: [1, 10]; bootstrap: yes or no. Training selects 640 observations per fitting profile; validation selects 640 per validation profile. Twelve evaluations include one freshly rescored pilot seed. GP/EI, sampling acquisition optimisation, five initial points and seed 42 are preserved.


#### 10.3.4. Results and verification evidence

scaled_results["Random Forest"] records all 12 current-run proposals, parameter JSON, fold MSE, means, timing and warnings. The manifest independently verifies trial counts and reselects the same minimum.


#### 10.3.5. Interpretation and limitations

The different model-family training budgets are part of the supplied feasibility design. Interpret rankings with that restriction rather than claiming equal-budget universal superiority.


<a id="s-10-4"></a>

### 10.4. Extra Trees

**Research trace:** RQ2, RQ3, RQ4. Common execution: following unchanged evaluate_larger_model_cv and optimiser cells.


#### 10.4.1. Purpose

Assess Extra Trees with the original larger sampling budget and select its fresh configuration.


#### 10.4.2. Mathematical basis and assumptions

Extra Trees averages $B$ regression trees, $\hat z=B^{-1}\sum_bh_b(x)$. Random candidate thresholds introduce additional split randomness compared with the Random Forest implementation [GEURTS06]. The same original tree-count, depth, leaf-size, split-size and bootstrap dimensions are searched. `max_features='sqrt'`, seed 42 and two workers remain fixed; the algorithm is not substituted with a differently configured forest.


#### 10.4.3. Implementation and parameter choices

n_estimators: integer [20, 100]; max_depth: [5, 30]; min_samples_split: [2, 20]; min_samples_leaf: [1, 10]; bootstrap: yes or no. Training selects 640 observations per fitting profile; validation selects 640 per validation profile. Twelve evaluations include one freshly rescored pilot seed. GP/EI, sampling acquisition optimisation, five initial points and seed 42 are preserved.


#### 10.4.4. Results and verification evidence

scaled_results["Extra Trees"] records all 12 current-run proposals, parameter JSON, fold MSE, means, timing and warnings. The manifest independently verifies trial counts and reselects the same minimum.


#### 10.4.5. Interpretation and limitations

The different model-family training budgets are part of the supplied feasibility design. Interpret rankings with that restriction rather than claiming equal-budget universal superiority.


<a id="s-10-5"></a>

### 10.5. MLP

**Research trace:** RQ2, RQ3, RQ4. Common execution: following unchanged evaluate_larger_model_cv and optimiser cells.


#### 10.5.1. Purpose

Assess MLP with the original larger sampling budget and select its fresh configuration.


#### 10.5.2. Mathematical basis and assumptions

For hidden layer $\ell$, $h^{(\ell)}=\max(0,W^{(\ell)}h^{(\ell-1)}+b^{(\ell)})$ and the output layer is linear. Adam adjusts weights using gradients of squared prediction loss plus the implementation’s L2 penalty controlled by $\alpha$ [KINGMA15,SKLEARN]. Equal hidden-layer widths are $(n_{units},)\times n_{layers}$. Input and target scaling improves numerical conditioning; it does not establish physical validity.


#### 10.5.3. Implementation and parameter choices

n_layers: integer [1, 3]; n_units: [4, 32]; alpha: log-uniform [1e-7, 0.01]; learning_rate_init: log-uniform [1e-4, 0.01]. Training selects 640 observations per fitting profile; validation selects 640 per validation profile. Twelve evaluations include one freshly rescored pilot seed. GP/EI, sampling acquisition optimisation, five initial points and seed 42 are preserved. The larger MLP retains max_iter=400, early_stopping=True, validation_fraction=0.1 and patience 15. This differs deliberately from the historical pilot.


#### 10.5.4. Results and verification evidence

scaled_results["MLP"] records all 12 current-run proposals, parameter JSON, fold MSE, means, timing and warnings. The manifest independently verifies trial counts and reselects the same minimum.


#### 10.5.5. Interpretation and limitations

The internal early-stopping split is row-based, although external validation is profile-disjoint. This can provide an optimistic internal stopping signal; it is disclosed and preserved.


In [ ]:
# STEP 6.3D.7A — Larger-scale model evaluation

import os
import json
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

# Model-specific sample sizes
TRAIN_SAMPLES = {
    "SVR": 320,
    "k-NN": 640,
    "Random Forest": 640,
    "Extra Trees": 640,
    "MLP": 640
}

VALIDATION_SAMPLES = 640

# Existing development-profile fold assignments
fold_lookup = (
    profile_cv.set_index("profile_id")["cv_fold"]
    .astype(int)
    .to_dict()
)

assert len(train_df) == 1055198
assert len(fold_lookup) == 55


In [ ]:
for function in [
    make_paper_inputs,
    make_temporal_features,
    make_regressor,
    nested_sample_order
]:
    assert callable(function)




print("PASS: Larger-scale CV objective defined")

evaluate_larger_model_cv=study.evaluate_larger_model_cv


<a id="s-10-6"></a>

### 10.6. Common larger-search execution and consolidation

**Research trace:** RQ2, RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 10.6.


#### 10.6.1. Purpose

Execute and consolidate the original 12-evaluation larger Bayesian campaign for each family.


#### 10.6.2. Mathematical basis and assumptions

Each profile belongs to one validation fold. Its other two folds provide the fitting profiles. For fold $f$,
$$MSE_f=\frac1{N_f}\sum_{i\in f}(\hat T_i-T_i)^2,\qquad J=\frac13\sum_{f=1}^{3}MSE_f.$$
The objective $J$ has units °C² and weights folds equally. It is different from pooling every validation row before calculating MSE. Feature and target standardisers are fitted only on the fitting profiles and predictions are inverse-transformed before scoring. Profile-level temperature maxima define three strata for the fixed shuffled `StratifiedKFold(3, random_state=42)` assignment [ROBERTS17].


#### 10.6.3. Implementation and parameter choices

Retain every source ask/tell step and rescore the freshly selected pilot seed. The summary is assembled only from completed current-run scaled_results tables.


#### 10.6.4. Results and verification evidence

The five larger-search tables determine k-NN, RF, ET and MLP final settings and the Stage 9 RBF reference. The full-data OLS selection remains separate.


#### 10.6.5. Interpretation and limitations

Repeatedly inspecting development CV is an exploratory workflow. The fixed held-out partition must not be used to refine these proposals or select between their outcomes.


In [ ]:
# STEP 6.3D.7B — Automatic large-sample Bayesian optimisation (five models)
# Fresh Run All: start from the freshly produced 6.3C pilot CSVs.
# No checkpoint files, no Google Drive login, no per-model manual action.
import os
import json
import time
import numpy as np
import pandas as pd
from skopt import Optimizer

SEARCH_DIR = str(BASE_DIR / "scaled_model_bayesian")
os.makedirs(SEARCH_DIR, exist_ok=True)
MODEL_ORDER = ["SVR", "k-NN", "Random Forest", "Extra Trees", "MLP"]
SCALED_TRIALS = 12  # each includes a newly scored pilot seed


In [ ]:
def json_safe(values):
    """Convert NumPy proposal scalars to standard JSON-compatible scalars.
    
    Parameters
    ----------
    values : sequence
        Proposal scalars in original dimension order; C/epsilon use standardised target units.
    
    Returns
    -------
    list
        Same values and order; no parameter selection.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    return [v.item() if isinstance(v, np.generic) else v for v in values]


In [ ]:
def run_scaled_bayesian(model_name, n_trials=SCALED_TRIALS):
    """Run the original larger-budget twelve-evaluation GP/EI search.
    
    Parameters
    ----------
    model_name : str
        Original estimator family in the common search/feasibility loop.
    n_trials : int
        Original stage trial count (8 pilot or 12 larger); production call preserves it.
    
    Returns
    -------
    pandas.DataFrame
        Fresh candidate and score rows, including the freshly rescored pilot seed.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    OSError
        If the original per-trial output tables cannot be written.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    assert model_name in MODEL_ORDER
    basename = model_name.lower().replace(" ", "_").replace("-", "")
    old_path = os.path.join(OUTPUT_DIR, f"{basename}_trials.csv")
    previous = pd.read_csv(old_path)
    assert len(previous) == 8, f"Missing pilot results for {model_name}"
    pilot_best = previous.loc[previous["mean_cv_mse"].idxmin()]
    seed = json_safe(json.loads(pilot_best["parameters_json"]))

    model_optimizer = Optimizer(
        dimensions=SPAN_SPACE + MODEL_SPACES[model_name],
        base_estimator="GP", acq_func="EI", acq_optimizer="sampling",
        n_initial_points=5, random_state=42
    )
    trials = []
    for number in range(1, n_trials + 1):
        parameters = seed if number == 1 else json_safe(model_optimizer.ask())
        start = time.perf_counter()
        evaluation = evaluate_larger_model_cv(model_name, parameters)
        score = float(evaluation["mean_cv_mse"])
        assert np.isfinite(score)
        model_optimizer.tell(parameters, score)

        record = {
            "trial": number,
            "source": "Reevaluated pilot seed" if number == 1 else "Bayesian proposal",
            "parameters_json": json.dumps(parameters),
            "hyperparameters_json": json.dumps(evaluation["hyperparameters"],
                                                default=lambda x: x.item()),
            "s1": evaluation["spans"][0], "s2": evaluation["spans"][1],
            "s3": evaluation["spans"][2], "s4": evaluation["spans"][3],
            "fold1_mse": evaluation["fold_scores"][0],
            "fold2_mse": evaluation["fold_scores"][1],
            "fold3_mse": evaluation["fold_scores"][2],
            "mean_cv_mse": score,
            "feature_time_s": evaluation["feature_time_s"],
            "training_time_s": evaluation["training_time_s"],
            "inference_time_s": evaluation["inference_time_s"],
            "total_time_s": evaluation["total_time_s"],
            "convergence_warnings": evaluation["convergence_warnings"]
        }
        trials.append(record)
        print(f"{model_name} — {number}/{n_trials}, CV MSE={score:.4f}, "
              f"runtime={record['total_time_s']:.1f}s", flush=True)

    result = pd.DataFrame(trials)
    path = os.path.join(SEARCH_DIR, f"{basename}_trials.csv")
    result.to_csv(path, index=False)
    return result


In [ ]:
# Completely unattended sequential execution
scaled_results = {}
for number, model_name in enumerate(MODEL_ORDER, 1):
    print(f"\n{'='*50}\nMODEL {number}/5 — {model_name}\n{'='*50}", flush=True)
    scaled_results[model_name] = run_scaled_bayesian(model_name, SCALED_TRIALS)
    print(f"PASS: {model_name} large-sample optimisation complete", flush=True)

print("PASS: All five larger-scale searches completed")


In [ ]:
# STEP 6.3D.7C — Automatically consolidate completed model results
rows = []
for model_name in MODEL_ORDER:
    trials = scaled_results[model_name]
    assert len(trials) == SCALED_TRIALS
    folds = trials[["fold1_mse", "fold2_mse", "fold3_mse"]]
    np.testing.assert_allclose(folds.mean(axis=1), trials["mean_cv_mse"])
    b = trials.loc[trials["mean_cv_mse"].idxmin()]
    rows.append({
        "Model": model_name,
        "Training samples/profile": TRAIN_SAMPLES[model_name],
        "Evaluations": len(trials),
        "Best CV MSE": b["mean_cv_mse"],
        "Span 1": int(b["s1"]), "Span 2": int(b["s2"]),
        "Span 3": int(b["s3"]), "Span 4": int(b["s4"]),
        "Total runtime (min)": trials["total_time_s"].sum() / 60,
        "Convergence warnings": int(trials["convergence_warnings"].sum())
    })
scaled_summary = pd.DataFrame(rows).sort_values("Best CV MSE")
scaled_summary.to_csv(os.path.join(SEARCH_DIR, "larger_scale_model_summary.csv"), index=False)
print("PASS: all five model searches and consistency checks completed")
display(scaled_summary.round(4))
print("Notes: OLS used all development observations; the other five models used")
print("nested subsamples. These CV results are NOT the final held-out 2 Hz test benchmark.")


## 11. Full-development training feasibility


<a id="s-11-1"></a>

### 11.1. Historical training-feasibility measurements

**Research trace:** RQ2, RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 11.1.


#### 11.1.1. Purpose

Retain the original final-model feasibility investigation and its scaling observations.


#### 11.1.2. Mathematical basis and assumptions

Per-model fit time, observation count and process RSS describe computational workload. Process RSS is total resident process memory, not isolated incremental model storage. Linear and quadratic extrapolations are estimates, not measured full-run timings.


#### 11.1.3. Implementation and parameter choices

Execute the supplied feasibility loop with its exact budgets and fitted settings derived from the current larger searches. The original later summary cell performs no extra training.


#### 11.1.4. Results and verification evidence

Feasibility tables and projected costs are generated. The selected exact RBF-SVR is a reference for Stage 9, while Notebook 01 ultimately fits the selected direct LinearSVR alternative.


#### 11.1.5. Interpretation and limitations

Colab memory and CPU allocation vary. RSS values below known array sizes cannot support a memory-use claim. Time projections do not substitute for the student’s measured final training/inference.


In [ ]:
# STEP 6.3D.8A — Automatically profile final-model training feasibility
# Intended to run after Step 6.3D.7C in the Run-All notebook.
# No checkpoints are loaded and the final 14 test profiles are untouched.

import os
import json
import gc
import time
import threading
import warnings
import psutil
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

FEASIBILITY_DIR = str(BASE_DIR / 'training_feasibility')
os.makedirs(FEASIBILITY_DIR, exist_ok=True)

# SVR is initially limited because of kernel complexity.
# The other estimators are assessed using progressively larger samples.
FEASIBILITY_SIZES = {
    'SVR': [320, 640, 1280],
    'k-NN': [640, 1280, 2048],
    'Random Forest': [640, 1280, 2048],
    'Extra Trees': [640, 1280, 2048],
    'MLP': [640, 1280, 2048]
}
VAL_FOLD = 1                     # One existing development fold
VAL_SAMPLES_PER_PROFILE = 64     # Identical validation observations at each size
NEXT_SIZE_FIT_LIMIT_S = 90.0     # Avoid larger fits after a slow result

assert len(train_df) == 1055198
assert train_df['profile_id'].nunique() == 55
assert set(train_df.profile_id).isdisjoint(set(test_df.profile_id))


In [ ]:
assert set(FEASIBILITY_SIZES) == set(MODEL_ORDER)
assert min(train_df.groupby('profile_id').size()) >= 2048

FEASIBILITY_FOLDS = profile_cv.set_index('profile_id')['cv_fold'].astype(int).to_dict()
assert set(FEASIBILITY_FOLDS.values()) == {1, 2, 3}

# Read best parameters directly from this notebook's completed 6.3D.7 results.
# This intentionally does not restore any checkpoint or depend on a past session.
def best_scaled_configuration(name):
    """Read the lowest-MSE configuration from the current larger search.
    
    Parameters
    ----------
    name : str
        Estimator family whose current-run larger search has completed.
    
    Returns
    -------
    tuple[list[int], dict]
        Current-run ordered temporal spans and candidate hyperparameters.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    source = scaled_results[name]
    winner = source.loc[source['mean_cv_mse'].idxmin()]
    spans = [int(winner[f's{i}']) for i in range(1, 5)]
    hp = json.loads(winner['hyperparameters_json'])
    return spans, hp


In [ ]:
class PeakProcessRAM:
    """Approximate process peak RSS (MiB) during one fit/predict cycle."""
    def __init__(self):
        self.process = psutil.Process()
        self.peak_mib = 0.0
        self.stop_event = threading.Event()

    def _sample(self):
        while not self.stop_event.wait(0.05):
            self.peak_mib = max(
                self.peak_mib,
                self.process.memory_info().rss / (1024 ** 2)
            )

    def __enter__(self):
        self.initial_mib = self.process.memory_info().rss / (1024 ** 2)
        self.peak_mib = self.initial_mib
        self.worker = threading.Thread(target=self._sample, daemon=True)
        self.worker.start()
        return self

    def __exit__(self, *args):
        self.stop_event.set()
        self.worker.join()
        self.peak_mib = max(
            self.peak_mib,
            self.process.memory_info().rss / (1024 ** 2)
        )


In [ ]:
def approximate_fitted_model_mib(estimator):
    """Estimate storage of the original inspected estimator arrays.
    
    Parameters
    ----------
    estimator : sklearn estimator
        Already fitted object inspected by the original feasibility code.
    
    Returns
    -------
    float
        Inspected-array bytes divided by 1024²; not complete process RSS.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    arrays = []
    if hasattr(estimator, 'support_vectors_'):       # SVR
        arrays = [estimator.support_vectors_, estimator.dual_coef_]
    elif hasattr(estimator, '_fit_X'):                # k-NN
        arrays = [estimator._fit_X, estimator._y]
    elif hasattr(estimator, 'estimators_'):           # Tree ensembles
        for tree in estimator.estimators_:
            t = tree.tree_
            arrays.extend([t.children_left, t.children_right,
                           t.feature, t.threshold, t.value])
    elif hasattr(estimator, 'coefs_'):                # MLP
        arrays = list(estimator.coefs_) + list(estimator.intercepts_)
    return sum(np.asarray(a).nbytes for a in arrays) / (1024 ** 2)


In [ ]:
def build_feasibility_arrays(model_name, spans, max_samples):
    """Build the original causal, nested-sample feasibility arrays.
    
    Parameters
    ----------
    model_name : str
        Original estimator family in the common search/feasibility loop.
    spans : sequence[int]
        Four ordered temporal spans in original 2-Hz samples.
    max_samples : int
        Maximum original nested fitting observations per profile for this diagnostic.
    
    Returns
    -------
    tuple
        Predictors, temperatures (°C), profile IDs, sample ranks, folds and seconds.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    tic = time.perf_counter()
    x_chunks, y_chunks, id_chunks, rank_chunks = [], [], [], []

    for pid, profile in train_df.groupby('profile_id', sort=False):
        order = nested_sample_order(len(profile), max_samples)
        positions = np.sort(np.asarray(order, dtype=int))
        position_rank = {int(pos): rank for rank, pos in enumerate(order)}

        # Features are derived from the COMPLETE 2-Hz input history first.
        base = make_paper_inputs(profile)
        temporal = make_temporal_features(base, profile['profile_id'], spans)
        x_chunks.append(temporal.iloc[positions].to_numpy(dtype=np.float64))
        y_chunks.append(profile['pm'].iloc[positions].to_numpy(dtype=np.float64))
        id_chunks.append(np.full(len(positions), int(pid), dtype=np.int32))
        rank_chunks.append(np.array(
            [position_rank[int(pos)] for pos in positions], dtype=np.int32
        ))
        del temporal, base

    X = np.vstack(x_chunks)
    y = np.concatenate(y_chunks)
    pids = np.concatenate(id_chunks)
    ranks = np.concatenate(rank_chunks)
    assert X.shape == (55 * max_samples, 108)
    assert np.isfinite(X).all() and np.isfinite(y).all()
    folds = np.array([FEASIBILITY_FOLDS[int(pid)] for pid in pids], dtype=np.int8)
    return X, y, pids, ranks, folds, time.perf_counter() - tic


In [ ]:
feasibility_records = []
feasibility_errors = []


**Atomic experiment block.** The following original timed or grouped-fold loop is kept together to preserve its execution scope and timer boundaries. Its inputs, mathematical objective and generated outputs are explained in this section.


In [ ]:
for model_number, model_name in enumerate(MODEL_ORDER, start=1):
    sizes = FEASIBILITY_SIZES[model_name]
    spans, hp = best_scaled_configuration(model_name)
    print(f'\nMODEL {model_number}/5: {model_name} | spans={spans}', flush=True)
    X, y, pids, ranks, folds, feature_seconds = build_feasibility_arrays(
        model_name, spans, max(sizes)
    )
    val_mask = (folds == VAL_FOLD) & (ranks < VAL_SAMPLES_PER_PROFILE)
    assert val_mask.sum() == 19 * VAL_SAMPLES_PER_PROFILE
    X_val, y_val = X[val_mask], y[val_mask]
    prev_fit_seconds = None

    for n in sizes:
        if prev_fit_seconds is not None and prev_fit_seconds > NEXT_SIZE_FIT_LIMIT_S:
            print(f'  SKIP {n}/profile: previous fit exceeded the time limit')
            break

        train_mask = (folds != VAL_FOLD) & (ranks < n)
        train_ids = set(pids[train_mask])
        val_ids = set(pids[val_mask])
        assert train_ids.isdisjoint(val_ids)
        assert train_mask.sum() == 36 * n

        X_train, y_train = X[train_mask], y[train_mask]
        estimator = make_regressor(model_name, hp)
        if model_name == 'MLP':
            estimator.max_iter = 400
            estimator.early_stopping = True
            estimator.validation_fraction = 0.10
            estimator.n_iter_no_change = 15

        pipeline = TransformedTargetRegressor(
            regressor=Pipeline([
                ('scaler', StandardScaler()),
                ('model', estimator)
            ]),
            transformer=StandardScaler()
        )

        try:
            with PeakProcessRAM() as ram:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always', ConvergenceWarning)
                    tic = time.perf_counter()
                    pipeline.fit(X_train, y_train)
                    fit_seconds = time.perf_counter() - tic
                    tic = time.perf_counter()
                    predictions = pipeline.predict(X_val)
                    predict_seconds = time.perf_counter() - tic

            assert np.isfinite(predictions).all()
            fitted = pipeline.regressor_.named_steps['model']
            warning_count = sum(issubclass(w.category, ConvergenceWarning)
                                for w in caught)
            mse = float(mean_squared_error(y_val, predictions))
            model_mib = approximate_fitted_model_mib(fitted)
            support_vectors = len(fitted.support_) if model_name == 'SVR' else np.nan

            feasibility_records.append({
                'Model': model_name,
                'Samples_per_profile': n,
                'Training_rows': len(y_train),
                'Fixed_validation_rows': len(y_val),
                'Feature_build_s': feature_seconds,
                'Fit_s': fit_seconds,
                'Predict_s': predict_seconds,
                'Process_peak_RSS_MiB': ram.peak_mib,
                'Fit_peak_increase_MiB': max(0., ram.peak_mib - ram.initial_mib),
                'Input_arrays_MiB': (X_train.nbytes + X_val.nbytes) / (1024 ** 2),
                'Estimator_array_MiB': model_mib,
                'Support_vectors': support_vectors,
                'Fold1_MSE_diagnostic': mse,
                'Convergence_warnings': warning_count,
                'Status': 'Completed'
            })
            pd.DataFrame(feasibility_records).to_csv(
                os.path.join(FEASIBILITY_DIR, 'feasibility_measurements.csv'),
                index=False
            )
            prev_fit_seconds = fit_seconds
            print(f'  {n}/profile: train={len(y_train):,}, '
                  f'fit={fit_seconds:.1f}s, predict={predict_seconds:.1f}s, '
                  f'peak RSS={ram.peak_mib:.0f} MiB, MSE={mse:.3f}', flush=True)
        except Exception as error:
            feasibility_errors.append({'Model': model_name,
                                       'Samples_per_profile': n,
                                       'Error': repr(error)})
            print(f'  STOP {model_name} at {n}: {error!r}', flush=True)
            break
        finally:
            del pipeline, estimator, X_train, y_train
            gc.collect()

    del X, y, pids, ranks, folds, X_val, y_val
    gc.collect()


In [ ]:
feasibility_df = pd.DataFrame(feasibility_records)
assert not feasibility_df.empty, 'All training feasibility runs failed'
feasibility_df.to_csv(
    os.path.join(FEASIBILITY_DIR, 'feasibility_measurements.csv'), index=False
)
pd.DataFrame(feasibility_errors).to_csv(
    os.path.join(FEASIBILITY_DIR, 'feasibility_errors.csv'), index=False
)
print('\nCompleted feasibility fits:', len(feasibility_df))
print('Errors:', len(feasibility_errors))
display(feasibility_df.round(3))
print('Note: these are one-fold feasibility results, NOT final test-set scores.')


In [ ]:
from pmsm_project.plotting import set_figure_context
set_figure_context(section='11.1', source_notebook='fresh_reconstruction.ipynb', source_cell=111)


In [ ]:
# STEP 6.3D.8B — Summarise observed scaling (no further model training)
import os
import matplotlib.pyplot as plt
import pandas as pd

summary = pd.read_csv(
    os.path.join(FEASIBILITY_DIR, 'feasibility_measurements.csv')
)
summary = summary.sort_values(['Model', 'Training_rows'])
assert set(summary['Model']).issubset(set(MODEL_ORDER))

print('Model-specific largest completed sample:')
summary_max = (
    summary.sort_values('Training_rows')
    .groupby('Model', as_index=False)
    .tail(1)
    .copy()
)
summary_max['Fraction_of_full_development'] = (
    summary_max['Training_rows'] / len(train_df)
)
display(summary_max[
    ['Model', 'Samples_per_profile', 'Training_rows',
     'Fraction_of_full_development', 'Fit_s', 'Predict_s',
     'Process_peak_RSS_MiB', 'Convergence_warnings']
].round(3))

# Independent figure for training time
plt.figure(figsize=(9, 5))
for name, group in summary.groupby('Model'):
    plt.plot(group['Training_rows'], group['Fit_s'], marker='o', label=name)
plt.xlabel('Training observations (one development fold)')
plt.ylabel('Fitting time (seconds)')
plt.title('Step 6.3D.8 — Model Training Scalability')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Independent figure for approximate process RSS
plt.figure(figsize=(9, 5))
for name, group in summary.groupby('Model'):
    plt.plot(group['Training_rows'], group['Process_peak_RSS_MiB'],
             marker='o', label=name)
plt.xlabel('Training observations (one development fold)')
plt.ylabel('Approximate peak Python-process RSS (MiB)')
plt.title('Step 6.3D.8 — Observed Memory Usage')
plt.legend()
plt.grid(True)
plt.show()

summary_max.to_csv(
    os.path.join(FEASIBILITY_DIR, 'feasibility_largest_completed.csv'),
    index=False
)

print('PASS: Scalability assessment saved')
print('Caution: sampled-data benchmarks do not establish full-data feasibility.')
print('Step 6.4 must separately validate any proposed full-data fit.')


## 12. Efficient SVR investigations: original Stage 9A–9M


<a id="s-12-1"></a>

### 12.1. Stage 9A: fresh SVR reference and invariants

**Research trace:** RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.1.


#### 12.1.1. Purpose

Pass the newly selected larger Bayesian RBF-SVR configuration to the preserved efficient-SVR investigations.


#### 12.1.2. Mathematical basis and assumptions

The RBF kernel is $k(x,z)=\exp(-\gamma\|x-z\|^2)$ and the epsilon-insensitive loss is $L_\epsilon(r)=\max(0,|r|-\epsilon)$. The primal objective is $\frac12\|w\|^2+C\sum_iL_\epsilon(z_i-f(x_i))$ [SMOLA04]. $C$ and $\epsilon$ apply to the standardised target in the supplied pipelines. `gamma='scale'` is inferred only from fitting inputs; fresh spans, $C$ and $\epsilon$ come from the original GP search.


#### 12.1.3. Implementation and parameter choices

The adapter exposes equivalent original names only. Stage 9A reads scaled_results["SVR"], requires 12 finite evaluations and derives spans, C and epsilon from its minimum. Screening fold 1, seed 42 and 640 observations per profile remain invariant inputs.


#### 12.1.4. Results and verification evidence

Printed settings identify the exact current-run source of every Stage 9 reference parameter. Dataset/profile/fold assertions run before the kernel comparisons.


#### 12.1.5. Interpretation and limitations

Fresh parameter handoff is an intentional correction to historical hardcoded winners. It changes the parameter values between runs without changing the investigation algorithm.


In [ ]:
from pathlib import Path
import sklearn, scipy, importlib.metadata as imd, json
# Source code above uses a string output directory. Stage 9 historical cells
# expect pathlib.Path; conversion changes only file-handling behaviour.
OUTPUT_DIR = Path(OUTPUT_DIR)
(OUTPUT_DIR/'figures').mkdir(parents=True, exist_ok=True)
development_df = df.loc[~df.profile_id.isin(test_ids)].copy()
heldout_count = int(df.profile_id.isin(test_ids).sum())  # count audit only
LOCKED_TEST_IDS = [13,16,23,24,29,42,44,47,53,56,62,67,71,78]
development_ids = sorted(set(development_df.profile_id))
assert sorted(test_ids) == LOCKED_TEST_IDS
assert len(development_df) == 1_055_198
assert heldout_count == 275_618
assert len(profile_cv)==55
fold_train_val = {}
for fold in (1,2,3):
    group_validation = sorted(profile_cv.loc[profile_cv.cv_fold.eq(fold),'profile_id'].astype(int).tolist())
    group_training = sorted(set(development_ids)-set(group_validation))
    assert set(group_training).isdisjoint(group_validation)
    fold_train_val[fold]=(group_training,group_validation)
assert len(fold_train_val[1][0])==36 and len(fold_train_val[1][1])==19
print('PASS: stage 9 is linked to the original 55-profile, 3-fold development split')


In [ ]:
# Stage 5 retains all four budgets and three timing repetitions.
stage5_data=nested_df.copy()
from pmsm_project.features import vectorised_temporal_sample, stage6_feature_names
assert len(results_df)==36
assert stage5_data.groupby('profile_id').size().eq(640).all()


In [ ]:
# ============================================================
# STAGE 9A — Experimental configuration and invariants
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# SAME Stage-9 methods, but a genuinely fresh reference from this Run All's
# 12-evaluation, larger-sample Gaussian-process SVR search. No old winners.
assert len(scaled_results['SVR']) == SCALED_TRIALS == 12
_stage9_source = scaled_results['SVR']
_stage9_eligible = _stage9_source.loc[
    _stage9_source['mean_cv_mse'].map(np.isfinite)
].copy()
assert len(_stage9_eligible) == 12, 'Fresh SVR search was incomplete'
_stage9_best = _stage9_eligible.loc[_stage9_eligible['mean_cv_mse'].idxmin()]
STAGE9_SPANS = tuple(int(_stage9_best[f's{j}']) for j in (1, 2, 3, 4))
_stage9_hp = json.loads(_stage9_best['hyperparameters_json'])
STAGE9_SVR_HP = {
    'C': float(_stage9_hp['C']),
    'epsilon': float(_stage9_hp['epsilon']),
    'kernel': 'rbf',
    'gamma': 'scale'
}
assert np.isclose(STAGE9_SVR_HP['C'], _stage9_hp['C'])
assert len(STAGE9_SPANS) == 4 and list(STAGE9_SPANS) == sorted(STAGE9_SPANS)
STAGE9_CONFIG_SOURCE = 'current execution: larger-sample 12-trial GP SVR winner'

STAGE9_SEED = 42

# Initial computational budget
STAGE9_SAMPLES_PER_PROFILE = 640
STAGE9_SCREENING_FOLD = 1

# Output directory
STAGE9_DIR = OUTPUT_DIR / "stage9_efficient_svr"
STAGE9_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ------------------------------------------------------------
# Verify the original dataset and partition are unchanged
# ------------------------------------------------------------

assert len(development_df) == 1_055_198
assert len(development_ids) == 55
assert len(test_ids) == 14

assert sorted(test_ids) == LOCKED_TEST_IDS

assert set(development_ids).isdisjoint(set(test_ids))

assert set(profile_cv["profile_id"]) == set(development_ids)

assert set(profile_cv["cv_fold"]) == {1, 2, 3}

# Retrieve original fixed Fold 1 profile allocations
train_ids, val_ids = fold_train_val[STAGE9_SCREENING_FOLD]

assert len(train_ids) == 36
assert len(val_ids) == 19

assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(test_ids)
assert set(val_ids).isdisjoint(test_ids)

expected_train_rows = len(train_ids) * STAGE9_SAMPLES_PER_PROFILE
expected_val_rows = len(val_ids) * STAGE9_SAMPLES_PER_PROFILE

# ------------------------------------------------------------
# Print methodological verification
# ------------------------------------------------------------

print("STAGE 9A — CONFIGURATION CHECK")
print("=" * 50)

print("Development profiles:", len(development_ids))


In [ ]:
print("Held-out test profiles:", len(test_ids))

print("\nCross-validation:")
print("Screening fold:", STAGE9_SCREENING_FOLD)
print("Training profiles:", len(train_ids))
print("Validation profiles:", len(val_ids))

print("\nPlanned screening observations:")
print("Training:", expected_train_rows)
print("Validation:", expected_val_rows)

print("\nFresh Bayesian-selected reference SVR:")
print("C:", STAGE9_SVR_HP["C"])
print("Epsilon:", STAGE9_SVR_HP["epsilon"])
print("Temporal spans:", STAGE9_SPANS)

print("\nPASS: Stage 9A configuration verified.")


<a id="s-12-2"></a>

### 12.2. Stage 9B: common screened dataset and feature parity

**Research trace:** RQ1, RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.2.


#### 12.2.1. Purpose

Use a common development-only feature dataset for fair kernel and direct-linear comparisons.


#### 12.2.2. Mathematical basis and assumptions

For a span $s$ measured in samples, $\alpha=2/(s+1)$ and $w_{t,j}=(1-\alpha)^{t-j}$ for $j\leq t$. The adjusted mean and biased standard deviation are
$$\mu_t=\frac{\sum_{j=0}^{t}w_{t,j}x_j}{\sum_{j=0}^{t}w_{t,j}},\qquad
\sigma_t=\sqrt{\frac{\sum_{j=0}^{t}w_{t,j}(x_j-\mu_t)^2}{\sum_{j=0}^{t}w_{t,j}}}.$$
Pandas `adjust=True` and `std(bias=True)` implement these moments [PANDAS]. Thus $\mu_0=x_0$ and $\sigma_0=0$. Four spans for each of 12 inputs give $12+2(12)(4)=108$ features. Filters reset at every recording and use full original chronological histories before selecting rows. A span of 200 samples corresponds to 100 s of sample count at 2 Hz; an exponential filter has infinite decaying support rather than a hard 100-s window.


#### 12.2.3. Implementation and parameter choices

Retain 640 selected rows per profile, complete pre-sampling histories, fixed feature ordering and the original comparison to the Stage 5 nested data. Fold 1 gives 36 fitting and 19 validation profiles.


#### 12.2.4. Results and verification evidence

stage9_data has 55×640 rows; screened matrices have 23,040 fitting and 12,160 validation rows, each with 108 features. Numerical parity and profile-isolation assertions establish the shared input contract.


#### 12.2.5. Interpretation and limitations

Identical input arrays are necessary for a controlled solver comparison but do not make distinct loss/intercept formulations equivalent.


In [ ]:
# ============================================================
# STAGE 9B.1 — Regenerate optimised temporal features
# ============================================================

import time
import numpy as np
import pandas as pd

start_time = time.perf_counter()

STAGE9_MAX_SAMPLES = STAGE9_SAMPLES_PER_PROFILE

# Original development-only fold assignments
stage9_fold_map = (
    profile_cv.set_index("profile_id")["cv_fold"]
    .astype(int)
    .to_dict()
)

stage9_parts = []


In [ ]:
# ------------------------------------------------------------
# Process each complete chronological development profile
# ------------------------------------------------------------

for count, (pid, profile) in enumerate(
    development_df.groupby("profile_id", sort=False),
    start=1
):

    # Verify chronological order within the recording
    assert profile.index.is_monotonic_increasing

    # Reuse the validated nested sampling procedure
    selection_order = nested_sample_order(
        len(profile),
        STAGE9_MAX_SAMPLES
    )

    positions = np.sort(
        np.asarray(selection_order, dtype=int)
    )

    # Rank describes the order in which each sample
    # was added by the nested sampling algorithm
    rank_lookup = {
        int(position): rank
        for rank, position in enumerate(selection_order)
    }

    # Step 1: Construct instantaneous engineering predictors
    X_base = make_paper_inputs(profile)

    # Step 2: Generate temporal features from complete history,
    # then retrieve the selected rows
    X_sample = vectorised_temporal_sample(
        X_base,
        positions,
        STAGE9_SPANS
    )

    # Ensure consistent 108-column ordering
    ordered_features = stage6_feature_names(
        X_base.columns,
        STAGE9_SPANS
    )

    X_sample = X_sample.loc[:, ordered_features].copy()

    # Step 3: Attach target and audit metadata
    X_sample["pm_target"] = (
        profile["pm"].iloc[positions].to_numpy(dtype=np.float64)
    )

    X_sample["profile_id"] = int(pid)
    X_sample["raw_position"] = positions
    X_sample["source_row"] = X_sample.index.to_numpy()

    X_sample["nest_rank"] = [
        rank_lookup[int(position)]
        for position in positions
    ]

    X_sample["cv_fold"] = stage9_fold_map[int(pid)]

    stage9_parts.append(X_sample)

    if count % 10 == 0 or count == 55:
        print(f"Processed {count}/55 development profiles")


In [ ]:
# ------------------------------------------------------------
# Combine the selected observations
# ------------------------------------------------------------

stage9_data = pd.concat(
    stage9_parts,
    ignore_index=True
)

stage9_feature_cols = stage6_feature_names(
    make_paper_inputs(development_df.iloc[:2]).columns,
    STAGE9_SPANS
)

stage9_feature_time = time.perf_counter() - start_time

print("\nSTAGE 9B — FEATURE CONSTRUCTION")
print("=" * 50)
print("Observations:", f"{len(stage9_data):,}")
print("Profiles:", stage9_data.profile_id.nunique())
print("Predictors:", len(stage9_feature_cols))
print("Temporal spans:", STAGE9_SPANS)
print("Construction time:", round(stage9_feature_time, 2), "s")


In [ ]:
# ============================================================
# STAGE 9B.2 — Numerical and methodological verification
# ============================================================

# 1. Feature dimensions and finite values
assert len(stage9_data) == 55 * 640
assert len(stage9_feature_cols) == 108

assert stage9_feature_cols == list(stage9_data.columns[:108])

assert np.isfinite(
    stage9_data[stage9_feature_cols + ["pm_target"]]
    .to_numpy(dtype=np.float64)
).all()

# 2. No leakage from held-out test profiles
assert set(stage9_data.profile_id).isdisjoint(test_ids)

# 3. Exactly 640 unique sampling ranks per profile
assert stage9_data.groupby("profile_id").size().eq(640).all()

assert (
    stage9_data.groupby("profile_id")
    .nest_rank.nunique()
    .eq(640)
    .all()
)

assert stage9_data["source_row"].is_unique

# 4. Verify original targets and sample positions
source_rows = stage9_data["source_row"].to_numpy(dtype=int)


In [ ]:
np.testing.assert_allclose(
    stage9_data["pm_target"].to_numpy(),
    development_df.loc[source_rows, "pm"].to_numpy(),
    rtol=0,
    atol=0
)

# 5. Confirm identical sampling ranks and profile assignments
#    against Version 1.0 Stage 5.
comparison_cols = ["profile_id", "nest_rank", "cv_fold"]

assert stage9_data[comparison_cols].equals(
    stage5_data[comparison_cols]
), "Stage 9 sampling does not match the verified Stage 5."

# Verify the 12 instantaneous predictors are unchanged
base_cols = list(make_paper_inputs(development_df.iloc[:2]).columns)

np.testing.assert_allclose(
    stage9_data[base_cols].to_numpy(),
    stage5_data[base_cols].to_numpy(),
    rtol=0,
    atol=1e-12
)

# 6. Independent first-observation boundary condition
first_rows = stage9_data.groupby("profile_id", sort=False).head(1)

assert first_rows["raw_position"].eq(0).all()


In [ ]:
for span in STAGE9_SPANS:
    np.testing.assert_allclose(
        first_rows[f"i_q_ewma_s{span}"],
        first_rows["i_q"],
        rtol=0,
        atol=1e-12
    )
    np.testing.assert_allclose(
        first_rows[f"i_q_ewms_s{span}"],
        0.0,
        rtol=0,
        atol=1e-12
    )

print("PASS: Features, sampled positions and targets verified")

# ------------------------------------------------------------
# 7. Prepare the original fixed Fold 1
# ------------------------------------------------------------

stage9_train_mask = (
    stage9_data["cv_fold"].ne(STAGE9_SCREENING_FOLD)
)

stage9_val_mask = (
    stage9_data["cv_fold"].eq(STAGE9_SCREENING_FOLD)
)

stage9_train = stage9_data.loc[stage9_train_mask]
stage9_val = stage9_data.loc[stage9_val_mask]

# Ensure the intended grouped split
assert stage9_train.profile_id.nunique() == 36
assert stage9_val.profile_id.nunique() == 19

assert len(stage9_train) == 23_040
assert len(stage9_val) == 12_160


In [ ]:
assert set(stage9_train.profile_id).isdisjoint(
    set(stage9_val.profile_id)
)

# ------------------------------------------------------------
# 8. Establish common model input arrays
# ------------------------------------------------------------

X9_train = stage9_train[
    stage9_feature_cols
].to_numpy(dtype=np.float64)

y9_train = stage9_train[
    "pm_target"
].to_numpy(dtype=np.float64)

X9_val = stage9_val[
    stage9_feature_cols
].to_numpy(dtype=np.float64)

y9_val = stage9_val[
    "pm_target"
].to_numpy(dtype=np.float64)

assert X9_train.shape == (23_040, 108)
assert X9_val.shape == (12_160, 108)

assert y9_train.shape == (23_040,)
assert y9_val.shape == (12_160,)

print("\nSTAGE 9B — FINAL VERIFICATION")
print("=" * 50)

print("Training:")
print("  Profiles:", stage9_train.profile_id.nunique())
print("  X shape:", X9_train.shape)
print("  y shape:", y9_train.shape)


In [ ]:
print("\nValidation:")
print("  Profiles:", stage9_val.profile_id.nunique())
print("  X shape:", X9_val.shape)
print("  y shape:", y9_val.shape)

print("\nPASS: Stage 9B completed.")
print("No estimators fitted; held-out test profiles untouched.")


<a id="s-12-3"></a>

### 12.3. Stage 9C: training-only scaling and Nyström kernel check

**Research trace:** RQ1, RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.3.


#### 12.3.1. Purpose

Verify the nonlinear approximation separately from predictive accuracy.


#### 12.3.2. Mathematical basis and assumptions

With landmarks B, $Z=K_{XB}K_{BB}^{-1/2}$ approximates the RBF Gram matrix as $ZZ^T\approx K_{XX}$ [WILLIAMS01]. The relative Frobenius error is $\|K-ZZ^T\|_F/\|K\|_F$. A shared gamma calculated from fitting inputs matches the exact and approximate kernels.


#### 12.3.3. Implementation and parameter choices

Fit input and target scalers only on the screened fitting rows. Retain the original 192-point/192-landmark kernel reconstruction probe and 1e-5 relative-error tolerance.


#### 12.3.4. Results and verification evidence

The standardisation moments, gamma and kernel reconstruction error are printed. The kernel test is numerical verification; validation MSE appears only in subsequent prediction experiments.


#### 12.3.5. Interpretation and limitations

A good probe reconstruction does not guarantee accurate temperature predictions or a sufficient number of landmarks on the full domain. Ill-conditioned landmark kernels also limit numerical accuracy.


In [ ]:
# ============================================================
# STAGE 9C.1 — Standardisation and shared RBF gamma
# ============================================================

import numpy as np

from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. Fit input scaler on training observations ONLY
# ------------------------------------------------------------

stage9_x_scaler = StandardScaler()

X9_train_scaled = stage9_x_scaler.fit_transform(
    X9_train
)

X9_val_scaled = stage9_x_scaler.transform(
    X9_val
)

# ------------------------------------------------------------
# 2. Fit target scaler on training temperatures ONLY
# ------------------------------------------------------------

stage9_y_scaler = StandardScaler()

y9_train_scaled = (
    stage9_y_scaler.fit_transform(
        y9_train.reshape(-1, 1)
    ).ravel()
)


In [ ]:
# The validation targets remain in physical units (°C).
# They are not used to fit any scaler.

# ------------------------------------------------------------
# 3. Verify shapes, finite values and transformations
# ------------------------------------------------------------

assert X9_train_scaled.shape == (23_040, 108)
assert X9_val_scaled.shape == (12_160, 108)

assert y9_train_scaled.shape == (23_040,)

assert np.isfinite(X9_train_scaled).all()
assert np.isfinite(X9_val_scaled).all()
assert np.isfinite(y9_train_scaled).all()

# Training input means should be approximately zero.
np.testing.assert_allclose(
    X9_train_scaled.mean(axis=0),
    np.zeros(108),
    atol=1e-9
)

# The standardised target should have zero mean
# and unit population standard deviation.
np.testing.assert_allclose(
    y9_train_scaled.mean(),
    0.0,
    atol=1e-10
)

np.testing.assert_allclose(
    y9_train_scaled.std(),
    1.0,
    atol=1e-10
)


In [ ]:
# Verify that the target scaling is reversible.
reconstructed_y = (
    stage9_y_scaler.inverse_transform(
        y9_train_scaled.reshape(-1, 1)
    ).ravel()
)

np.testing.assert_allclose(
    reconstructed_y,
    y9_train,
    rtol=1e-12,
    atol=1e-9
)

# ------------------------------------------------------------
# 4. Compute the reference RBF kernel parameter
# ------------------------------------------------------------

assert STAGE9_SVR_HP["gamma"] == "scale"

STAGE9_GAMMA = 1.0 / (
    X9_train_scaled.shape[1] *
    X9_train_scaled.var()
)

assert np.isfinite(STAGE9_GAMMA)
assert STAGE9_GAMMA > 0

# For all nonconstant features successfully standardised,
# gamma should be close to 1/108.
constant_features = int(
    np.sum(stage9_x_scaler.var_ == 0)
)


In [ ]:
# ------------------------------------------------------------
# 5. Report results
# ------------------------------------------------------------

print("STAGE 9C.1 — STANDARDISATION VERIFICATION")
print("=" * 55)

print("Training X:", X9_train_scaled.shape)
print("Validation X:", X9_val_scaled.shape)

print("\nTarget scaling:")
print("Training mean (°C):", stage9_y_scaler.mean_[0])
print("Training std (°C):", stage9_y_scaler.scale_[0])

print("\nRBF kernel:")
print("Reference gamma:", STAGE9_GAMMA)
print("1 / 108:", 1 / 108)
print("Constant input features:", constant_features)

print("\nPASS: Training-only standardisation verified.")


In [ ]:
# ============================================================
# STAGE 9C.2 — Verify Nyström kernel approximation
# ============================================================

import numpy as np
import pandas as pd

from sklearn.kernel_approximation import Nystroem
from sklearn.metrics.pairwise import rbf_kernel

# ------------------------------------------------------------
# 1. Select a small reproducible training-only probe
# ------------------------------------------------------------

rng = np.random.default_rng(STAGE9_SEED)

probe_indices = rng.choice(
    len(X9_train_scaled),
    size=192,
    replace=False
)

X9_probe = X9_train_scaled[probe_indices]

assert X9_probe.shape == (192, 108)

# ------------------------------------------------------------
# 2. Calculate the exact RBF kernel
# ------------------------------------------------------------

K_exact = rbf_kernel(
    X9_probe,
    X9_probe,
    gamma=STAGE9_GAMMA
)

assert K_exact.shape == (192, 192)


In [ ]:
np.testing.assert_allclose(
    K_exact,
    K_exact.T,
    atol=1e-12
)

# ------------------------------------------------------------
# 3. Test different Nyström component counts
# ------------------------------------------------------------

component_counts = (24, 64, 128, 192)

kernel_records = []


In [ ]:
for m in component_counts:

    transformer = Nystroem(
        kernel="rbf",
        gamma=STAGE9_GAMMA,
        n_components=m,
        random_state=STAGE9_SEED
    )

    # Fit the Nyström basis and transform the probe.
    Z = transformer.fit_transform(X9_probe)

    assert Z.shape == (192, m)
    assert np.isfinite(Z).all()

    # Reconstruct the approximate kernel.
    K_approx = Z @ Z.T

    relative_error = (
        np.linalg.norm(K_exact - K_approx, ord="fro")
        /
        np.linalg.norm(K_exact, ord="fro")
    )

    assert np.isfinite(relative_error)
    assert relative_error >= 0

    kernel_records.append({
        "Components": m,
        "Kernel_Relative_Error": float(relative_error)
    })

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

stage9_kernel_results = pd.DataFrame(kernel_records)


In [ ]:
display(stage9_kernel_results)

# Full basis should reconstruct this small kernel
# to within numerical precision.
full_basis_error = stage9_kernel_results.loc[
    stage9_kernel_results["Components"].eq(192),
    "Kernel_Relative_Error"
].iloc[0]

assert full_basis_error < 1e-5, (
    "Full-basis reconstruction error is unexpectedly large."
)

print("\nPASS: Nyström kernel construction verified.")
print("Full-basis relative error:", full_basis_error)
print("No regressors fitted; held-out test data untouched.")


<a id="s-12-4"></a>

### 12.4. Stage 9D: controlled four-method screening

**Research trace:** RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.4.


#### 12.4.1. Purpose

Compare exact RBF-SVR, two Nyström/linear variants and direct LinearSVR on the same screened fold.


#### 12.4.2. Mathematical basis and assumptions

The RBF kernel is $k(x,z)=\exp(-\gamma\|x-z\|^2)$ and the epsilon-insensitive loss is $L_\epsilon(r)=\max(0,|r|-\epsilon)$. The primal objective is $\frac12\|w\|^2+C\sum_iL_\epsilon(z_i-f(x_i))$ [SMOLA04]. $C$ and $\epsilon$ apply to the standardised target in the supplied pipelines. `gamma='scale'` is inferred only from fitting inputs; fresh spans, $C$ and $\epsilon$ come from the original GP search.

Direct LinearSVR uses $f(x)=w^Tx+b$ with $\frac12\|w\|^2+C\sum_i\max(0,|z_i-f(x_i)|-\epsilon)$, subject to LIBLINEAR’s intercept-regularisation convention [FAN08,SKLEARN]. It removes the nonlinear kernel. The final configuration keeps `loss='epsilon_insensitive'`, `dual=True`, `tol=1e-4`, `max_iter=10000` and seed 42. The inherited $\epsilon$ and spans originate in the fresh larger RBF search; Stage 9 selects the reference $C$.


#### 12.4.3. Implementation and parameter choices

Retain the exact original model definitions, shared gamma, training/validation matrices, fit/prediction timing boundaries, support-vector/iteration diagnostics and process-RSS monitoring. The original timed per-model loop is kept atomic.


#### 12.4.4. Results and verification evidence

stage9d_results records MSE/RMSE, training and prediction seconds, convergence warnings, support-vector/iteration diagnostics and RSS. It is a screening comparison on one external fold.


#### 12.4.5. Interpretation and limitations

LIBSVM and LIBLINEAR differ in intercept treatment; matched C and epsilon do not make them mathematically identical. One-fold screening cannot support a universal ranking.


In [ ]:
# ============================================================
# STAGE 9D — Four-method SVR screening
# ============================================================

import os
import gc
import time
import psutil
import warnings
import threading

import numpy as np
import pandas as pd

from sklearn.svm import SVR, LinearSVR
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9D = True  # Change to True to run

# ------------------------------------------------------------
# 1. Reference hyperparameters
# ------------------------------------------------------------

C_REF = STAGE9_SVR_HP["C"]
EPSILON_REF = STAGE9_SVR_HP["epsilon"]


In [ ]:
def make_linear_svr():
    """Construct the original Stage 9 screening LinearSVR.
    
    
    Returns
    -------
    sklearn.svm.LinearSVR
        Unfitted estimator using this run’s C/epsilon and original screening limits.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    return LinearSVR(
        C=C_REF,
        epsilon=EPSILON_REF,
        loss="epsilon_insensitive",
        dual=True,
        tol=1e-4,
        max_iter=3000,
        random_state=STAGE9_SEED
    )


In [ ]:
def make_nystroem_svr(components):
    """Construct the original matched-RBF Nyström and linear-SVR pipeline.
    
    Parameters
    ----------
    components : int
        Nyström landmark count from the original experimental grid.
    
    Returns
    -------
    sklearn.pipeline.Pipeline
        Unfitted kernel map and screening estimator; no input/target rescaling here.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    return Pipeline([
        (
            "nystroem",
            Nystroem(
                kernel="rbf",
                gamma=STAGE9_GAMMA,
                n_components=components,
                random_state=STAGE9_SEED,
                n_jobs=1
            )
        ),
        ("linear_svr", make_linear_svr())
    ])


In [ ]:
stage9d_models = {
    "Exact RBF-SVR": SVR(
        kernel="rbf",
        C=C_REF,
        epsilon=EPSILON_REF,
        gamma=STAGE9_GAMMA,
        cache_size=512
    ),
    "Nystroem-128": make_nystroem_svr(128),
    "Nystroem-256": make_nystroem_svr(256),
    "Direct LinearSVR": make_linear_svr()
}


In [ ]:
# ------------------------------------------------------------
# 2. Process-memory sampling helper
# ------------------------------------------------------------

def measure_peak_memory(stop_event, measurements):
    """Sample whole-process resident memory until the original stop event.
    
    Parameters
    ----------
    stop_event : threading.Event
        Stop signal used by the original timing block.
    measurements : list[int]
        Mutable process-RSS samples in bytes.
    
    Returns
    -------
    None
        Appends RSS byte values to the supplied list; does not isolate model allocations.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    process = psutil.Process(os.getpid())

    while not stop_event.is_set():
        try:
            rss = process.memory_info().rss
            measurements.append(rss)
        except psutil.Error:
            break

        stop_event.wait(0.05)


In [ ]:
# ------------------------------------------------------------
# 3. Fit, predict and record
# ------------------------------------------------------------

stage9d_records = []

if RUN_STAGE9D:

    assert X9_train_scaled.shape == (23040, 108)
if RUN_STAGE9D:
    assert X9_val_scaled.shape == (12160, 108)
if RUN_STAGE9D:

    assert np.isfinite(X9_train_scaled).all()
if RUN_STAGE9D:
    assert np.isfinite(X9_val_scaled).all()


**Atomic experiment block.** The following original timed or grouped-fold loop is kept together to preserve its execution scope and timer boundaries. Its inputs, mathematical objective and generated outputs are explained in this section.


In [ ]:
if RUN_STAGE9D:

    for name, model in stage9d_models.items():

        print("\n" + "=" * 55)
        print("Training:", name)

        gc.collect()

        process = psutil.Process(os.getpid())
        rss_before = process.memory_info().rss

        memory_samples = [rss_before]
        stop_event = threading.Event()

        monitor = threading.Thread(
            target=measure_peak_memory,
            args=(stop_event, memory_samples),
            daemon=True
        )

        monitor.start()

        try:
            # Fit entire estimator, including Nyström mapping
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter(
                    "always",
                    ConvergenceWarning
                )

                t0 = time.perf_counter()

                model.fit(
                    X9_train_scaled,
                    y9_train_scaled
                )

                fit_seconds = time.perf_counter() - t0

            convergence_count = sum(
                issubclass(w.category, ConvergenceWarning)
                for w in caught
            )

            # Predict, including any Nyström transformation
            t1 = time.perf_counter()

            pred_scaled = model.predict(X9_val_scaled)

            pred_celsius = (
                stage9_y_scaler.inverse_transform(
                    pred_scaled.reshape(-1, 1)
                ).ravel()
            )

            predict_seconds = time.perf_counter() - t1

            assert np.isfinite(pred_celsius).all()

            mse = float(
                np.mean((y9_val - pred_celsius) ** 2)
            )

            rmse = float(np.sqrt(mse))

            status = (
                "convergence_warning"
                if convergence_count > 0
                else "success"
            )

            record = {
                "Method": name,
                "Status": status,
                "Training_Rows": len(X9_train_scaled),
                "Validation_Rows": len(X9_val_scaled),
                "MSE_C2": mse,
                "RMSE_C": rmse,
                "Fit_Seconds": fit_seconds,
                "Predict_Seconds": predict_seconds,
                "Convergence_Warnings": convergence_count
            }

            print("MSE (°C²):", round(mse, 4))
            print("RMSE (°C):", round(rmse, 4))
            print("Fit time (s):", round(fit_seconds, 3))
            print("Prediction time (s):",
                  round(predict_seconds, 3))
            print("Status:", status)

        except Exception as exc:
            record = {
                "Method": name,
                "Status": "failed",
                "Training_Rows": len(X9_train_scaled),
                "Validation_Rows": len(X9_val_scaled),
                "MSE_C2": np.nan,
                "RMSE_C": np.nan,
                "Fit_Seconds": np.nan,
                "Predict_Seconds": np.nan,
                "Convergence_Warnings": np.nan,
                "Error": repr(exc)
            }
            print("FAILED:", repr(exc))

        finally:
            stop_event.set()
            monitor.join(timeout=2)

            memory_samples.append(
                process.memory_info().rss
            )

            peak_rss = max(memory_samples)

            record["Peak_RSS_GiB"] = (
                peak_rss / (1024 ** 3)
            )

            record["Peak_RSS_Increase_GiB"] = (
                max(0, peak_rss - rss_before) / (1024 ** 3)
            )

        stage9d_records.append(record)

        # Save after every method, including failed attempts
        stage9d_results = pd.DataFrame(stage9d_records)

        stage9d_results.to_csv(
            STAGE9_DIR / "stage9d_fold1_screening.csv",
            index=False
        )


In [ ]:
if RUN_STAGE9D:

    print("\nSTAGE 9D — SCREENING RESULTS")
if RUN_STAGE9D:
    display(stage9d_results)
else:
    print(
        "Stage 9D is disabled. "
        "Set RUN_STAGE9D = True to execute screening."
    )


In [ ]:
from pmsm_project.plotting import set_figure_context
set_figure_context(section='12.4', source_notebook='fresh_reconstruction.ipynb', source_cell=133)


In [ ]:
# ============================================================
# STAGE 9D — Accuracy and computational-cost summary
# ============================================================

assert "stage9d_results" in globals(), (
    "Run the Stage 9D screening experiment first."
)

summary_cols = [
    "Method",
    "Status",
    "MSE_C2",
    "RMSE_C",
    "Fit_Seconds",
    "Predict_Seconds",
    "Peak_RSS_GiB",
    "Convergence_Warnings"
]

stage9d_summary = stage9d_results[
    summary_cols
].copy()

display(
    stage9d_summary.sort_values(
        "MSE_C2",
        na_position="last"
    ).round(4)
)

# Plot observed accuracy versus fitting time
import matplotlib.pyplot as plt

valid = stage9d_summary[
    stage9d_summary["MSE_C2"].notna()
]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for _, row in valid.iterrows():
    ax.scatter(
        row["Fit_Seconds"],
        row["MSE_C2"],
        s=80
    )

    ax.annotate(
        row["Method"],
        (row["Fit_Seconds"], row["MSE_C2"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9
    )

ax.set_xlabel("Model fitting time (seconds)")
ax.set_ylabel("Validation MSE (°C²)")
ax.set_title("Stage 9D: SVR Accuracy–Cost Screening")
ax.grid(alpha=0.25)

plt.show()


<a id="s-12-5"></a>

### 12.5. Stage 9E: regularisation and convergence sensitivity

**Research trace:** RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.5.


#### 12.5.1. Purpose

Check whether apparent solver performance depends on regularisation or incomplete convergence.


#### 12.5.2. Mathematical basis and assumptions

For the sum-loss objective $\frac12\|w\|^2+C\sum_iL_\epsilon$, increasing C weakens relative regularisation. A convergence warning indicates the stopping condition was not satisfied; low observed MSE alone does not establish a reliable solve.


#### 12.5.3. Implementation and parameter choices

Retain the original inner profile split, C factors, loss/dual choices, tolerances, iteration budgets and warning capture. C values are derived from the current-run Bayesian RBF reference rather than historical winners.


#### 12.5.4. Results and verification evidence

The original inner-validation results report C, prediction errors, iteration counts, convergence and time. Warning counts remain evidence; they are not suppressed to improve the table.


#### 12.5.5. Interpretation and limitations

This sensitivity investigation does not replace the preceding Bayesian optimisation. Choices based on an inner split still require the subsequent outer grouped validation.


In [ ]:
# ============================================================
# STAGE 9E.1 — Inner development-only grouped split
# ============================================================

import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# These are the same 23,040 unscaled training rows
# previously used in Stage 9D.
inner_groups = stage9_train["profile_id"].to_numpy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=STAGE9_SEED
)

fit_idx, check_idx = next(
    splitter.split(
        X9_train,
        y9_train,
        groups=inner_groups
    )
)

# Verify group isolation
fit_groups = set(inner_groups[fit_idx])
check_groups = set(inner_groups[check_idx])

assert fit_groups.isdisjoint(check_groups)
assert len(fit_groups) == 27
assert len(check_groups) == 9

# Refit both scalers on inner-training rows ONLY
x_scaler_9e = StandardScaler()
y_scaler_9e = StandardScaler()


In [ ]:
X9e_fit = x_scaler_9e.fit_transform(
    X9_train[fit_idx]
)

X9e_check = x_scaler_9e.transform(
    X9_train[check_idx]
)

y9e_fit = y_scaler_9e.fit_transform(
    y9_train[fit_idx].reshape(-1, 1)
).ravel()

y9e_check_celsius = y9_train[check_idx]

# Compute gamma from inner-training features
GAMMA_9E = 1.0 / (
    X9e_fit.shape[1] * X9e_fit.var()
)

assert X9e_fit.shape == (17280, 108)
assert X9e_check.shape == (5760, 108)

assert np.isfinite(X9e_fit).all()
assert np.isfinite(X9e_check).all()
assert np.isfinite(y9e_fit).all()

print("STAGE 9E — INNER SPLIT")
print("Inner training:", X9e_fit.shape)
print("Inner validation:", X9e_check.shape)
print("Training profiles:", len(fit_groups))
print("Validation profiles:", len(check_groups))
print("RBF gamma:", GAMMA_9E)
print("PASS: Grouped split and scaling verified.")


In [ ]:
# ============================================================
# STAGE 9E.2 — C sensitivity screening
# ============================================================

import time
import warnings
import pandas as pd

from sklearn.svm import LinearSVR
from sklearn.kernel_approximation import Nystroem
from sklearn.pipeline import Pipeline
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9E = True  # Set True to execute

C_GRID_9E = [
    STAGE9_SVR_HP["C"] / 10,
    STAGE9_SVR_HP["C"],
    STAGE9_SVR_HP["C"] * 10
]

METHODS_9E = {
    "Nystroem-128": 128,
    "Nystroem-256": 256,
    "Direct LinearSVR": None
}


In [ ]:
def build_stage9e_model(method, components, C):

    """Construct the original direct or Nyström regularisation candidate.
    
    Parameters
    ----------
    method : str
        Original direct-linear or Nyström method identifier.
    components : int
        Nyström landmark count from the original experimental grid.
    C : float
        Regularisation coefficient in standardised-target epsilon-insensitive loss.
    
    Returns
    -------
    sklearn estimator
        Unfitted estimator with unchanged stage-specific loss and stopping settings.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    linear_model = LinearSVR(
        C=float(C),
        epsilon=STAGE9_SVR_HP["epsilon"],
        loss="epsilon_insensitive",
        dual=True,
        tol=1e-4,
        max_iter=10000,
        random_state=STAGE9_SEED
    )

    if components is None:
        return linear_model

    return Pipeline([
        ("nystroem", Nystroem(
            kernel="rbf",
            gamma=GAMMA_9E,
            n_components=components,
            random_state=STAGE9_SEED,
            n_jobs=1
        )),
        ("linear_svr", linear_model)
    ])


In [ ]:
if RUN_STAGE9E:

    stage9e_records = []


In [ ]:
if RUN_STAGE9E:

    for method, components in METHODS_9E.items():
        for C in C_GRID_9E:

            model = build_stage9e_model(
                method, components, C
            )

            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                fit_start = time.perf_counter()
                model.fit(X9e_fit, y9e_fit)
                fit_time = time.perf_counter() - fit_start

            convergence_warnings = sum(
                issubclass(w.category, ConvergenceWarning)
                for w in caught
            )

            prediction_start = time.perf_counter()

            predictions_scaled = model.predict(X9e_check)

            predictions_celsius = (
                y_scaler_9e.inverse_transform(
                    predictions_scaled.reshape(-1, 1)
                ).ravel()
            )

            prediction_time = (
                time.perf_counter() - prediction_start
            )

            mse = float(np.mean(
                (y9e_check_celsius - predictions_celsius) ** 2
            ))

            record = {
                "Method": method,
                "C": float(C),
                "Components": components,
                "MSE_C2": mse,
                "RMSE_C": float(np.sqrt(mse)),
                "Fit_Seconds": fit_time,
                "Predict_Seconds": prediction_time,
                "Convergence_Warnings": convergence_warnings,
                "Status": (
                    "convergence_warning"
                    if convergence_warnings else "success"
                )
            }

            stage9e_records.append(record)

            print(
                f"{method:18s} "
                f"C={C:.5g} "
                f"MSE={mse:.3f} "
                f"Fit={fit_time:.2f}s "
                f"Warnings={convergence_warnings}"
            )

            pd.DataFrame(stage9e_records).to_csv(
                STAGE9_DIR / "stage9e_C_sensitivity.csv",
                index=False
            )


In [ ]:
if RUN_STAGE9E:

    stage9e_results = pd.DataFrame(stage9e_records)
if RUN_STAGE9E:

    print("\nSTAGE 9E RESULTS")
if RUN_STAGE9E:
    display(stage9e_results.round(4))
else:
    print("Set RUN_STAGE9E = True to run the experiment.")


<a id="s-12-6"></a>

### 12.6. Stage 9F: Nyström approximation capacity

**Research trace:** RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.6.


#### 12.6.1. Purpose

Assess whether insufficient landmark capacity explains approximation error.


#### 12.6.2. Mathematical basis and assumptions

Increasing the landmark count m raises approximation capacity but also requires an n×m feature matrix and landmark-kernel factorisation. Nyström reduces exact-kernel cost through approximation; it does not promise identical SVR predictions.


#### 12.6.3. Implementation and parameter choices

Retain the original landmark grid, C grid, inner fitting/check datasets and shared kernel parameters. No new search budget or algorithm is substituted.


#### 12.6.4. Results and verification evidence

stage9f_results and the combined inner-validation comparison expose error/cost/convergence across the original capacities. Every displayed figure is indexed by section and source cell.


#### 12.6.5. Interpretation and limitations

The observed finite landmark grid cannot prove the best possible approximation. Larger m can improve kernel fidelity while making memory and solver convergence less practical.


In [ ]:
# ============================================================
# STAGE 9F — Nyström-512 capacity investigation
# ============================================================

import time
import warnings
import gc
import os
import threading

import numpy as np
import pandas as pd
import psutil

from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9F = True  # Change to True to run

C_GRID_9F = [
    STAGE9_SVR_HP["C"],
    STAGE9_SVR_HP["C"] * 10
]


In [ ]:
def stage9f_memory_monitor(stop_event, samples):
    """Sample whole-process resident memory until the original stop event.
    
    Parameters
    ----------
    stop_event : threading.Event
        Stop signal used by the original timing block.
    samples : list[int]
        Mutable process-RSS samples in bytes.
    
    Returns
    -------
    None
        Appends RSS byte values to the supplied list; does not isolate model allocations.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    process = psutil.Process(os.getpid())

    while not stop_event.is_set():
        try:
            samples.append(process.memory_info().rss)
        except psutil.Error:
            break
        stop_event.wait(0.05)

if RUN_STAGE9F:

    assert X9e_fit.shape == (17280, 108)
if RUN_STAGE9F:
    assert X9e_check.shape == (5760, 108)
if RUN_STAGE9F:

    stage9f_records = []


**Atomic experiment block.** The following original timed or grouped-fold loop is kept together to preserve its execution scope and timer boundaries. Its inputs, mathematical objective and generated outputs are explained in this section.


In [ ]:
if RUN_STAGE9F:

    for C in C_GRID_9F:

        model = build_stage9e_model(
            method="Nystroem-512",
            components=512,
            C=C
        )

        gc.collect()

        process = psutil.Process(os.getpid())
        baseline_rss = process.memory_info().rss

        memory_samples = [baseline_rss]
        stop_event = threading.Event()

        monitor = threading.Thread(
            target=stage9f_memory_monitor,
            args=(stop_event, memory_samples),
            daemon=True
        )

        monitor.start()

        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                t0 = time.perf_counter()
                model.fit(X9e_fit, y9e_fit)
                fit_seconds = time.perf_counter() - t0

            warning_count = sum(
                issubclass(w.category, ConvergenceWarning)
                for w in caught
            )

            t1 = time.perf_counter()
            predictions_scaled = model.predict(X9e_check)

            predictions_celsius = (
                y_scaler_9e.inverse_transform(
                    predictions_scaled.reshape(-1, 1)
                ).ravel()
            )
            predict_seconds = time.perf_counter() - t1

            mse = float(np.mean(
                (y9e_check_celsius - predictions_celsius) ** 2
            ))

            record = {
                "Method": "Nystroem-512",
                "Components": 512,
                "C": float(C),
                "MSE_C2": mse,
                "RMSE_C": float(np.sqrt(mse)),
                "Fit_Seconds": fit_seconds,
                "Predict_Seconds": predict_seconds,
                "Convergence_Warnings": warning_count,
                "Status": (
                    "convergence_warning"
                    if warning_count else "success"
                )
            }

        except Exception as exc:
            record = {
                "Method": "Nystroem-512",
                "Components": 512,
                "C": float(C),
                "MSE_C2": np.nan,
                "RMSE_C": np.nan,
                "Fit_Seconds": np.nan,
                "Predict_Seconds": np.nan,
                "Convergence_Warnings": np.nan,
                "Status": "failed",
                "Error": repr(exc)
            }

        finally:
            stop_event.set()
            monitor.join(timeout=2)

            memory_samples.append(process.memory_info().rss)

            peak_rss = max(memory_samples)

            record["Peak_Process_RSS_GiB"] = (
                peak_rss / 1024**3
            )

            record["RSS_Increase_GiB"] = (
                max(0, peak_rss - baseline_rss) / 1024**3
            )

        stage9f_records.append(record)

        pd.DataFrame(stage9f_records).to_csv(
            STAGE9_DIR / "stage9f_nystroem_512.csv",
            index=False
        )

        print(
            f"C={C:.6f}, "
            f"MSE={record['MSE_C2']:.4f}, "
            f"Fit={record['Fit_Seconds']:.2f}s, "
            f"Status={record['Status']}"
        )

        del model
        gc.collect()


In [ ]:
if RUN_STAGE9F:

    stage9f_results = pd.DataFrame(stage9f_records)
if RUN_STAGE9F:

    # Compare against existing Stage 9E results
    comparison_columns = [
        "Method", "Components", "C",
        "MSE_C2", "RMSE_C",
        "Fit_Seconds", "Predict_Seconds",
        "Convergence_Warnings", "Status"
    ]
if RUN_STAGE9F:

    stage9f_comparison = pd.concat(
        [
            stage9e_results[comparison_columns],
            stage9f_results[comparison_columns]
        ],
        ignore_index=True
    )
if RUN_STAGE9F:

    print("\nSTAGE 9F — COMBINED INNER-VALIDATION RESULTS")
if RUN_STAGE9F:

    display(
        stage9f_comparison.sort_values(
            "MSE_C2",
            na_position="last"
        ).round(4)
    )
else:
    print("Set RUN_STAGE9F = True to execute.")


<a id="s-12-7"></a>

### 12.7. Stage 9G: confirmation on the original outer fold

**Research trace:** RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.7.


#### 12.7.1. Purpose

Evaluate the methods selected on the screened inner data using the original outer validation profiles.


#### 12.7.2. Mathematical basis and assumptions

Each profile belongs to one validation fold. Its other two folds provide the fitting profiles. For fold $f$,
$$MSE_f=\frac1{N_f}\sum_{i\in f}(\hat T_i-T_i)^2,\qquad J=\frac13\sum_{f=1}^{3}MSE_f.$$
The objective $J$ has units °C² and weights folds equally. It is different from pooling every validation row before calculating MSE. Feature and target standardisers are fitted only on the fitting profiles and predictions are inverse-transformed before scoring. Profile-level temperature maxima define three strata for the fixed shuffled `StratifiedKFold(3, random_state=42)` assignment [ROBERTS17].


#### 12.7.3. Implementation and parameter choices

Retain the original inner-selected configurations and refit on all fold-1 fitting profiles. Scaling is fitted on those profiles; the outer validation profiles did not select the inner candidates.


#### 12.7.4. Results and verification evidence

stage9g_results reports the outer-fold errors, timing, convergence and resource diagnostics for the original compared methods.


#### 12.7.5. Interpretation and limitations

This stage remains a one-fold confirmation inside development. It is not the 14-profile final evaluation or an independently unseen project-level confirmation.


In [ ]:
# ============================================================
# STAGE 9G — Confirm selected approaches on Fold 1
# ============================================================

import gc
import os
import time
import warnings
import threading

import numpy as np
import pandas as pd
import psutil

from sklearn.svm import LinearSVR
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9G = True  # Set True to execute

# ------------------------------------------------------------
# 1. Selected configurations from Stage 9E
# ------------------------------------------------------------

C_LINEAR = STAGE9_SVR_HP["C"] / 10
C_NYSTROEM = STAGE9_SVR_HP["C"] * 10


In [ ]:
def stage9g_linear_model(C):
    """Construct the original stage-specific direct LinearSVR candidate.
    
    Parameters
    ----------
    C : float
        Regularisation coefficient in standardised-target epsilon-insensitive loss.
    
    Returns
    -------
    sklearn.svm.LinearSVR
        Unfitted estimator with fresh inherited epsilon and original loss/stopping settings.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    return LinearSVR(
        C=float(C),
        epsilon=STAGE9_SVR_HP["epsilon"],
        loss="epsilon_insensitive",
        dual=True,
        tol=1e-4,
        max_iter=10000,
        random_state=STAGE9_SEED
    )


In [ ]:
models_9g = {
    "Direct LinearSVR (selected)": stage9g_linear_model(
        C_LINEAR
    ),

    "Nystroem-256 (selected)": Pipeline([
        ("nystroem", Nystroem(
            kernel="rbf",
            gamma=STAGE9_GAMMA,
            n_components=256,
            random_state=STAGE9_SEED,
            n_jobs=1
        )),
        ("linear_svr", stage9g_linear_model(C_NYSTROEM))
    ])
}


In [ ]:
# ------------------------------------------------------------
# 2. Process-memory monitor
# ------------------------------------------------------------

def stage9g_memory_monitor(stop_event, samples):
    """Sample whole-process resident memory until the original stop event.
    
    Parameters
    ----------
    stop_event : threading.Event
        Stop signal used by the original timing block.
    samples : list[int]
        Mutable process-RSS samples in bytes.
    
    Returns
    -------
    None
        Appends RSS byte values to the supplied list; does not isolate model allocations.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    process = psutil.Process(os.getpid())

    while not stop_event.is_set():
        try:
            samples.append(process.memory_info().rss)
        except psutil.Error:
            break

        stop_event.wait(0.05)


In [ ]:
# ------------------------------------------------------------
# 3. Train, predict and evaluate
# ------------------------------------------------------------

if RUN_STAGE9G:

    assert X9_train_scaled.shape == (23040, 108)
if RUN_STAGE9G:
    assert X9_val_scaled.shape == (12160, 108)
if RUN_STAGE9G:

    stage9g_records = []


**Atomic experiment block.** The following original timed or grouped-fold loop is kept together to preserve its execution scope and timer boundaries. Its inputs, mathematical objective and generated outputs are explained in this section.


In [ ]:
if RUN_STAGE9G:

    for name, model in models_9g.items():

        print("\nTraining:", name)

        gc.collect()

        process = psutil.Process(os.getpid())
        initial_rss = process.memory_info().rss

        samples = [initial_rss]
        stop_event = threading.Event()

        monitor = threading.Thread(
            target=stage9g_memory_monitor,
            args=(stop_event, samples),
            daemon=True
        )
        monitor.start()

        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                t0 = time.perf_counter()

                model.fit(
                    X9_train_scaled,
                    y9_train_scaled
                )

                fit_seconds = time.perf_counter() - t0

            warning_count = sum(
                issubclass(w.category, ConvergenceWarning)
                for w in caught
            )

            t1 = time.perf_counter()

            pred_scaled = model.predict(X9_val_scaled)

            pred_celsius = (
                stage9_y_scaler.inverse_transform(
                    pred_scaled.reshape(-1, 1)
                ).ravel()
            )

            predict_seconds = time.perf_counter() - t1

            assert np.isfinite(pred_celsius).all()

            mse = float(np.mean(
                (y9_val - pred_celsius) ** 2
            ))

            record = {
                "Method": name,
                "C": (
                    C_LINEAR if "Direct" in name
                    else C_NYSTROEM
                ),
                "Training_Rows": len(X9_train_scaled),
                "Validation_Rows": len(X9_val_scaled),
                "MSE_C2": mse,
                "RMSE_C": float(np.sqrt(mse)),
                "Fit_Seconds": fit_seconds,
                "Predict_Seconds": predict_seconds,
                "Convergence_Warnings": warning_count,
                "Status": (
                    "convergence_warning"
                    if warning_count else "success"
                )
            }

        except Exception as exc:
            record = {
                "Method": name,
                "MSE_C2": np.nan,
                "RMSE_C": np.nan,
                "Fit_Seconds": np.nan,
                "Predict_Seconds": np.nan,
                "Convergence_Warnings": np.nan,
                "Status": "failed",
                "Error": repr(exc)
            }

        finally:
            stop_event.set()
            monitor.join(timeout=2)

            samples.append(process.memory_info().rss)

            peak_rss = max(samples)

            record["Peak_RSS_GiB"] = peak_rss / 1024**3
            record["RSS_Increase_GiB"] = (
                max(0, peak_rss - initial_rss) / 1024**3
            )

        stage9g_records.append(record)

        pd.DataFrame(stage9g_records).to_csv(
            STAGE9_DIR / "stage9g_fold1_confirmation.csv",
            index=False
        )

        print(
            f"MSE={record['MSE_C2']:.4f}, "
            f"Fit={record['Fit_Seconds']:.3f}s, "
            f"Status={record['Status']}"
        )

        del model
        gc.collect()


In [ ]:
if RUN_STAGE9G:

    stage9g_results = pd.DataFrame(stage9g_records)
if RUN_STAGE9G:

    display(stage9g_results.round(4))
else:
    print("Set RUN_STAGE9G = True to run confirmation.")


In [ ]:
# ============================================================
# STAGE 9G — Same-fold comparison
# ============================================================

assert "stage9g_results" in globals()
assert "stage9d_results" in globals()

reference = stage9d_results.loc[
    stage9d_results["Method"].eq("Exact RBF-SVR")
].copy()

assert len(reference) == 1
assert reference.iloc[0]["Status"] == "success"

comparison_columns = [
    "Method",
    "Status",
    "MSE_C2",
    "RMSE_C",
    "Fit_Seconds",
    "Predict_Seconds",
    "Peak_RSS_GiB"
]

stage9g_comparison = pd.concat([
    reference[comparison_columns],
    stage9g_results[comparison_columns]
], ignore_index=True)

stage9g_comparison = stage9g_comparison.sort_values(
    "MSE_C2",
    na_position="last"
)

print("STAGE 9G — ORIGINAL FOLD 1 COMPARISON")
print("=" * 55)


In [ ]:
display(stage9g_comparison.round(4))

stage9g_comparison.to_csv(
    STAGE9_DIR / "stage9g_final_comparison.csv",
    index=False
)


<a id="s-12-8"></a>

### 12.8. Stage 9H: three-fold grouped efficient-SVR comparison

**Research trace:** RQ1, RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.8.


#### 12.8.1. Purpose

Check whether the screened comparisons persist across the three established outer folds.


#### 12.8.2. Mathematical basis and assumptions

Each profile belongs to one validation fold. Its other two folds provide the fitting profiles. For fold $f$,
$$MSE_f=\frac1{N_f}\sum_{i\in f}(\hat T_i-T_i)^2,\qquad J=\frac13\sum_{f=1}^{3}MSE_f.$$
The objective $J$ has units °C² and weights folds equally. It is different from pooling every validation row before calculating MSE. Feature and target standardisers are fitted only on the fitting profiles and predictions are inverse-transformed before scoring. Profile-level temperature maxima define three strata for the fixed shuffled `StratifiedKFold(3, random_state=42)` assignment [ROBERTS17].


#### 12.8.3. Implementation and parameter choices

Retain original inner candidate selection and outer refits for every fold, original matrices, warnings and timers. The long outer-fold loop is deliberately intact: splitting its sequential nested computations could alter timing boundaries or variable scope.


#### 12.8.4. Results and verification evidence

stage9h_results contains nine outer records (three methods × three folds); inner-selection records remain separate. Fold-to-fold accuracy and cost figures use those current-run rows.


#### 12.8.5. Interpretation and limitations

Nested selection reduces direct reuse of an outer score, but all results still belong to the development investigation. Differences across only three folds should be interpreted with restraint.


In [ ]:
# ============================================================
# STAGE 9H — Nested grouped validation
# ============================================================

import os
import gc
import time
import warnings
import threading
import psutil
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR, LinearSVR
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9H = True  # Set True to execute

C_REF = STAGE9_SVR_HP["C"]
EPS_REF = STAGE9_SVR_HP["epsilon"]

C_GRID_9H = [
    C_REF / 100,
    C_REF / 10,
    C_REF
]


In [ ]:
def build_linear_9h(C):
    """Construct the original stage-specific direct LinearSVR candidate.
    
    Parameters
    ----------
    C : float
        Regularisation coefficient in standardised-target epsilon-insensitive loss.
    
    Returns
    -------
    sklearn.svm.LinearSVR
        Unfitted estimator with fresh inherited epsilon and original loss/stopping settings.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    return LinearSVR(
        C=float(C),
        epsilon=EPS_REF,
        loss="epsilon_insensitive",
        dual=True,
        tol=1e-4,
        max_iter=10000,
        random_state=STAGE9_SEED
    )


In [ ]:
def scale_partition(X_fit, y_fit, X_check):
    """Fit original scalers on fitting rows and transform the check rows.
    
    Parameters
    ----------
    X_fit : numpy.ndarray
        Fitting predictor rows; scaling convention is specified by this stage.
    y_fit : numpy.ndarray
        Fitting target; °C before scaling, dimensionless in the outer evaluator.
    X_check : numpy.ndarray
        Check predictor rows; no scaler is fitted on these observations.
    
    Returns
    -------
    tuple
        Scaled fitting/check inputs, scaled fitting target and fitted target scaler.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    xs = StandardScaler()
    ys = StandardScaler()

    X_fit_s = xs.fit_transform(X_fit)
    X_check_s = xs.transform(X_check)
    y_fit_s = ys.fit_transform(
        y_fit.reshape(-1, 1)
    ).ravel()

    return X_fit_s, X_check_s, y_fit_s, ys


In [ ]:
def monitor_memory_9h(stop, values):
    """Sample whole-process resident memory until the original stop event.
    
    Parameters
    ----------
    stop : threading.Event
        Stop signal used by the original timing block.
    values : list[int]
        Mutable process-RSS samples in bytes.
    
    Returns
    -------
    None
        Appends RSS byte values to the supplied list; does not isolate model allocations.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    process = psutil.Process(os.getpid())
    while not stop.wait(0.05):
        try:
            values.append(process.memory_info().rss)
        except psutil.Error:
            break


**Atomic experiment block.** The following original timed or grouped-fold loop is kept together to preserve its execution scope and timer boundaries. Its inputs, mathematical objective and generated outputs are explained in this section.


In [ ]:
def evaluate_outer_9h(
    model, X_fit, y_fit, X_check, y_check, y_scaler
):
    """Fit and measure one original Stage 9H outer-validation estimator.
    
    Parameters
    ----------
    model : sklearn estimator
        Unfitted original estimator; the evaluator fits it on the supplied fitting rows.
    X_fit : numpy.ndarray
        Fitting predictor rows; scaling convention is specified by this stage.
    y_fit : numpy.ndarray
        Fitting target; °C before scaling, dimensionless in the outer evaluator.
    X_check : numpy.ndarray
        Check predictor rows; no scaler is fitted on these observations.
    y_check : numpy.ndarray
        Measured check temperatures in °C.
    y_scaler : sklearn.preprocessing.StandardScaler
        Target scaler fitted only on the fitting partition.
    
    Returns
    -------
    dict
        Errors in °C/°C², validation rows, seconds, iteration/warning and RSS diagnostics.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    gc.collect()

    process = psutil.Process(os.getpid())
    initial_rss = process.memory_info().rss
    memory_values = [initial_rss]
    stop = threading.Event()

    monitor = threading.Thread(
        target=monitor_memory_9h,
        args=(stop, memory_values),
        daemon=True
    )
    monitor.start()

    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            start = time.perf_counter()
            model.fit(X_fit, y_fit)
            fit_seconds = time.perf_counter() - start

        warning_count = sum(
            issubclass(w.category, ConvergenceWarning)
            for w in caught
        )

        start = time.perf_counter()

        prediction_scaled = model.predict(X_check)

        prediction_celsius = (
            y_scaler.inverse_transform(
                prediction_scaled.reshape(-1, 1)
            ).ravel()
        )

        predict_seconds = time.perf_counter() - start

        assert np.isfinite(prediction_celsius).all()

        squared_errors = (
            y_check - prediction_celsius
        ) ** 2

    finally:
        stop.set()
        monitor.join(timeout=2)
        memory_values.append(process.memory_info().rss)

    mse = float(np.mean(squared_errors))

    return {
        "MSE_C2": mse,
        "RMSE_C": float(np.sqrt(mse)),
        "SSE_C2": float(np.sum(squared_errors)),
        "Validation_Rows": len(y_check),
        "Fit_Seconds": fit_seconds,
        "Predict_Seconds": predict_seconds,
        "Peak_RSS_GiB": max(memory_values) / 1024**3,
        "Convergence_Warnings": warning_count,
        "Status": (
            "convergence_warning"
            if warning_count else "success"
        )
    }


In [ ]:
if RUN_STAGE9H:

    stage9h_inner_records = []
if RUN_STAGE9H:
    stage9h_outer_records = []


**Atomic experiment block.** The following original timed or grouped-fold loop is kept together to preserve its execution scope and timer boundaries. Its inputs, mathematical objective and generated outputs are explained in this section.


In [ ]:
if RUN_STAGE9H:

    for outer_fold in (1, 2, 3):

        print(f"\n{'=' * 55}")
        print(f"OUTER FOLD {outer_fold}")

        outer_fit_df = stage9_data.loc[
            stage9_data.cv_fold.ne(outer_fold)
        ]
        outer_val_df = stage9_data.loc[
            stage9_data.cv_fold.eq(outer_fold)
        ]

        assert set(outer_fit_df.profile_id).isdisjoint(
            set(outer_val_df.profile_id)
        )
        assert set(outer_fit_df.profile_id).isdisjoint(
            test_ids
        )
        assert set(outer_val_df.profile_id).isdisjoint(
            test_ids
        )

        X_fit_raw = outer_fit_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        y_fit_raw = outer_fit_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        X_val_raw = outer_val_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        y_val_raw = outer_val_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        groups = outer_fit_df.profile_id.to_numpy()

        # -----------------------------------------------
        # Inner grouped CV for LinearSVR regularisation
        # -----------------------------------------------

        inner_cv = GroupKFold(n_splits=2)
        inner_scores = []

        for C in C_GRID_9H:

            C_errors = []
            C_warning_counts = []

            for inner_fold, (idx_fit, idx_check) in enumerate(
                inner_cv.split(X_fit_raw, y_fit_raw, groups),
                start=1
            ):

                assert set(groups[idx_fit]).isdisjoint(
                    set(groups[idx_check])
                )

                Xi_fit, Xi_check, yi_fit, yi_scaler = (
                    scale_partition(
                        X_fit_raw[idx_fit],
                        y_fit_raw[idx_fit],
                        X_fit_raw[idx_check]
                    )
                )

                inner_model = build_linear_9h(C)

                with warnings.catch_warnings(
                    record=True
                ) as caught:
                    warnings.simplefilter("always")

                    start = time.perf_counter()
                    inner_model.fit(Xi_fit, yi_fit)
                    inner_fit_seconds = (
                        time.perf_counter() - start
                    )

                count = sum(
                    issubclass(w.category, ConvergenceWarning)
                    for w in caught
                )

                predicted = (
                    yi_scaler.inverse_transform(
                        inner_model.predict(
                            Xi_check
                        ).reshape(-1, 1)
                    ).ravel()
                )

                inner_mse = float(np.mean(
                    (
                        y_fit_raw[idx_check] - predicted
                    ) ** 2
                ))

                stage9h_inner_records.append({
                    "Outer_Fold": outer_fold,
                    "Inner_Fold": inner_fold,
                    "C": C,
                    "MSE_C2": inner_mse,
                    "Fit_Seconds": inner_fit_seconds,
                    "Convergence_Warnings": count
                })

                C_errors.append(inner_mse)
                C_warning_counts.append(count)

                del inner_model

            inner_scores.append({
                "C": C,
                "Mean_MSE": np.mean(C_errors),
                "Warnings": sum(C_warning_counts)
            })

        # Select only converged configurations
        eligible = [
            score for score in inner_scores
            if score["Warnings"] == 0
        ]

        if not eligible:
            raise RuntimeError(
                "No converged LinearSVR candidate."
            )

        best = min(
            eligible,
            key=lambda row: row["Mean_MSE"]
        )

        selected_C = float(best["C"])

        print("Selected LinearSVR C:", selected_C)
        print("Inner CV MSE:", round(best["Mean_MSE"], 4))

        # -----------------------------------------------
        # Refit common outer-fold preprocessing
        # -----------------------------------------------

        Xo_fit, Xo_val, yo_fit, yo_scaler = (
            scale_partition(
                X_fit_raw,
                y_fit_raw,
                X_val_raw
            )
        )

        outer_gamma = 1.0 / (
            Xo_fit.shape[1] * Xo_fit.var()
        )

        assert Xo_fit.shape[1] == 108

        models = {
            "Exact RBF-SVR": SVR(
                kernel="rbf",
                C=C_REF,
                epsilon=EPS_REF,
                gamma=outer_gamma,
                cache_size=512
            ),
            "Direct LinearSVR": build_linear_9h(
                selected_C
            )
        }

        # -----------------------------------------------
        # Common outer-fold evaluation
        # -----------------------------------------------

        for method, model in models.items():

            result = evaluate_outer_9h(
                model,
                Xo_fit,
                yo_fit,
                Xo_val,
                y_val_raw,
                yo_scaler
            )

            result.update({
                "Outer_Fold": outer_fold,
                "Method": method,
                "C": (
                    C_REF if method == "Exact RBF-SVR"
                    else selected_C
                ),
                "Training_Rows": len(Xo_fit),
                "Training_Profiles":
                    outer_fit_df.profile_id.nunique(),
                "Validation_Profiles":
                    outer_val_df.profile_id.nunique()
            })

            stage9h_outer_records.append(result)

            print(
                f"{method}: "
                f"MSE={result['MSE_C2']:.3f}, "
                f"fit={result['Fit_Seconds']:.2f}s, "
                f"status={result['Status']}"
            )

            del model

        # Save after every completed outer fold
        pd.DataFrame(
            stage9h_inner_records
        ).to_csv(
            STAGE9_DIR / "stage9h_inner_cv.csv",
            index=False
        )

        pd.DataFrame(
            stage9h_outer_records
        ).to_csv(
            STAGE9_DIR / "stage9h_outer_cv.csv",
            index=False
        )


In [ ]:
if RUN_STAGE9H:

    stage9h_results = pd.DataFrame(
        stage9h_outer_records
    )
if RUN_STAGE9H:

    print("\nSTAGE 9H — OUTER CV RESULTS")
if RUN_STAGE9H:
    display(stage9h_results.round(4))
else:
    print("Set RUN_STAGE9H = True to execute.")


In [ ]:
# ============================================================
# STAGE 9H — Three-fold performance summary
# ============================================================

assert "stage9h_results" in globals()

# Do not silently include failed or nonconverged fits.
assert (
    stage9h_results["Status"].eq("success")
).all(), "Investigate convergence before summarising."

assert (
    stage9h_results.groupby("Method")
    .Outer_Fold.nunique().eq(3)
).all()

summary = (
    stage9h_results.groupby("Method")
    .agg(
        Mean_MSE_C2=("MSE_C2", "mean"),
        Std_MSE_C2=("MSE_C2", "std"),
        Mean_RMSE_C=("RMSE_C", "mean"),
        Total_Fit_S=("Fit_Seconds", "sum"),
        Total_Predict_S=("Predict_Seconds", "sum"),
        Max_Observed_RSS_GiB=("Peak_RSS_GiB", "max"),
        Total_SSE_C2=("SSE_C2", "sum"),
        Total_Validation_Rows=("Validation_Rows", "sum")
    )
)

# Pooled out-of-fold error, weighted by observations
summary["Pooled_MSE_C2"] = (
    summary["Total_SSE_C2"]
    / summary["Total_Validation_Rows"]
)


In [ ]:
summary["Pooled_RMSE_C"] = np.sqrt(
    summary["Pooled_MSE_C2"]
)

print("STAGE 9H — THREE-FOLD SUMMARY")
display(
    summary.sort_values("Mean_MSE_C2").round(4)
)

summary.to_csv(
    STAGE9_DIR / "stage9h_three_fold_summary.csv"
)

print("\nSelected LinearSVR penalties by fold:")
display(
    stage9h_results.loc[
        stage9h_results.Method.eq("Direct LinearSVR"),
        ["Outer_Fold", "C", "MSE_C2", "Fit_Seconds"]
    ].round(5)
)


In [ ]:
from pmsm_project.plotting import set_figure_context
set_figure_context(section='12.8', source_notebook='fresh_reconstruction.ipynb', source_cell=147)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

_pivot = stage9h_results.pivot(index="Outer_Fold",columns="Method",values="MSE_C2")
_candidates = [k for k in ["Exact RBF-SVR","Direct LinearSVR"] if k in _pivot.columns]
assert len(_candidates)==2, f"Unexpected Stage 9H labels: {list(_pivot.columns)}"
_x = np.arange(len(_pivot))
_width=.36
fig,ax=plt.subplots(figsize=(8.7,4.1))
for j,label in enumerate(_candidates):
    ax.bar(_x+(j-.5)*_width,_pivot[label],width=_width,label=label)
ax.set(xticks=_x,xticklabels=[f"Fold {f}" for f in _pivot.index],
       ylabel="Development validation MSE (°C²)",
       title="Three grouped folds: nonlinear kernel accuracy versus linear efficiency")
ax.legend(frameon=False)
ax.grid(axis="y",alpha=.2)
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR)/"figures"/"chapter9_foldwise_svr_mse.png",dpi=180)
plt.show()


<a id="s-12-9"></a>

### 12.9. Stage 9I: direct LinearSVR C grid

**Research trace:** RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.9.


#### 12.9.1. Purpose

Retain the original numerical sensitivity selection of direct LinearSVR regularisation after Bayesian RBF selection.


#### 12.9.2. Mathematical basis and assumptions

The five original multiplicative C factors are $10^{-2},10^{-1.5},10^{-1},10^{-0.5},1$ times the freshly selected RBF C. The same inherited epsilon and spans are used. Each C is assessed on three profile-grouped folds, giving 15 fits.


#### 12.9.3. Implementation and parameter choices

Retain all original loss, dual, tolerance and iteration settings. This finite grid is a historical sensitivity experiment following the GP search; it is not a replacement for that search.


#### 12.9.4. Results and verification evidence

stage9i_trials has 15 records. Grouped summaries and convergence diagnostics identify the reference candidate used by the subsequent sample-size investigation.


#### 12.9.5. Interpretation and limitations

Because these outer fold results also select C, they are development selection evidence and not unbiased final performance estimates.


In [ ]:
# ============================================================
# STAGE 9I — LinearSVR hyperparameter selection
# ============================================================

import gc
import json
import os
import time
import warnings
import threading

import numpy as np
import pandas as pd
import psutil

from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9I = True  # Change to True to execute

C_REF = STAGE9_SVR_HP["C"]
EPS_REF = STAGE9_SVR_HP["epsilon"]

# Logarithmically spaced C values.
C_GRID_9I = C_REF * np.array([
    0.01,
    0.0316227766,
    0.1,
    0.3162277660,
    1.0
])


In [ ]:
def make_stage9i_model(C):
    """Construct the original stage-specific direct LinearSVR candidate.
    
    Parameters
    ----------
    C : float
        Regularisation coefficient in standardised-target epsilon-insensitive loss.
    
    Returns
    -------
    sklearn.svm.LinearSVR
        Unfitted estimator with fresh inherited epsilon and original loss/stopping settings.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    return LinearSVR(
        C=float(C),
        epsilon=EPS_REF,
        loss="epsilon_insensitive",
        dual=True,
        tol=1e-4,
        max_iter=10000,
        random_state=STAGE9_SEED
    )


In [ ]:
# Sample process RSS during model fitting and prediction.
def stage9i_monitor(stop_event, samples):
    """Sample whole-process resident memory until the original stop event.
    
    Parameters
    ----------
    stop_event : threading.Event
        Stop signal used by the original timing block.
    samples : list[int]
        Mutable process-RSS samples in bytes.
    
    Returns
    -------
    None
        Appends RSS byte values to the supplied list; does not isolate model allocations.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    process = psutil.Process(os.getpid())

    while not stop_event.wait(0.05):
        try:
            samples.append(process.memory_info().rss)
        except psutil.Error:
            break

if RUN_STAGE9I:

    assert len(stage9_data) == 55 * 640
if RUN_STAGE9I:
    assert stage9_data.profile_id.nunique() == 55
if RUN_STAGE9I:

    assert set(stage9_data.profile_id).isdisjoint(test_ids)


In [ ]:
if RUN_STAGE9I:
    assert set(stage9_data.cv_fold.unique()) == {1, 2, 3}
if RUN_STAGE9I:

    stage9i_records = []


**Atomic experiment block.** The following original timed or grouped-fold loop is kept together to preserve its execution scope and timer boundaries. Its inputs, mathematical objective and generated outputs are explained in this section.


In [ ]:
if RUN_STAGE9I:

    for fold in (1, 2, 3):

        print("\n" + "=" * 60)
        print(f"OUTER DEVELOPMENT FOLD {fold}")

        # ------------------------------------------------
        # 1. Select original grouped development fold
        # ------------------------------------------------

        train_df = stage9_data.loc[
            stage9_data.cv_fold.ne(fold)
        ]

        val_df = stage9_data.loc[
            stage9_data.cv_fold.eq(fold)
        ]

        assert set(train_df.profile_id).isdisjoint(
            set(val_df.profile_id)
        )

        assert set(train_df.profile_id).isdisjoint(test_ids)
        assert set(val_df.profile_id).isdisjoint(test_ids)

        X_train_raw = train_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        X_val_raw = val_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        y_train_raw = train_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        y_val_raw = val_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        # ------------------------------------------------
        # 2. Fit scalers on training observations ONLY
        # ------------------------------------------------

        x_scaler = StandardScaler()
        y_scaler = StandardScaler()

        X_train = x_scaler.fit_transform(X_train_raw)
        X_val = x_scaler.transform(X_val_raw)

        y_train = y_scaler.fit_transform(
            y_train_raw.reshape(-1, 1)
        ).ravel()

        assert X_train.shape[1] == 108
        assert X_val.shape[1] == 108

        print("Training observations:", len(X_train))
        print("Validation observations:", len(X_val))

        # ------------------------------------------------
        # 3. Compare each regularisation candidate
        # ------------------------------------------------

        for C in C_GRID_9I:

            gc.collect()

            model = make_stage9i_model(C)

            process = psutil.Process(os.getpid())
            rss_initial = process.memory_info().rss

            memory_samples = [rss_initial]
            stop_event = threading.Event()

            monitor = threading.Thread(
                target=stage9i_monitor,
                args=(stop_event, memory_samples),
                daemon=True
            )

            monitor.start()

            try:
                with warnings.catch_warnings(
                    record=True
                ) as caught:

                    warnings.simplefilter(
                        "always",
                        ConvergenceWarning
                    )

                    t0 = time.perf_counter()

                    model.fit(X_train, y_train)

                    fit_seconds = (
                        time.perf_counter() - t0
                    )

                warning_count = sum(
                    issubclass(
                        w.category,
                        ConvergenceWarning
                    )
                    for w in caught
                )

                # ----------------------------------------
                # 4. Validation prediction
                # ----------------------------------------

                t1 = time.perf_counter()

                predictions_scaled = model.predict(X_val)

                predictions_celsius = (
                    y_scaler.inverse_transform(
                        predictions_scaled.reshape(-1, 1)
                    ).ravel()
                )

                predict_seconds = (
                    time.perf_counter() - t1
                )

                assert np.isfinite(predictions_celsius).all()

                squared_errors = (
                    y_val_raw - predictions_celsius
                ) ** 2

                mse = float(np.mean(squared_errors))

                record = {
                    "Fold": fold,
                    "C": float(C),
                    "Training_Rows": len(X_train),
                    "Validation_Rows": len(X_val),
                    "MSE_C2": mse,
                    "RMSE_C": float(np.sqrt(mse)),
                    "SSE_C2": float(np.sum(squared_errors)),
                    "Fit_Seconds": fit_seconds,
                    "Predict_Seconds": predict_seconds,
                    "Iterations": int(
                        np.max(np.atleast_1d(model.n_iter_))
                    ),
                    "Convergence_Warnings": warning_count,
                    "Status": (
                        "convergence_warning"
                        if warning_count else "success"
                    )
                }

            except Exception as exc:
                # Unexpected errors must not be treated
                # as legitimate validation results.
                print(f"FAILED: Fold={fold}, C={C}")
                raise

            finally:
                stop_event.set()
                monitor.join(timeout=2)

                memory_samples.append(
                    process.memory_info().rss
                )

            peak_rss = max(memory_samples)

            record["Peak_Process_RSS_GiB"] = (
                peak_rss / 1024**3
            )

            record["RSS_Increase_GiB"] = (
                max(0, peak_rss-rss_initial) / 1024**3
            )

            stage9i_records.append(record)

            print(
                f"C={C:.6f} | "
                f"MSE={record['MSE_C2']:.4f} | "
                f"Fit={fit_seconds:.3f}s | "
                f"Warnings={warning_count}"
            )

            # Save each successful trial immediately.
            pd.DataFrame(stage9i_records).to_csv(
                STAGE9_DIR / "stage9i_linear_C_trials.csv",
                index=False
            )

            del model
            gc.collect()


In [ ]:
if RUN_STAGE9I:

    stage9i_trials = pd.DataFrame(stage9i_records)
if RUN_STAGE9I:

    assert len(stage9i_trials) == 15
if RUN_STAGE9I:

    print("\nSTAGE 9I — ALL TRIAL RESULTS")
if RUN_STAGE9I:
    display(stage9i_trials.round(5))
else:
    print("Set RUN_STAGE9I = True to run the C search.")


In [ ]:
# ============================================================
# STAGE 9I — Hyperparameter selection and summary
# ============================================================

assert "stage9i_trials" in globals()
assert len(stage9i_trials) == 15

# ------------------------------------------------------------
# 1. Aggregate across three development folds
# ------------------------------------------------------------

stage9i_summary = (
    stage9i_trials
    .groupby("C", as_index=False)
    .agg(
        Mean_MSE_C2=("MSE_C2", "mean"),
        Std_MSE_C2=("MSE_C2", "std"),
        Mean_RMSE_C=("RMSE_C", "mean"),
        Total_Fit_Seconds=("Fit_Seconds", "sum"),
        Total_Predict_Seconds=("Predict_Seconds", "sum"),
        Total_SSE_C2=("SSE_C2", "sum"),
        Total_Validation_Rows=("Validation_Rows", "sum"),
        Convergence_Warnings=(
            "Convergence_Warnings", "sum"
        ),
        Completed_Folds=("Fold", "nunique"),
        Max_Process_RSS_GiB=(
            "Peak_Process_RSS_GiB", "max"
        )
    )
)

stage9i_summary["Pooled_MSE_C2"] = (
    stage9i_summary["Total_SSE_C2"]
    / stage9i_summary["Total_Validation_Rows"]
)


In [ ]:
stage9i_summary["Pooled_RMSE_C"] = np.sqrt(
    stage9i_summary["Pooled_MSE_C2"]
)

# ------------------------------------------------------------
# 2. Select only completed, converged configurations
# ------------------------------------------------------------

eligible = stage9i_summary.loc[
    stage9i_summary.Completed_Folds.eq(3)
    & stage9i_summary.Convergence_Warnings.eq(0)
].copy()

assert len(eligible) > 0, (
    "No fully converged configuration. "
    "Investigate solver convergence before selection."
)

eligible = eligible.sort_values(
    ["Mean_MSE_C2", "Total_Fit_Seconds", "C"]
)

STAGE9I_SELECTED_C = float(eligible.iloc[0]["C"])

# ------------------------------------------------------------
# 3. Report results
# ------------------------------------------------------------

print("STAGE 9I — LINEARSVR HYPERPARAMETER SUMMARY")
print("=" * 60)

display(
    stage9i_summary.sort_values("Mean_MSE_C2").round(5)
)

print("\nSELECTED REGULARISATION")
print("C =", STAGE9I_SELECTED_C)


In [ ]:
print(
    "Mean development CV MSE =",
    round(float(eligible.iloc[0]["Mean_MSE_C2"]), 4),
    "°C²"
)

print(
    "Pooled development CV MSE =",
    round(float(eligible.iloc[0]["Pooled_MSE_C2"]), 4),
    "°C²"
)

print("\nPer-fold performance of selected C:")

display(
    stage9i_trials.loc[
        stage9i_trials.C.eq(STAGE9I_SELECTED_C),
        [
            "Fold",
            "C",
            "MSE_C2",
            "RMSE_C",
            "Fit_Seconds",
            "Convergence_Warnings"
        ]
    ].round(5)
)

# ------------------------------------------------------------
# 4. Save the selection and experimental record
# ------------------------------------------------------------

stage9i_summary.to_csv(
    STAGE9_DIR / "stage9i_linear_C_summary.csv",
    index=False
)


In [ ]:
selection_manifest = {
    "stage": "9I",
    "method": "LinearSVR",
    "selected_C": STAGE9I_SELECTED_C,
    "epsilon": float(EPS_REF),
    "loss": "epsilon_insensitive",
    "dual": True,
    "max_iter": 10000,
    "tol": 1e-4,
    "temporal_spans": list(STAGE9_SPANS),
    "features": 108,
    "observations_per_profile": 640,
    "cv_folds": 3,
    "selection_metric": "mean_fold_MSE_C2",
    "test_profiles_accessed": False
}

with open(
    STAGE9_DIR / "stage9i_selection_manifest.json",
    "w"
) as f:
    json.dump(selection_manifest, f, indent=2)

print("\nPASS: Stage 9I completed.")
print("Selected C saved with reproducibility metadata.")


In [ ]:
from pmsm_project.plotting import set_figure_context
set_figure_context(section='12.9', source_notebook='fresh_reconstruction.ipynb', source_cell=153)


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

_cplot=stage9i_summary.sort_values("C")
fig,ax=plt.subplots(figsize=(8.7,4.6))
ax.plot(_cplot["Total_Fit_Seconds"],_cplot["Mean_MSE_C2"],"o-",color="#4D7994")
for r in _cplot.itertuples():
    ax.annotate(f"C={r.C:.4g}",(r.Total_Fit_Seconds,r.Mean_MSE_C2),
                xytext=(5,7),textcoords="offset points",fontsize=8)
_selected=_cplot.iloc[(abs(_cplot.C-STAGE9I_SELECTED_C)).argmin()]
ax.scatter([_selected.Total_Fit_Seconds],[_selected.Mean_MSE_C2],s=115,
           facecolors="none",edgecolors="#C86D3C",linewidths=2,label="Selected by mean fold MSE")
ax.set(xlabel="Total three-fold fitting time (s)",ylabel="Mean grouped-CV MSE (°C²)",
       title="LinearSVR: regularisation and compute cost")
ax.grid(alpha=.2)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR)/"figures"/"chapter9_regularisation_tradeoff.png",dpi=180)
plt.show()


<a id="s-12-10"></a>

### 12.10. Stage 9J: larger training samples

**Research trace:** RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.10.


#### 12.10.1. Purpose

Assess LinearSVR when fitting grows from 640 to 1,280 observations per profile while validation remains at 640.


#### 12.10.2. Mathematical basis and assumptions

The sum-loss regularisation balance depends on C and the number of fitting rows. Holding C fixed while increasing N changes the relative loss/penalty weighting. This stage records that sensitivity before applying sample-count reasoning.


#### 12.10.3. Implementation and parameter choices

Retain the original sample selection, all grouped folds, two investigated C values, validation rows and convergence/time/RSS measurements.


#### 12.10.4. Results and verification evidence

stage9j_results contains six records. Its comparison and summaries link the original smaller and larger sample experiments without introducing test-profile feedback.


#### 12.10.5. Interpretation and limitations

More rows from the same recordings improve within-domain coverage but do not add independent motor sessions. Any sample-count heuristic must be distinguished from a new full-data optimum.


In [ ]:
# ============================================================
# STAGE 9J.1 — Expand training sampling to 1280/profile
# ============================================================

import time
import numpy as np
import pandas as pd

STAGE9J_BUDGET = 1280

t0 = time.perf_counter()
stage9j_parts = []

fold_lookup = (
    profile_cv.set_index("profile_id")["cv_fold"]
    .astype(int).to_dict()
)


In [ ]:
for pid, profile in development_df.groupby(
    "profile_id", sort=False
):
    order = nested_sample_order(
        len(profile), STAGE9J_BUDGET
    )

    assert len(order) == STAGE9J_BUDGET
    assert len(set(order)) == STAGE9J_BUDGET

    positions = np.sort(np.asarray(order, dtype=int))

    rank_lookup = {
        int(pos): rank
        for rank, pos in enumerate(order)
    }

    base = make_paper_inputs(profile)

    sample = vectorised_temporal_sample(
        base, positions, STAGE9_SPANS
    )

    sample = sample.loc[
        :, stage9_feature_cols
    ].copy()

    sample["pm_target"] = (
        profile["pm"].iloc[positions].to_numpy()
    )
    sample["profile_id"] = int(pid)
    sample["source_row"] = sample.index.to_numpy()
    sample["nest_rank"] = [
        rank_lookup[int(p)] for p in positions
    ]
    sample["cv_fold"] = fold_lookup[int(pid)]

    stage9j_parts.append(sample)


In [ ]:
stage9j_data = pd.concat(
    stage9j_parts, ignore_index=True
)

assert len(stage9j_data) == 55 * 1280
assert stage9j_data.profile_id.nunique() == 55
assert set(stage9j_data.profile_id).isdisjoint(test_ids)

assert np.isfinite(
    stage9j_data[
        stage9_feature_cols + ["pm_target"]
    ].to_numpy(dtype=np.float64)
).all()

# Verify that the original Stage 9B training/validation
# sample is exactly preserved by the larger nested sample.
cols = [
    "profile_id", "source_row",
    "nest_rank", "pm_target"
]

old = (
    stage9_data
    .sort_values(["profile_id", "nest_rank"])
    .reset_index(drop=True)
)

new = (
    stage9j_data.loc[stage9j_data.nest_rank.lt(640)]
    .sort_values(["profile_id", "nest_rank"])
    .reset_index(drop=True)
)


In [ ]:
pd.testing.assert_frame_equal(
    old[cols],
    new[cols],
    check_exact=True
)

np.testing.assert_allclose(
    old[stage9_feature_cols].to_numpy(),
    new[stage9_feature_cols].to_numpy(),
    rtol=0,
    atol=1e-10
)

print("PASS: Original 640 observations/profile preserved.")
print("Expanded observations:", len(stage9j_data))
print("Features:", len(stage9_feature_cols))
print("Preparation time:", round(time.perf_counter()-t0, 2), "s")
print("Held-out test profiles untouched.")


In [ ]:
# ============================================================
# STAGE 9J.2 — LinearSVR scaling comparison
# ============================================================

import gc
import os
import time
import threading
import warnings
import psutil
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9J = True  # Change to True to execute

# Reuse the EXACT two C values from Stage 9I.
C_GRID_9J = [
    float(C_GRID_9I[3]),
    float(C_GRID_9I[4])
]


In [ ]:
def stage9j_memory_monitor(stop, samples):
    """Sample whole-process resident memory until the original stop event.
    
    Parameters
    ----------
    stop : threading.Event
        Stop signal used by the original timing block.
    samples : list[int]
        Mutable process-RSS samples in bytes.
    
    Returns
    -------
    None
        Appends RSS byte values to the supplied list; does not isolate model allocations.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    process = psutil.Process(os.getpid())

    while not stop.wait(0.05):
        try:
            samples.append(process.memory_info().rss)
        except psutil.Error:
            break

if RUN_STAGE9J:

    stage9j_records = []


**Atomic experiment block.** The following original timed or grouped-fold loop is kept together to preserve its execution scope and timer boundaries. Its inputs, mathematical objective and generated outputs are explained in this section.


In [ ]:
if RUN_STAGE9J:

    for fold in (1, 2, 3):

        train_df = stage9j_data.loc[
            stage9j_data.cv_fold.ne(fold)
        ]

        val_df = stage9j_data.loc[
            stage9j_data.cv_fold.eq(fold)
            & stage9j_data.nest_rank.lt(640)
        ]

        assert set(train_df.profile_id).isdisjoint(
            set(val_df.profile_id)
        )

        X_train_raw = train_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        X_val_raw = val_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        y_train_raw = train_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        y_val_raw = val_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        x_scaler = StandardScaler()
        y_scaler = StandardScaler()

        X_train = x_scaler.fit_transform(X_train_raw)
        X_val = x_scaler.transform(X_val_raw)

        y_train = y_scaler.fit_transform(
            y_train_raw.reshape(-1, 1)
        ).ravel()

        print(f"\nFOLD {fold}")
        print("Training rows:", len(X_train))
        print("Validation rows:", len(X_val))

        for C in C_GRID_9J:

            gc.collect()

            model = LinearSVR(
                C=float(C),
                epsilon=STAGE9_SVR_HP["epsilon"],
                loss="epsilon_insensitive",
                dual=True,
                tol=1e-4,
                max_iter=10000,
                random_state=STAGE9_SEED
            )

            process = psutil.Process(os.getpid())
            baseline = process.memory_info().rss

            memory_samples = [baseline]
            stop = threading.Event()

            monitor = threading.Thread(
                target=stage9j_memory_monitor,
                args=(stop, memory_samples),
                daemon=True
            )
            monitor.start()

            try:
                with warnings.catch_warnings(
                    record=True
                ) as caught:
                    warnings.simplefilter(
                        "always", ConvergenceWarning
                    )

                    fit_start = time.perf_counter()
                    model.fit(X_train, y_train)
                    fit_seconds = (
                        time.perf_counter() - fit_start
                    )

                warning_count = sum(
                    issubclass(w.category, ConvergenceWarning)
                    for w in caught
                )

                predict_start = time.perf_counter()

                prediction = (
                    y_scaler.inverse_transform(
                        model.predict(X_val).reshape(-1, 1)
                    ).ravel()
                )

                predict_seconds = (
                    time.perf_counter() - predict_start
                )

                assert np.isfinite(prediction).all()

                squared_errors = (y_val_raw-prediction)**2
                mse = float(np.mean(squared_errors))

                record = {
                    "Fold": fold,
                    "C": float(C),
                    "Training_Rows": len(X_train),
                    "Validation_Rows": len(X_val),
                    "MSE_C2": mse,
                    "RMSE_C": float(np.sqrt(mse)),
                    "SSE_C2": float(np.sum(squared_errors)),
                    "Fit_Seconds": fit_seconds,
                    "Predict_Seconds": predict_seconds,
                    "Iterations": int(model.n_iter_),
                    "Convergence_Warnings": warning_count,
                    "Status": (
                        "success" if warning_count == 0
                        else "convergence_warning"
                    )
                }

            finally:
                stop.set()
                monitor.join(timeout=2)
                memory_samples.append(
                    process.memory_info().rss
                )

            record["Peak_Process_RSS_GiB"] = (
                max(memory_samples) / 1024**3
            )

            stage9j_records.append(record)

            print(
                f"C={C:.6f}, "
                f"MSE={mse:.4f}, "
                f"Fit={fit_seconds:.2f}s, "
                f"Warnings={warning_count}"
            )

            # Save after every completed fit.
            pd.DataFrame(stage9j_records).to_csv(
                STAGE9_DIR / "stage9j_scalability_trials.csv",
                index=False
            )

            del model
            gc.collect()


In [ ]:
if RUN_STAGE9J:

    stage9j_results = pd.DataFrame(stage9j_records)
if RUN_STAGE9J:

    print("\nSTAGE 9J — EXPANDED TRAINING RESULTS")
if RUN_STAGE9J:
    display(stage9j_results.round(4))
else:
    print("Set RUN_STAGE9J = True to execute.")


In [ ]:
# ============================================================
# STAGE 9J — Accuracy and scalability summary
# ============================================================

assert "stage9j_results" in globals()
assert len(stage9j_results) == 6

stage9j_summary = (
    stage9j_results.groupby("C", as_index=False)
    .agg(
        Mean_MSE_C2=("MSE_C2", "mean"),
        Mean_RMSE_C=("RMSE_C", "mean"),
        Total_Fit_S=("Fit_Seconds", "sum"),
        Total_Predict_S=("Predict_Seconds", "sum"),
        Warnings=("Convergence_Warnings", "sum"),
        Max_RSS_GiB=("Peak_Process_RSS_GiB", "max")
    )
)

baseline = stage9i_summary[
    ["C", "Mean_MSE_C2", "Total_Fit_Seconds"]
].rename(columns={
    "Mean_MSE_C2": "MSE_640",
    "Total_Fit_Seconds": "Fit_640_S"
})

stage9j_summary = stage9j_summary.merge(
    baseline, on="C", validate="one_to_one"
)

stage9j_summary["MSE_Change_C2"] = (
    stage9j_summary.Mean_MSE_C2
    - stage9j_summary.MSE_640
)


In [ ]:
stage9j_summary["Fit_Time_Ratio"] = (
    stage9j_summary.Total_Fit_S
    / stage9j_summary.Fit_640_S
)

print("STAGE 9J — SCALABILITY COMPARISON")
display(stage9j_summary.round(4))

stage9j_summary.to_csv(
    STAGE9_DIR / "stage9j_scalability_summary.csv",
    index=False
)


<a id="s-12-11"></a>

### 12.11. Stage 9K: sample-size-aware regularisation

**Research trace:** RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.11.


#### 12.11.1. Purpose

Retain the smaller-C extension at the 1,280/profile fitting budget and combine it with Stage 9J.


#### 12.11.2. Mathematical basis and assumptions

The combined four-C table averages MSE across three folds and sums warnings. The source selects the lowest mean MSE among converged candidates. For extrapolation, $C_{full}=C_{ref}\bar N_{CV}/N_{full}$ approximately preserves C×N in the sum-loss objective.


#### 12.11.3. Implementation and parameter choices

Retain the original additional two C values and six fits. Combine the six Stage 9J records with six Stage 9K records. The exporter recomputes the eligible minimum and actual mean fold fitting-row count.


#### 12.11.4. Results and verification evidence

stage9k_results has six rows and stage9k_summary four candidates. The accuracy/cost plot reports current-run grouped means and convergence.


#### 12.11.5. Interpretation and limitations

Preserving C×N is a heuristic under a changed sample distribution. It is neither a separate full-development tuning search nor a guarantee of optimal final regularisation.


In [ ]:
# ============================================================
# STAGE 9K — Sample-size-aware C sensitivity
# ============================================================

import gc
import os
import time
import warnings
import threading

import numpy as np
import pandas as pd
import psutil

from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9K = True  # Change to True when ready

# Reference: 640 -> 1280 training observations/profile
SAMPLE_RATIO_9K = 640 / 1280

C_GRID_9K = [
    float(C_GRID_9I[3] * SAMPLE_RATIO_9K),
    float(C_GRID_9I[4] * SAMPLE_RATIO_9K)
]

print("New C candidates:", C_GRID_9K)

if RUN_STAGE9K:

    assert len(stage9j_data) == 55 * 1280
if RUN_STAGE9K:
    assert set(stage9j_data.profile_id).isdisjoint(test_ids)


In [ ]:
if RUN_STAGE9K:

    stage9k_records = []


**Atomic experiment block.** The following original timed or grouped-fold loop is kept together to preserve its execution scope and timer boundaries. Its inputs, mathematical objective and generated outputs are explained in this section.


In [ ]:
if RUN_STAGE9K:

    for fold in (1, 2, 3):

        train_df = stage9j_data.loc[
            stage9j_data.cv_fold.ne(fold)
        ]

        val_df = stage9j_data.loc[
            stage9j_data.cv_fold.eq(fold)
            & stage9j_data.nest_rank.lt(640)
        ]

        assert set(train_df.profile_id).isdisjoint(
            set(val_df.profile_id)
        )

        X_train_raw = train_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        X_val_raw = val_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        y_train_raw = train_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        y_val_raw = val_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        # Training-fold-only preprocessing
        xs = StandardScaler()
        ys = StandardScaler()

        X_train = xs.fit_transform(X_train_raw)
        X_val = xs.transform(X_val_raw)

        y_train = ys.fit_transform(
            y_train_raw.reshape(-1, 1)
        ).ravel()

        print(f"\nFOLD {fold}")

        for C in C_GRID_9K:

            gc.collect()

            model = LinearSVR(
                C=C,
                epsilon=STAGE9_SVR_HP["epsilon"],
                loss="epsilon_insensitive",
                dual=True,
                tol=1e-4,
                max_iter=10000,
                random_state=STAGE9_SEED
            )

            process = psutil.Process(os.getpid())
            rss_before = process.memory_info().rss

            memory_samples = [rss_before]
            stop = threading.Event()

            monitor = threading.Thread(
                target=stage9j_memory_monitor,
                args=(stop, memory_samples),
                daemon=True
            )

            monitor.start()

            try:
                with warnings.catch_warnings(
                    record=True
                ) as caught:

                    warnings.simplefilter(
                        "always",
                        ConvergenceWarning
                    )

                    t0 = time.perf_counter()
                    model.fit(X_train, y_train)
                    fit_seconds = time.perf_counter() - t0

                warning_count = sum(
                    issubclass(w.category, ConvergenceWarning)
                    for w in caught
                )

                t1 = time.perf_counter()

                y_pred = ys.inverse_transform(
                    model.predict(X_val).reshape(-1, 1)
                ).ravel()

                predict_seconds = time.perf_counter() - t1

                assert np.isfinite(y_pred).all()

                squared_errors = (y_val_raw - y_pred) ** 2
                mse = float(np.mean(squared_errors))

                record = {
                    "Fold": fold,
                    "C": C,
                    "Training_Rows": len(X_train),
                    "Validation_Rows": len(X_val),
                    "MSE_C2": mse,
                    "RMSE_C": float(np.sqrt(mse)),
                    "Fit_Seconds": fit_seconds,
                    "Predict_Seconds": predict_seconds,
                    "Iterations": int(model.n_iter_),
                    "Convergence_Warnings": warning_count,
                    "Status": (
                        "success" if warning_count == 0
                        else "convergence_warning"
                    )
                }

            finally:
                stop.set()
                monitor.join(timeout=2)
                memory_samples.append(
                    process.memory_info().rss
                )

            record["Peak_Process_RSS_GiB"] = (
                max(memory_samples) / 1024**3
            )

            stage9k_records.append(record)

            print(
                f"C={C:.6f} | "
                f"MSE={mse:.4f} | "
                f"Fit={fit_seconds:.2f}s | "
                f"Warnings={warning_count}"
            )

            pd.DataFrame(stage9k_records).to_csv(
                STAGE9_DIR / "stage9k_C_scaling_trials.csv",
                index=False
            )

            del model
            gc.collect()


In [ ]:
if RUN_STAGE9K:

    stage9k_results = pd.DataFrame(stage9k_records)
if RUN_STAGE9K:

    assert len(stage9k_results) == 6
if RUN_STAGE9K:

    print("\nSTAGE 9K RESULTS")
if RUN_STAGE9K:
    display(stage9k_results.round(5))
else:
    print("Stage 9K disabled. Set RUN_STAGE9K=True.")


In [ ]:
# ============================================================
# STAGE 9K — Combined regularisation comparison
# ============================================================

assert "stage9k_results" in globals()
assert "stage9j_results" in globals()

combined = pd.concat(
    [stage9j_results, stage9k_results],
    ignore_index=True
)

assert len(combined) == 12

assert (
    combined.groupby("C")["Fold"].nunique().eq(3)
).all()

stage9k_summary = (
    combined.groupby("C", as_index=False)
    .agg(
        Mean_MSE_C2=("MSE_C2", "mean"),
        Mean_RMSE_C=("RMSE_C", "mean"),
        Total_Fit_S=("Fit_Seconds", "sum"),
        Total_Predict_S=("Predict_Seconds", "sum"),
        Warnings=("Convergence_Warnings", "sum"),
        Max_RSS_GiB=("Peak_Process_RSS_GiB", "max")
    )
)

print("STAGE 9K — COMBINED RESULTS")
display(
    stage9k_summary
    .sort_values("Mean_MSE_C2")
    .round(5)
)


In [ ]:
stage9k_summary.to_csv(
    STAGE9_DIR / "stage9k_combined_summary.csv",
    index=False
)


In [ ]:
from pmsm_project.plotting import set_figure_context
set_figure_context(section='12.11', source_notebook='fresh_reconstruction.ipynb', source_cell=163)


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

_scaled=stage9k_summary.sort_values("C")
fig,ax=plt.subplots(figsize=(8.7,4.4))
ax.semilogx(_scaled["C"],_scaled["Mean_MSE_C2"],"o-",color="#4A8790")
for r in _scaled.itertuples():
    ax.annotate(f"{r.Total_Fit_S:.1f}s",(r.C,r.Mean_MSE_C2),
                xytext=(4,9),textcoords="offset points",fontsize=9)
ax.set(xlabel="Regularisation parameter C (log scale)",ylabel="Mean grouped-CV MSE (°C²)",
       title="Sample-size-aware C screening (times annotated)")
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR)/"figures"/"chapter9_scaled_regularisation.png",dpi=180)
plt.show()


<a id="s-12-12"></a>

### 12.12. Stage 9L: full-development feature reconstruction

**Research trace:** RQ1, RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.12.


#### 12.12.1. Purpose

Reconstruct all original development features for the final LinearSVR continuation.


#### 12.12.2. Mathematical basis and assumptions

For a span $s$ measured in samples, $\alpha=2/(s+1)$ and $w_{t,j}=(1-\alpha)^{t-j}$ for $j\leq t$. The adjusted mean and biased standard deviation are
$$\mu_t=\frac{\sum_{j=0}^{t}w_{t,j}x_j}{\sum_{j=0}^{t}w_{t,j}},\qquad
\sigma_t=\sqrt{\frac{\sum_{j=0}^{t}w_{t,j}(x_j-\mu_t)^2}{\sum_{j=0}^{t}w_{t,j}}}.$$
Pandas `adjust=True` and `std(bias=True)` implement these moments [PANDAS]. Thus $\mu_0=x_0$ and $\sigma_0=0$. Four spans for each of 12 inputs give $12+2(12)(4)=108$ features. Filters reset at every recording and use full original chronological histories before selecting rows. A span of 200 samples corresponds to 100 s of sample count at 2 Hz; an exponential filter has infinite decaying support rather than a hard 100-s window.


#### 12.12.3. Implementation and parameter choices

Retain the original full-data feature construction and feature-order assertions. The spans are inherited from the current larger RBF search, not the old Stage 9 constants.


#### 12.12.4. Results and verification evidence

X9_full has 1,055,198 rows and 108 columns and y9_full contains only the 55 development profiles. Finiteness, ordering and allocation checks precede fitting.


#### 12.12.5. Interpretation and limitations

This allocation can be memory-intensive on Colab. Do not silently downsample if it fails; document the resource limit and rerun in a sufficient runtime.


In [ ]:
# ============================================================
# STAGE 9L — Full-development feature reconstruction
# ============================================================

import gc
import time
import numpy as np
import pandas as pd

start = time.perf_counter()

N_FULL = len(development_df)
P_FULL = len(stage9_feature_cols)

assert N_FULL == 1_055_198
assert P_FULL == 108

# Allocate one feature matrix instead of concatenating
# a large collection of full-profile DataFrames.
X9_full = np.empty(
    (N_FULL, P_FULL),
    dtype=np.float64
)

y9_full = np.empty(N_FULL, dtype=np.float64)

# Metadata for numerical verification
source_rows_full = np.empty(N_FULL, dtype=np.int64)
profile_ids_full = np.empty(N_FULL, dtype=np.int64)

cursor = 0


In [ ]:
for count, (pid, profile) in enumerate(
    development_df.groupby("profile_id", sort=False),
    start=1
):

    assert profile.index.is_monotonic_increasing
    assert profile.index.is_unique

    n = len(profile)
    positions = np.arange(n, dtype=np.int64)

    # Use the original validated instantaneous predictors.
    original_inputs = make_paper_inputs(profile)

    # Generate temporal features from full chronological
    # history, retaining EVERY observation.
    features = vectorised_temporal_sample(
        original_inputs,
        positions,
        STAGE9_SPANS
    )

    features = features.loc[
        :, stage9_feature_cols
    ]

    assert features.shape == (n, 108)

    block = slice(cursor, cursor + n)

    X9_full[block] = features.to_numpy(
        dtype=np.float64
    )

    y9_full[block] = profile[
        "pm"
    ].to_numpy(dtype=np.float64)

    source_rows_full[block] = (
        profile.index.to_numpy(dtype=np.int64)
    )

    profile_ids_full[block] = int(pid)

    cursor += n

    del original_inputs, features

    if count % 10 == 0 or count == 55:
        print(
            f"Processed {count}/55 profiles; "
            f"{cursor:,} observations"
        )


In [ ]:
assert cursor == N_FULL
assert X9_full.shape == (1_055_198, 108)
assert y9_full.shape == (1_055_198,)

assert np.isfinite(X9_full).all()
assert np.isfinite(y9_full).all()

assert len(np.unique(profile_ids_full)) == 55
assert set(np.unique(profile_ids_full)).isdisjoint(test_ids)
assert len(np.unique(source_rows_full)) == N_FULL

print("\nPASS: Complete development matrix constructed.")

# ------------------------------------------------------------
# Numerical verification against Stage 9J
# ------------------------------------------------------------

# Match each historical sampled observation to its row
# in the complete development feature matrix.
lookup = pd.Index(source_rows_full).get_indexer(
    stage9j_data["source_row"].to_numpy(
        dtype=np.int64
    )
)

assert (lookup >= 0).all()

np.testing.assert_allclose(
    X9_full[lookup],
    stage9j_data[
        stage9_feature_cols
    ].to_numpy(dtype=np.float64),
    rtol=0,
    atol=1e-9
)


In [ ]:
np.testing.assert_allclose(
    y9_full[lookup],
    stage9j_data[
        "pm_target"
    ].to_numpy(dtype=np.float64),
    rtol=0,
    atol=0
)

np.testing.assert_array_equal(
    profile_ids_full[lookup],
    stage9j_data[
        "profile_id"
    ].to_numpy(dtype=np.int64)
)

print("PASS: All Stage 9J sampled features match.")

# ------------------------------------------------------------
# Computational report
# ------------------------------------------------------------

feature_gib = X9_full.nbytes / 1024**3

print("\nSTAGE 9L — FINAL VERIFICATION")
print("=" * 55)

print("Development observations:", f"{N_FULL:,}")
print("Development profiles:", len(np.unique(profile_ids_full)))
print("Predictors:", P_FULL)
print("Feature matrix shape:", X9_full.shape)
print("Target vector shape:", y9_full.shape)

print("\nFloat64 feature memory:",
      round(feature_gib, 3), "GiB")


In [ ]:
print("Construction time:",
      round(time.perf_counter() - start, 2), "s")

print("\nPASS: Stage 9L complete.")
print("No estimator trained.")
print("Final held-out test profiles untouched.")


<a id="s-12-13"></a>

### 12.13. Stage 9M: full-development LinearSVR and persistence

**Research trace:** RQ3, RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 12.13.


#### 12.13.1. Purpose

Retain the original full-development LinearSVR continuation and saved-model verification before publishing the selection handoff.


#### 12.13.2. Mathematical basis and assumptions

Direct LinearSVR uses $f(x)=w^Tx+b$ with $\frac12\|w\|^2+C\sum_i\max(0,|z_i-f(x_i)|-\epsilon)$, subject to LIBLINEAR’s intercept-regularisation convention [FAN08,SKLEARN]. It removes the nonlinear kernel. The final configuration keeps `loss='epsilon_insensitive'`, `dual=True`, `tol=1e-4`, `max_iter=10000` and seed 42. The inherited $\epsilon$ and spans originate in the fresh larger RBF search; Stage 9 selects the reference $C$. The final C is $C_{ref}\bar N_{CV}/1{,}055{,}198$.


#### 12.13.3. Implementation and parameter choices

Retain original scaling, stopping, warning capture, process RSS monitoring, serialization and reload comparison. The regularisation input must come from the current Stage 9J/K summary. Notebook 01 independently refits this selected configuration later.


#### 12.13.4. Results and verification evidence

The model, manifest_9m, warning count, iterations, timings and reload-prediction differences are written. A nonconverged full-development continuation cannot be certified by the final selection-export gate.


#### 12.13.5. Interpretation and limitations

This training-only continuation is not a held-out evaluation. A convergence failure must be reported for the original setting; changing C or iteration counts requires a separately documented methodological decision.


In [ ]:
# ============================================================
# STAGE 9M.1 — Final LinearSVR configuration
# ============================================================

import numpy as np
import pandas as pd

assert len(X9_full) == 1_055_198
assert X9_full.shape[1] == 108
assert len(y9_full) == len(X9_full)

# Select lowest-MSE converged Stage 9K configuration
eligible_9k = stage9k_summary.loc[
    stage9k_summary["Warnings"].eq(0)
].sort_values("Mean_MSE_C2")

assert len(eligible_9k) > 0

C_REFERENCE_9M = float(
    eligible_9k.iloc[0]["C"]
)

# Actual training rows used within Stage 9K folds
N_REFERENCE_9M = float(
    stage9k_results
    .drop_duplicates("Fold")["Training_Rows"]
    .mean()
)

N_FULL_9M = len(X9_full)

# Preserve approximately the same C*N product
C_FULL_9M = (
    C_REFERENCE_9M
    * N_REFERENCE_9M
    / N_FULL_9M
)


In [ ]:
assert C_FULL_9M > 0
assert np.isfinite(C_FULL_9M)

print("STAGE 9M — FINAL CONFIGURATION")
print("=" * 55)

print("Reference C:", C_REFERENCE_9M)
print("Reference fitting rows:", N_REFERENCE_9M)
print("Full-development rows:", N_FULL_9M)

print("\nProposed full-data C:", C_FULL_9M)
print("Epsilon:", STAGE9_SVR_HP["epsilon"])
print("Features:", X9_full.shape[1])

print("\nReference C*N:",
      C_REFERENCE_9M * N_REFERENCE_9M)

print("Final C*N:",
      C_FULL_9M * N_FULL_9M)

print("\nPASS: Full-data configuration established.")


In [ ]:
# ============================================================
# STAGE 9M.2 — Full-development LinearSVR fit
# ============================================================

import gc
import os
import json
import time
import joblib
import psutil
import sklearn
import warnings
import threading

import numpy as np

from pathlib import Path
from sklearn.svm import LinearSVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9M = True  # Change to True when ready


In [ ]:
# ------------------------------------------------------------
# 1. Construct a self-contained regression pipeline
# ------------------------------------------------------------

def build_final_linear_svr():

    """Construct the original full-development LinearSVR with scaled X and target.
    
    
    Returns
    -------
    sklearn.compose.TransformedTargetRegressor
        Unfitted pipeline with current-run C×N adjustment and inherited epsilon.
    
    Raises
    ------
    ValueError, TypeError
        If stage inputs or underlying estimator settings are incompatible.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    input_pipeline = Pipeline([
        ("input_scaler", StandardScaler()),
        ("linear_svr", LinearSVR(
            C=C_FULL_9M,
            epsilon=STAGE9_SVR_HP["epsilon"],
            loss="epsilon_insensitive",
            dual=True,
            tol=1e-4,
            max_iter=10000,
            random_state=STAGE9_SEED
        ))
    ])

    return TransformedTargetRegressor(
        regressor=input_pipeline,
        transformer=StandardScaler()
    )


In [ ]:
# ------------------------------------------------------------
# 2. Memory monitor
# ------------------------------------------------------------

def stage9m_monitor(stop, samples):
    """Sample whole-process resident memory until the original stop event.
    
    Parameters
    ----------
    stop : threading.Event
        Stop signal used by the original timing block.
    samples : list[int]
        Mutable process-RSS samples in bytes.
    
    Returns
    -------
    None
        Appends RSS byte values to the supplied list; does not isolate model allocations.
    
    Notes
    -----
    Uses this section’s current-run state and original numerical statements.
    Refer to the numbered mathematics, results and limitations immediately above.
    """
    process = psutil.Process(os.getpid())

    while not stop.wait(0.1):
        try:
            samples.append(process.memory_info().rss)
        except psutil.Error:
            break


In [ ]:
# ------------------------------------------------------------
# 3. Fit, verify and persist
# ------------------------------------------------------------

if RUN_STAGE9M:

    assert X9_full.shape == (1_055_198, 108)
if RUN_STAGE9M:
    assert len(y9_full) == 1_055_198
if RUN_STAGE9M:
    assert np.isfinite(X9_full).all()
if RUN_STAGE9M:
    assert np.isfinite(y9_full).all()
if RUN_STAGE9M:

    gc.collect()
if RUN_STAGE9M:

    model_9m = build_final_linear_svr()
if RUN_STAGE9M:

    process = psutil.Process(os.getpid())
if RUN_STAGE9M:
    rss_before = process.memory_info().rss
if RUN_STAGE9M:

    rss_samples = [rss_before]
if RUN_STAGE9M:
    stop = threading.Event()
if RUN_STAGE9M:

    monitor = threading.Thread(
        target=stage9m_monitor,
        args=(stop, rss_samples),
        daemon=True
    )


In [ ]:
if RUN_STAGE9M:

    monitor.start()
if RUN_STAGE9M:

    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter(
                "always", ConvergenceWarning
            )

            t0 = time.perf_counter()

            model_9m.fit(X9_full, y9_full)

            fit_seconds = time.perf_counter() - t0

        warning_count = sum(
            issubclass(w.category, ConvergenceWarning)
            for w in caught
        )

    finally:
        stop.set()
        monitor.join(timeout=2)
        rss_samples.append(
            process.memory_info().rss
        )
if RUN_STAGE9M:

    peak_rss_gib = max(rss_samples) / 1024**3
if RUN_STAGE9M:

    fitted_svr = (
        model_9m.regressor_
        .named_steps["linear_svr"]
    )


In [ ]:
if RUN_STAGE9M:

    iterations = int(
        np.max(np.atleast_1d(fitted_svr.n_iter_))
    )
if RUN_STAGE9M:

    status = (
        "success" if warning_count == 0
        else "convergence_warning"
    )
if RUN_STAGE9M:

    print("\nSTAGE 9M — TRAINING RESULTS")
if RUN_STAGE9M:
    print("=" * 55)
if RUN_STAGE9M:

    print("Status:", status)
if RUN_STAGE9M:
    print("Training observations:", len(X9_full))
if RUN_STAGE9M:
    print("C:", C_FULL_9M)
if RUN_STAGE9M:
    print("Fit time (s):", round(fit_seconds, 3))
if RUN_STAGE9M:
    print("Iterations:", iterations)
if RUN_STAGE9M:
    print("Convergence warnings:", warning_count)
if RUN_STAGE9M:
    print("Peak process RSS (GiB):",
          round(peak_rss_gib, 3))


In [ ]:
if RUN_STAGE9M:

    # -----------------------------------------------
    # 4. Prepare a development-only persistence audit
    # -----------------------------------------------

    rng = np.random.default_rng(STAGE9_SEED)
if RUN_STAGE9M:

    audit_indices = rng.choice(
        len(X9_full),
        size=128,
        replace=False
    )
if RUN_STAGE9M:

    predictions_before = model_9m.predict(
        X9_full[audit_indices]
    )
if RUN_STAGE9M:

    assert np.isfinite(predictions_before).all()
if RUN_STAGE9M:

    # Do not label a nonconverged model as final.
    suffix = (
        "final" if warning_count == 0
        else "nonconverged_candidate"
    )
if RUN_STAGE9M:

    model_path_9m = (
        STAGE9_DIR / f"stage9m_linear_svr_{suffix}.joblib"
    )
if RUN_STAGE9M:

    joblib.dump(model_9m, model_path_9m)


In [ ]:
if RUN_STAGE9M:

    # -----------------------------------------------
    # 5. Reload without retraining
    # -----------------------------------------------

    reloaded_9m = joblib.load(model_path_9m)
if RUN_STAGE9M:

    predictions_after = reloaded_9m.predict(
        X9_full[audit_indices]
    )
if RUN_STAGE9M:

    max_reload_error = float(np.max(
        np.abs(predictions_before - predictions_after)
    ))
if RUN_STAGE9M:

    np.testing.assert_allclose(
        predictions_before,
        predictions_after,
        rtol=1e-12,
        atol=1e-10
    )


In [ ]:
if RUN_STAGE9M:

    # -----------------------------------------------
    # 6. Reproducibility manifest
    # -----------------------------------------------

    manifest_9m = {
        "stage": "9M",
        "algorithm": "LinearSVR",
        "status": status,
        "training_rows": int(len(X9_full)),
        "features": list(stage9_feature_cols),
        "temporal_spans": list(STAGE9_SPANS),
        "C": float(C_FULL_9M),
        "epsilon": float(STAGE9_SVR_HP["epsilon"]),
        "loss": "epsilon_insensitive",
        "dual": True,
        "tol": 1e-4,
        "max_iter": 10000,
        "random_state": int(STAGE9_SEED),
        "fit_seconds": float(fit_seconds),
        "iterations": iterations,
        "convergence_warnings": int(warning_count),
        "peak_process_rss_gib": float(peak_rss_gib),
        "max_reload_prediction_error_celsius":
            max_reload_error,
        "sklearn_version": sklearn.__version__,
        "model_path": str(model_path_9m),
        "test_profiles_accessed": False
    }
if RUN_STAGE9M:

    manifest_path_9m = (
        STAGE9_DIR / "stage9m_training_manifest.json"
    )


In [ ]:
if RUN_STAGE9M:

    with open(manifest_path_9m, "w") as f:
        json.dump(manifest_9m, f, indent=2)
if RUN_STAGE9M:

    print("\nPERSISTENCE VERIFICATION")
if RUN_STAGE9M:
    print("Maximum prediction difference (°C):",
          max_reload_error)
if RUN_STAGE9M:

    print("\nModel saved:", model_path_9m)
if RUN_STAGE9M:
    print("Manifest saved:", manifest_path_9m)
if RUN_STAGE9M:

    if warning_count == 0:
        print("\nPASS: Full-data LinearSVR trained and reloaded.")
    else:
        print(
            "\nREVIEW REQUIRED: Model saved as a candidate "
            "because fitting produced a convergence warning."
        )
else:
    print("Set RUN_STAGE9M = True to fit the final model.")


## 13. Development selections and authoritative handoff


<a id="s-13-1"></a>

### 13.1. Evidence-backed configuration export

**Research trace:** RQ1–RQ4. Source-stage mapping and file-level evidence: `docs/EXPERIMENT_REGISTER.md`, section 13.1.


#### 13.1.1. Purpose

Provide the single authoritative connection from this completed development run to Notebook 01.


#### 13.1.2. Mathematical basis and assumptions

Selection is independently recomputed from the full OLS and five larger-search tables. LinearSVR uses the converged combined Stage 9J/K minimum and the recorded C×N adjustment. Hashes bind the dataset, source, manifest and evidence bytes.


#### 13.1.3. Implementation and parameter choices

export_manifest validates every required trial count, original dimension, objective mean, source profile map and numerical table; it atomically writes development_selected_configurations.json plus relative evidence CSV paths. There is no historical fallback.


#### 13.1.4. Results and verification evidence

A six-row agreement display confirms the exported model mapping. tests/test_handoff.py executes the real exporter/loader with explicitly synthetic tables and checks missing, altered, incomplete, smoke and incompatible cases.


#### 13.1.5. Interpretation and limitations

Synthetic handoff tests validate the software contract only. A production manifest is generated only after this actual Run All finishes all original stages; the delivered ZIP contains no precomputed authoritative manifest.


In [ ]:
from pmsm_project.handoff import export_manifest, configuration_agreement
evidence_tables={
    'ols_pilot':pd.DataFrame(trial_records), 'ols_expanded':expanded_df,
    'ols_full':full_trials, 'svr_nested':results_df, 'stage9h':stage9h_results,
    'stage9i':stage9i_trials, 'stage9j':stage9j_results,
    'stage9k':stage9k_results, 'stage9k_summary':stage9k_summary,
}
for name in MODEL_ORDER:
    evidence_tables[f'pilot_{name}']=all_results[name]
    evidence_tables[f'scaled_{name}']=scaled_results[name]
assert manifest_9m['convergence_warnings']==0, 'Review full-data LinearSVR convergence before final evaluation'
manifest=export_manifest(HANDOFF_PATH,project_root=PROJECT_ROOT,
    run_id=RUN_ID,started_utc=STARTED_UTC,dataset_hash=DATASET_SHA256_ACTUAL,
    profile_cv=profile_cv,tables=evidence_tables,mode=RUN_MODE)
assert manifest['models']['LinearSVR']['hyperparameters']['C']==C_FULL_9M
assert manifest['svr_reference']['spans']==list(STAGE9_SPANS)
assert manifest_9m['convergence_warnings']==0, 'Review full-data LinearSVR convergence before final evaluation'
display(configuration_agreement(manifest,manifest['models']))
print('AUTHORITATIVE HANDOFF:',HANDOFF_PATH)
print('Manifest digest:',manifest['manifest_sha256'])

from pmsm_project.handoff import write_handoff_archive
HANDOFF_ARCHIVE=write_handoff_archive(HANDOFF_PATH)
print("TRANSFER THIS COMPLETE ARCHIVE TO A NEW RUNTIME:",HANDOFF_ARCHIVE)


## 14. Evidence, limitations and conclusions


### 14.1. What each research question can establish

| Question | Evidence generated by this Run All | Permissible conclusion |
|---|---|---|
| RQ1 | split, fold, feature and prefix-isolation checks | implementation-level leakage controls hold for the checked data and code |
| RQ2 | full QR OLS and five Bayesian pilot/larger-search tables | comparative development accuracy and measured cost under the original budgets |
| RQ3 | nested exact-SVR timings; Stage 9A–9M | observed accuracy, approximation and convergence trade-offs for these configurations |
| RQ4 | QR parity, numerical tolerances, fold variability, C/sample-size studies | the support and limits of the development evidence |

### 14.2. Reading fresh results

Inspect objective values, warning counts, numerical rank and timing definitions before interpreting a minimum. A source-level or unit-test pass cannot supply the missing scientific results. Record changes in selected spans, parameters or rankings as new outcomes. Archived source outputs are comparison context only; no target numerical match is required.

### 14.3. Remaining limits and student conclusions

The data describe one laboratory motor and a fixed set of operating sessions. Profile grouping addresses recording overlap; it does not establish deployment safety, transfer to another motor or calibrated uncertainty. MLP internal early stopping is row-based. C×N scaling is heuristic. Three-fold selection scores have limited independent replication. Prior held-out inspection must remain disclosed.

After Run All, use the actual generated tables to write the engineering conclusions and personal learning/AI reflection. The separate 5–10-page report, Moodle submission, presentation, accessible GitHub repository and commit history are student deliverables, not completed by these notebooks.


## 15. Bibliography


References and citation integrity


Bracketed notebook keys resolve through this table. BibTeX uses the canonical records.

| Notebook key | Canonical record |
|---|---|
| KWB21 | kirchgassner2021 |
| BREIMAN01 | breiman2001 |
| GEURTS06 | geurts2006 |
| SMOLA04 | smola2004 |
| WILLIAMS01 | williams2001 |
| FAN08 | fan2008 |
| KINGMA15 | kingma2015 |
| ROBERTS17 | roberts2017 |
| PANDAS | pandas_ewm |
| SKOPT | skopt_gp |
| SKLEARN | sklearn16 |
| SCIPY | scipy115 |

Author–year citations refer to these records. Academic sources explain established methods; they are not evidence that an unexecuted project experiment occurred. Software entries identify the actual implementation/version. Access date for web documentation: 25 October 2026.


**kirchgassner2021.** Kirchgässner, Wilhelm; Wallscheid, Oliver; Böcker, Joachim (2021). *Data-Driven Permanent Magnet Temperature Estimation in Synchronous Motors With Supervised Machine Learning: A Benchmark*. IEEE Transactions on Energy Conversion; 36; 3; 2059--2067. DOI: [10.1109/TEC.2021.3052546](https://doi.org/10.1109/TEC.2021.3052546). [Primary source](https://doi.org/10.1109/TEC.2021.3052546). 


**kirgsn_dataset.** Kirgsn (Wilhelm Kirchgässner) (n.d.). *Electric Motor Temperature*. Kaggle, dataset version 3, measures_v2.csv. [Primary source](https://www.kaggle.com/datasets/wkirgsn/electric-motor-temperature). CC BY-SA 4.0; accessed 25 October 2026


**breiman2001.** Breiman, Leo (2001). *Random Forests*. Machine Learning; 45; 5--32. DOI: [10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324). [Primary source](https://doi.org/10.1023/A:1010933404324). 


**geurts2006.** Geurts, Pierre; Ernst, Damien; Wehenkel, Louis (2006). *Extremely randomized trees*. Machine Learning; 63; 3--42. DOI: [10.1007/s10994-006-6226-1](https://doi.org/10.1007/s10994-006-6226-1). [Primary source](https://doi.org/10.1007/s10994-006-6226-1). 


**smola2004.** Smola, Alex J.; Schölkopf, Bernhard (2004). *A tutorial on support vector regression*. Statistics and Computing; 14; 199--222. DOI: [10.1023/B:STCO.0000035301.49549.88](https://doi.org/10.1023/B:STCO.0000035301.49549.88). [Primary source](https://doi.org/10.1023/B:STCO.0000035301.49549.88). 


**williams2001.** Williams, Christopher K. I.; Seeger, Matthias (2001). *Using the Nyström Method to Speed Up Kernel Machines*. Advances in Neural Information Processing Systems 13. [Primary source](https://proceedings.neurips.cc/paper_files/paper/2000/hash/19de10adbaa1b2ee13f77f679fa1483a-Abstract.html). 


**snoek2012.** Snoek, Jasper; Larochelle, Hugo; Adams, Ryan P. (2012). *Practical Bayesian Optimization of Machine Learning Algorithms*. Advances in Neural Information Processing Systems 25; 2951--2959. [Primary source](https://papers.nips.cc/paper/4522-practical-bayesian-optimization-of-machine-learning-algorithms). 


**rumelhart1986.** Rumelhart, David E.; Hinton, Geoffrey E.; Williams, Ronald J. (1986). *Learning representations by back-propagating errors*. Nature; 323; 533--536. DOI: [10.1038/323533a0](https://doi.org/10.1038/323533a0). [Primary source](https://doi.org/10.1038/323533a0). 


**kingma2015.** Kingma, Diederik P.; Ba, Jimmy (2015). *Adam: A Method for Stochastic Optimization*. International Conference on Learning Representations. [Primary source](https://arxiv.org/abs/1412.6980). arXiv preprint first posted December 2014


**roberts2017.** Roberts, David R. et al. (2017). *Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure*. Ecography; 40; 913--929. DOI: [10.1111/ecog.02881](https://doi.org/10.1111/ecog.02881). [Primary source](https://doi.org/10.1111/ecog.02881). 


**pandas_ewm.** pandas development team (2024). *pandas.DataFrame.ewm and ExponentialMovingWindow.std*. . [Primary source](https://pandas.pydata.org/pandas-docs/version/2.2.3/reference/api/pandas.DataFrame.ewm.html). pandas 2.2.3; accessed 25 October 2026


**skopt_gp.** scikit-optimize developers (2024). *skopt.gp_minimize*. . [Primary source](https://scikit-optimize.readthedocs.io/en/stable/modules/generated/skopt.gp_minimize.html). scikit-optimize 0.10.2; accessed 25 October 2026


**sklearn16.** scikit-learn developers (2025). *scikit-learn 1.6.1 API documentation*. . [Primary source](https://scikit-learn.org/1.6/). LinearSVR, MLPRegressor, Nystroem, StandardScaler, GroupKFold and neighbours APIs; accessed 25 October 2026


**scipy115.** SciPy developers (2025). *scipy.linalg.qr and scipy.linalg.lstsq*. . [Primary source](https://docs.scipy.org/doc/scipy-1.15.3/reference/generated/scipy.linalg.lstsq.html). SciPy 1.15.3; accessed 25 October 2026


**pdoc15.** pdoc contributors (n.d.). *pdoc API documentation*. . [Primary source](https://pdoc.dev/docs/pdoc.html). Generated deliverable uses pdoc 15.0.4, NumPy docformat; accessed 25 October 2026


**monash2026brief.** Monash University (2026). *MMA3001 Numerical Methods and Machine Learning - Project Brief*. Course assessment document supplied by the student, 15 pages. Institutional requirements are cited by printed page in the compliance audit. No public URL or DOI is invented; the source PDF is not redistributed.


**fan2008.** Fan, Rong-En; Chang, Kai-Wei; Hsieh, Cho-Jui; Wang, Xiang-Rui; Lin, Chih-Jen (2008). *LIBLINEAR: A Library for Large Linear Classification*. Journal of Machine Learning Research, 9(61), 1871–1874. [Primary journal record](https://www.jmlr.org/papers/v9/fan08a.html). This cites the library framework; pinned scikit-learn LinearSVR documentation defines the regression and intercept behaviour used here.
